# **TABEL 1**

In [ ]:

# STEP 1 — Clone repository
%cd /content

!git clone https://github.com/MusaWiston/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML.git

%cd /content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML

# Tampilkan struktur file agar kita tahu entry point yang tersedia
!git rev-parse HEAD
!ls -lah
!find . -maxdepth 2 -type f | sort | head -250


/content
Cloning into 'Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML'...
remote: Enumerating objects: 200, done.
remote: Counting objects: 100% (53/53), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 200 (delta 49), reused 24 (delta 24), pack-reused 147 (from 1)
Receiving objects: 100% (200/200), 39.40 MiB | 14.48 MiB/s, done.
Resolving deltas: 100% (73/73), done.
/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML
253d59e5bf030d10480c50654db5bb620dc2fe44
total 71M
drwxr-xr-x 3 root root 4.0K Oct 24 16:38  .
drwxr-xr-x 1 root root 4.0K Oct 24 16:38  ..
-rw-r--r-- 1 root root  13K Oct 24 16:38  aggregate_benchmark.py
-rw-r--r-- 1 root root  32K Oct 24 16:38  Attn_lstm_MoE.py
-rw-r--r-- 1 root root 1.2M Oct 24 16:38  attnlstmoe_architecture.png
-rw-r--r-- 1 root root 8.9K Oct 24 16:38  Attn_lstm.py
-rw-r--r-- 1 root root 7.8K Oct 24 16:38  baseline_gate.py
-rw-r--r-- 1 root root 448K Oct 24 16:38  Baseline_mission_pro

In [ ]:

# STEP 2 — Baca petunjuk proyek
!sed -n '1,240p' README.md

# Cari petunjuk instalasi, data, dan cara menjalankan eksperimen
!find . -maxdepth 3 -type f \
  \( -iname '*requirement*' -o -iname 'environment.yml' \
     -o -iname 'Dockerfile' -o -iname '*.yaml' \
     -o -iname '*.yml' -o -iname '*.sh' \) | sort

# Cari script entry point dan konfigurasi model
!find . -maxdepth 3 -type f \
  \( -iname '*train*' -o -iname '*eval*' -o -iname '*benchmark*' \
     -o -iname '*config*' -o -iname '*test*' \) | sort


# Mission-Phase Feature Learning for eVTOL Battery Prognostics (SOC, SOH, RUL) using ML algorithms
[![Dataset: CMU eVTOL (CC-BY-4.0)](https://img.shields.io/badge/Dataset-CMU%20eVTOL-6f42c1.svg)](https://kilthub.cmu.edu/articles/dataset/eVTOL_Battery_Dataset/14226830)
[![Python](https://img.shields.io/badge/Python-3.10%2B-blue)]()
[![Models](https://img.shields.io/badge/Models-LSTM%2FGRU%2FTCN%2FTFT%20%7C%20MoE-6f42c1)]()


**What is this?** An end-to-end, **mission-phase-aware** ML pipeline for lithium-ion battery **prognostics in eVTOL duty cycles**.  
Engineered phase/mission features from the CMU eVTOL dataset and train both tree ensembles and sequence models (LSTM/GRU/BiLSTM/TCN/TFT, plus **Attention-LSTM-MoE**). The enforced **physical consistency** via a **monotone-RUL** penalty and **post-hoc isotonic calibration** across state of health **EOL thresholds (90/85/80%)** of the initial battery capacity.

Article Citation: Yan, S.; Wiston, M. Mission-Phase Feature Learning for eVTO

In [ ]:

from pathlib import Path
import pandas as pd
import json

ROOT = Path("/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML")

# 1. Lihat kolom, jumlah baris, dan contoh data pada artefak utama
files = [
    "health_mission_features_targets.csv",
    "mission_features.csv",
    "per_cycle_summary_allcells.csv",
    "capacity_tests.csv",
    "sequence_predictions.csv",
    "sequence_fold_metrics.csv",
]

for name in files:
    path = ROOT / name
    print(f"\n{'='*70}\n{name}")
    if not path.exists():
        print("FILE TIDAK DITEMUKAN")
        continue
    df = pd.read_csv(path, nrows=5)
    print("Kolom:", df.columns.tolist())
    print("Contoh:\n", df.head(2).to_string(index=False))

# 2. Periksa metrik yang sudah tersimpan
for name in ["sequence_metrics.json", "pipeline.json"]:
    path = ROOT / name
    print(f"\n{'='*70}\n{name}")
    if path.exists():
        print(path.read_text()[:12000])

# 3. Periksa apakah dependensi dan test tersedia
for name in ["requirements.txt", "requirements-ci.txt"]:
    path = ROOT / name
    print(f"\n{name}: {'ADA' if path.exists() else 'TIDAK ADA'}")

print("\nFolder tests:", (ROOT / "tests").exists())



health_mission_features_targets.csv
Kolom: ['cell_id', 'mission_id', 'cycle_instance_id', 'raw_cycle_number', 'chg_total_dur_s', 'chg_cc_dur_s', 'chg_cv_dur_s', 'chg_energy_Wh', 'chg_cc_energy_Wh', 'chg_cv_energy_Wh', 'chg_cv_fraction', 'cv_setpoint_V', 'chg_mean_I_A', 'chg_mean_C_rate', 'chg_max_C_rate', 'chg_cv_mean_I_A_proxy', 'dis_total_dur_s', 'dis_flight_Wh_abs', 'dis_mean_C_rate', 'dis_mean_C_rate_med', 'dis_max_C_rate', 'dis_max_C_rate_p90', 'dis_vi_R_ohm_med', 'dis_vi_R_ohm_p90', 'temp_mean_C', 'temp_mean_C_med', 'temp_max_C_max', 'temp_th_gain_med', 'temp_th_gain_max', 'temp_slope_med', 'gap_count', 'q0_ah_audit', 'chg_cum_Wh', 'dis_cum_Wh', 'SOH_end_pct', 'RUL90_event_mission', 'RUL90_observed', 'RUL90_missions', 'RUL85_event_mission', 'RUL85_observed', 'RUL85_missions', 'RUL80_event_mission', 'RUL80_observed', 'RUL80_missions', 'SOH_m_plus_5_pct', 'SOH_drop_5_pct']
Contoh:
 cell_id  mission_id  cycle_instance_id  raw_cycle_number  chg_total_dur_s  chg_cc_dur_s  chg_cv_dur_

In [ ]:

from pathlib import Path
import subprocess
import pandas as pd
import json

ROOT = Path("/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML")

# A. Periksa status dan file yang dilacak Git
def run(cmd):
    print(f"\n$ {' '.join(cmd)}")
    result = subprocess.run(
        cmd, cwd=ROOT, text=True, capture_output=True
    )
    print(result.stdout[:10000])
    if result.stderr:
        print("STDERR:", result.stderr[:2000])
    print("Exit code:", result.returncode)

run(["git", "status", "--short"])
run(["git", "ls-files"])
run(["git", "log", "-1", "--format=%H%n%ad%n%s", "--date=iso"])

# B. Periksa apakah arsip phase features berisi CSV yang diperlukan
import zipfile

for name in ["phase_features.rar", "Baseline_mission_profiles.zip", "eda.zip"]:
    p = ROOT / name
    print(f"\n{name}: exists={p.exists()}")
    if p.exists() and p.suffix == ".zip":
        with zipfile.ZipFile(p) as z:
            print("Isi arsip (maksimal 30):", z.namelist()[:30])

# C. Bandingkan ukuran dan rentang kolom durasi
for name in ["health_mission_features_targets.csv", "mission_features.csv"]:
    p = ROOT / name
    if p.exists():
        df = pd.read_csv(p)
        print(f"\n{name}: rows={len(df)}, columns={len(df.columns)}")
        for col in ["cell_id", "mission_id", "chg_total_dur_s",
                    "dis_total_dur_s", "SOH_end_pct", "SOH_pct"]:
            if col in df.columns:
                print(col, df[col].describe().to_dict())
        if {"cell_id", "mission_id"}.issubset(df.columns):
            print("Unique cells:", df["cell_id"].nunique())
            print("Duplicate cell/mission keys:",
                  df.duplicated(["cell_id", "mission_id"]).sum())

# D. Hitung ulang metrik dari prediksi yang tersimpan
p = ROOT / "sequence_predictions.csv"
if p.exists():
    pred = pd.read_csv(p)
    pred["abs_error"] = (pred["y_true"] - pred["y_pred"]).abs()
    pred["sq_error"] = (pred["y_true"] - pred["y_pred"]) ** 2

    print("\nMetrik hitung ulang per task/model:")
    print(pred.groupby(["task", "model"]).agg(
        n=("y_true", "size"),
        MAE=("abs_error", "mean"),
        RMSE=("sq_error", lambda x: (x.mean()) ** 0.5),
        cells=("test_cell", "nunique")
    ).to_string())

# E. Cari manifest dan dokumentasi reproduksibilitas
for name in ["reconstruction_manifest.json", "artifacts.sha256",
             "preprocessing_and_targets.md"]:
    p = ROOT / name
    print(f"\n{name}: exists={p.exists()}")
    if p.exists() and p.suffix == ".json":
        print(p.read_text()[:5000])
    elif p.exists() and p.suffix == ".md":
        print(p.read_text()[:5000])
    elif p.exists():
        print(p.read_text()[:3000])



$ git status --short

Exit code: 0

$ git ls-files
Attn_lstm.py
Attn_lstm_MoE.py
Baseline_mission_profiles.zip
CapacityTest_PSEUDOCODE.md
Capacity_test_pipeline.py
Dataset_ processing stage.py
Exploratory_Data_Analysis Stage.py
Feature engineering stage(features_extractions).py
LOCO_fleet-level cross validation.png
Phase labeling Stage.py
README.md
RUL_true_vs_pred_composite.png
SOC_prediction_accuracy_heatmap_robust.png
SOC_sequences_MAE_clean.png
SOC_sequences_RMSE_clean.png
SOC_trees_MAE_clean.png
SOC_trees_RMSE_clean.png
SOH_threshold_gauge_clean.tiff
Threshold gauge.png
aggregate_benchmark.py
attnlstmoe_architecture.png
baseline_gate.py
baseline_tress.py
battery_common.py
build_reproducible_pipeline.py
capacity_fade_accross all mission profiles.png
capacity_fade_overlays.zip
capacity_test02.yaml
capacity_test_01.py
capacity_tests.csv
combined_mission_phase_feature_importance_heatmap_grayscale_friendly_UPDATED.png
eda.zip
feature_dictionary.csv
gaugethresholds.png
health_mission_f

In [ ]:
# STEP 1 — Audit awal untuk replikasi Table 1

from pathlib import Path
import os
import subprocess
import sys

REPO = Path("/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML")

print("Python:", sys.version)
print("Repository ditemukan:", REPO.exists())

if REPO.exists():
    os.chdir(REPO)

    print("\nCommit repository:")
    subprocess.run(["git", "log", "-1", "--oneline"], check=False)

    print("\nFile yang berhubungan dengan RPT:")
    for p in REPO.rglob("*"):
        if p.is_file() and (
            "capacity" in p.name.lower()
            or "reconstruct" in p.name.lower()
            or "pipeline" in p.name.lower()
            or "rpt" in p.name.lower()
        ):
            print(p.relative_to(REPO))
else:
    print("Path repo tidak ditemukan. Periksa lokasi hasil clone.")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Repository ditemukan: True

Commit repository:

File yang berhubungan dengan RPT:
pipeline.json
capacity_fade_accross all mission profiles.png
build_reproducible_pipeline.py
capacity_tests.csv
capacity_fade_overlays.zip
reconstruct.py
capacity_test02.yaml
reconstruction_manifest.json
CapacityTest_PSEUDOCODE.md
pipeline.py
capacity_test_01.py
Capacity_test_pipeline.py


In [ ]:
# STEP 2 — Cari fungsi dan parameter RPT di source code

from pathlib import Path

keywords = [
    "c5_abs_current",
    "minimum_fraction",
    "start_voltage_min",
    "minimum_duration",
    "capacity_ah",
    "is_capacity_test",
    "valid_rpt",
    "RPT",
]

for filename in [
    "reconstruct.py",
    "build_reproducible_pipeline.py",
    "Capacity_test_pipeline.py",
    "capacity_test_01.py",
]:
    path = REPO / filename

    print(f"\n{'='*20} {filename} {'='*20}")

    if not path.exists():
        print("File tidak ditemukan.")
        continue

    lines = path.read_text(errors="replace").splitlines()
    matches = [
        i for i, line in enumerate(lines)
        if any(word.lower() in line.lower() for word in keywords)
    ]

    if not matches:
        print("Belum ditemukan keyword RPT.")
        continue

    shown = set()
    for i in matches:
        for j in range(max(0, i-3), min(len(lines), i+5)):
            if j not in shown:
                print(f"{j+1:4}: {lines[j]}")
                shown.add(j)
        print("---")


==================== reconstruct.py ====================
 214:         0.0,
 215:     )
 216:     dg = discharge.groupby("cycle_instance_id", sort=False, observed=True)
 217:     rpt = dg.agg(
 218:         raw_cycle_number=("cycleNumber", "first"),
 219:         source_start=("source_row", "min"),
 220:         discharge_duration_s=("dt_s", "sum"),
 221:         c5_duration_s=("c5_dt_s", "sum"),
---
 226:         qdis_min_mAh=("QDischarge_mA_h", "min"),
 227:         qdis_max_mAh=("QDischarge_mA_h", "max"),
 228:     ).reset_index()
 229:     rpt["c5_fraction"] = rpt["c5_duration_s"] / rpt["discharge_duration_s"].replace(0, np.nan)
 230:     rpt["capacity_ah"] = (rpt["qdis_max_mAh"] - rpt["qdis_min_mAh"]) / 1000.0
 231:     rpt["voltage_span_V"] = rpt["max_discharge_voltage_V"] - rpt["min_discharge_voltage_V"]
 232:     rpt["current_ok"] = rpt["c5_fraction"].ge(0.70)
 233:     rpt["voltage_ok"] = (
---
 234:         rpt["start_discharge_voltage_V"].ge(4.10)
---
 235:         & rpt["m

In [ ]:
# STEP 3 — Cari lokasi repository aktif di Google Colab
from pathlib import Path
import subprocess
import os

print("Working directory:", os.getcwd())

# Cari file unik di beberapa lokasi umum Colab
roots = [Path("/content"), Path.cwd()]
candidates = []

for root in roots:
    if root.exists():
        for p in root.glob("**/CapacityTest_PSEUDOCODE.md"):
            candidates.append(p.parent)

# Hilangkan duplikat
candidates = list(dict.fromkeys(candidates))

print("\nRepository candidates:")
for i, p in enumerate(candidates):
    print(i, p)

if not candidates:
    print("\nBelum menemukan repository. Jalankan:")
    print("!ls -la /content")
else:
    REPO = candidates[0]
    os.chdir(REPO)
    print("\nMenggunakan:", REPO)
    print("Git directory exists:", (REPO / ".git").exists())

    if (REPO / ".git").exists():
        subprocess.run(["git", "log", "-1", "--oneline"])
    else:
        print("Folder .git tidak ada; kita tetap bisa membaca source code.")

    for filename in [
        "CapacityTest_PSEUDOCODE.md",
        "capacity_test_01.py",
        "Capacity_test_pipeline.py",
        "reconstruct.py",
        "build_reproducible_pipeline.py",
    ]:
        path = REPO / filename
        print(f"\n{'='*16} {filename} {'='*16}")
        if path.exists():
            lines = path.read_text(errors="replace").splitlines()
            hits = [
                i for i, line in enumerate(lines)
                if any(k in line.lower() for k in [
                    "c5_abs_current", "minimum_fraction",
                    "start_voltage_min", "minimum_duration",
                    "capacity_ah", "is_capacity_test",
                    "valid_rpt", "rpt"
                ])
            ]
            if not hits:
                print("\n".join(lines[:35]))
            else:
                shown = set()
                for i in hits:
                    for j in range(max(0, i-2), min(len(lines), i+5)):
                        if j not in shown:
                            print(f"{j+1:4}: {lines[j]}")
                            shown.add(j)
                        if len(shown) >= 100:
                            break
                    if len(shown) >= 100:
                        break
        else:
            print("File tidak ditemukan.")

Working directory: /content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML

Repository candidates:
0 /content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML

Menggunakan: /content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML
Git directory exists: True

================ CapacityTest_PSEUDOCODE.md ================
  16: 
  17:         full_charge ← median(last N charge voltages)
  18:                       ≥ voltage_setpoint × minimum_fraction
  19:         capacity ← maximum cumulative charge capacity among charge_rows
  20:         chronological ← last(charge time) ≤ first(flight time)
  21: 
  22:         accept cycle iff:

================ capacity_test_01.py ================
  16:             "full_charge_voltage_v": 4.15, "full_charge_min_fraction": 0.98,
  17:             "full_charge_tail_points": 3, "min_charge_rows": 3, "min_discharge_rows": 3,
  18:             "min_capacity_ah": 1.0, "max_capacity_ah": 4.0,
 

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO = Path.cwd()
print("Repository:", REPO)

print("\nFolder data:")
for p in REPO.iterdir():
    if p.is_dir():
        print(p.name)

print("\nFile raw VAH*.csv yang ditemukan:")
raw_files = list(REPO.rglob("VAH*.csv"))
for p in raw_files[:30]:
    print(p)
print("Jumlah file:", len(raw_files))

print("\nArgumen pipeline:")
subprocess.run([
    sys.executable,
    str(REPO / "build_reproducible_pipeline.py"),
    "--help"
])

Repository: /content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML

Folder data:
.git

File raw VAH*.csv yang ditemukan:
Jumlah file: 0

Argumen pipeline:


CompletedProcess(args=['/usr/bin/python3', '/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML/build_reproducible_pipeline.py', '--help'], returncode=1)

In [ ]:
from pathlib import Path
import subprocess
import sys
import os

REPO = Path("/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML")

print("=== Isi repository ===")
for p in sorted(REPO.iterdir()):
    print(("DIR  " if p.is_dir() else "FILE "), p.name)

print("\n=== File CSV yang tersedia ===")
for p in list(REPO.rglob("*.csv"))[:50]:
    print(p.relative_to(REPO))

print("\n=== Error pipeline ===")
result = subprocess.run(
    [sys.executable, str(REPO / "build_reproducible_pipeline.py"), "--help"],
    cwd=REPO,
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)
print("\nSTDOUT:\n", result.stdout)
print("\nSTDERR:\n", result.stderr)

=== Isi repository ===
DIR   .git
FILE  Attn_lstm.py
FILE  Attn_lstm_MoE.py
FILE  Baseline_mission_profiles.zip
FILE  CapacityTest_PSEUDOCODE.md
FILE  Capacity_test_pipeline.py
FILE  Dataset_ processing stage.py
FILE  Exploratory_Data_Analysis Stage.py
FILE  Feature engineering stage(features_extractions).py
FILE  LOCO_fleet-level cross validation.png
FILE  Phase labeling Stage.py
FILE  README.md
FILE  RUL_true_vs_pred_composite.png
FILE  SOC_prediction_accuracy_heatmap_robust.png
FILE  SOC_sequences_MAE_clean.png
FILE  SOC_sequences_RMSE_clean.png
FILE  SOC_trees_MAE_clean.png
FILE  SOC_trees_RMSE_clean.png
FILE  SOH_threshold_gauge_clean.tiff
FILE  Threshold gauge.png
FILE  aggregate_benchmark.py
FILE  attnlstmoe_architecture.png
FILE  baseline_gate.py
FILE  baseline_tress.py
FILE  battery_common.py
FILE  build_reproducible_pipeline.py
FILE  capacity_fade_accross all mission profiles.png
FILE  capacity_fade_overlays.zip
FILE  capacity_test02.yaml
FILE  capacity_test_01.py
FILE  capac

In [ ]:
from pathlib import Path

REPO = Path("/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML")

print("=== Import dan konfigurasi pipeline ===")
for filename in [
    "build_reproducible_pipeline.py",
    "reconstruct.py",
    "pipeline.json",
    "reconstruction_manifest.json",
    "README.md",
]:
    path = REPO / filename
    print(f"\n--- {filename} ---")
    if path.exists() and path.is_file():
        lines = path.read_text(errors="replace").splitlines()
        if filename == "build_reproducible_pipeline.py":
            print("\n".join(f"{i+1}: {line}" for i, line in enumerate(lines[:100])))
        elif filename == "reconstruct.py":
            print("\n".join(f"{i+1}: {line}" for i, line in enumerate(lines[:90])))
        elif filename == "README.md":
            matches = [
                (i, line) for i, line in enumerate(lines)
                if any(term in line.lower() for term in [
                    "reproduc", "evt_battery", "install", "dataset",
                    "raw data", "requirements"
                ])
            ]
            for i, line in matches[:30]:
                print(f"{i+1}: {line}")
        else:
            print("\n".join(lines[:100]))
    else:
        print("File tidak ditemukan.")

=== Import dan konfigurasi pipeline ===

--- build_reproducible_pipeline.py ---
1: #!/usr/bin/env python3
2: """CLI for the complete reproducible preprocessing/target pipeline."""
3: 
4: from __future__ import annotations
5: 
6: import argparse
7: import json
8: from pathlib import Path
9: import sys
10: 
11: 
12: REPOSITORY_ROOT = Path(__file__).resolve().parents[1]
13: if str(REPOSITORY_ROOT) not in sys.path:
14:     sys.path.insert(0, str(REPOSITORY_ROOT))
15: 
16: from evt_battery.config import PipelineConfig  # noqa: E402
17: from evt_battery.pipeline import run_pipeline  # noqa: E402
18: 
19: 
20: def parse_args() -> argparse.Namespace:
21:     parser = argparse.ArgumentParser(
22:         description=(
23:             "Build cycle/RPT audits, phase and mission tables, explicit SOH/RUL "
24:             "targets, sliding-window samples, LOCO manifests, and checksums."
25:         )
26:     )
27:     parser.add_argument("--data-dir", type=Path, required=True, help="Directory conta

In [ ]:
from pathlib import Path

ROOT = Path("/content")

matches = [
    str(p) for p in ROOT.rglob("evt_battery")
    if p.is_dir()
]

print("Folder evt_battery:", matches or "Tidak ditemukan")

print("\nFolder src atau package yang mungkin relevan:")
for p in Path("/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML").rglob("*"):
    if p.is_dir() and (
        p.name in {"src", "evt_battery", "tests"}
        or p.name.endswith(".egg-info")
    ):
        print(p)

Folder evt_battery: Tidak ditemukan

Folder src atau package yang mungkin relevan:


In [ ]:
from pathlib import Path
import subprocess

REPO = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML"
)

def run_git(*args):
    result = subprocess.run(
        ["git", "-C", str(REPO), *args],
        capture_output=True,
        text=True
    )
    print(f"\n$ git {' '.join(args)}")
    print(result.stdout or result.stderr or "(tidak ada output)")
    return result

run_git("remote", "-v")
run_git("branch", "-a")
run_git("log", "-1", "--oneline")
run_git("status", "--short")

print("\n=== Apakah file penting tercatat di Git? ===")
for path in [
    "evt_battery/config.py",
    "evt_battery/pipeline.py",
    "config/pipeline.json",
    "scripts/build_reproducible_pipeline.py",
    "build_reproducible_pipeline.py",
    "reconstruct.py",
]:
    result = subprocess.run(
        ["git", "-C", str(REPO), "ls-files", "--error-unmatch", path],
        capture_output=True,
        text=True
    )
    print(f"{path}: {'TERCATAT' if result.returncode == 0 else 'TIDAK TERCATAT'}")


$ git remote -v
origin	https://github.com/MusaWiston/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML.git (fetch)
origin	https://github.com/MusaWiston/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML.git (push)


$ git branch -a
* main
  remotes/origin/HEAD -> origin/main
  remotes/origin/main


$ git log -1 --oneline
253d59e Update README.md


$ git status --short
(tidak ada output)

=== Apakah file penting tercatat di Git? ===
evt_battery/config.py: TIDAK TERCATAT
evt_battery/pipeline.py: TIDAK TERCATAT
config/pipeline.json: TIDAK TERCATAT
scripts/build_reproducible_pipeline.py: TIDAK TERCATAT
build_reproducible_pipeline.py: TERCATAT
reconstruct.py: TERCATAT


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML"
)

result = subprocess.run(
    [sys.executable, str(REPO / "reconstruct.py"), "--help"],
    cwd=REPO,
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)
print("STDOUT:\n", result.stdout)
print("STDERR:\n", result.stderr)

Return code: 0
STDOUT:
 usage: reconstruct.py [-h] --raw-dir RAW_DIR --output-dir OUTPUT_DIR

Reconstruct leakage-safe mission features and health targets from CMU
telemetry. This implementation follows the checked manuscript protocol rather
than the legacy feature CSVs in the project repository. It deliberately keeps
a superset of causal mission descriptors so that the exact 20-column manifest
can be audited against the published tree-control metrics before neural-model
training.

options:
  -h, --help            show this help message and exit
  --raw-dir RAW_DIR
  --output-dir OUTPUT_DIR

STDERR:
 


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path

p = Path('/content/drive/MyDrive/14226830.zip')
print("Ditemukan:", p.exists())

if p.exists():
    print("Ukuran GB:", round(p.stat().st_size / 1e9, 2))

Ditemukan: True
Ukuran GB: 1.54


In [ ]:
from pathlib import Path
import zipfile

zip_path = Path("/content/drive/MyDrive/14226830.zip")

with zipfile.ZipFile(zip_path) as z:
    infos = z.infolist()
    csvs = [i for i in infos if i.filename.lower().endswith(".csv")]
    vah_csvs = [
        i for i in csvs
        if Path(i.filename).name.upper().startswith("VAH")
    ]

    print("ZIP valid: True")
    print("Jumlah semua CSV:", len(csvs))
    print("Jumlah file VAH*.csv:", len(vah_csvs))

    print("\nContoh file VAH:")
    for item in vah_csvs[:30]:
        print(item.filename, "|", round(item.file_size / 1e6, 2), "MB")

    if not vah_csvs:
        print("\nContoh isi arsip:")
        for item in infos[:40]:
            print(item.filename)

ZIP valid: True
Jumlah semua CSV: 43
Jumlah file VAH*.csv: 43

Contoh file VAH:
VAH23.csv | 52.94 MB
VAH07.csv | 20.87 MB
VAH22.csv | 38.96 MB
VAH15.csv | 36.86 MB
VAH02.csv | 102.41 MB
VAH12.csv | 133.28 MB
VAH30.csv | 59.54 MB
VAH10.csv | 95.06 MB
VAH25.csv | 40.33 MB
VAH09.csv | 68.76 MB
VAH28.csv | 77.37 MB
VAH05.csv | 101.56 MB
VAH11.csv | 138.35 MB
VAH20.csv | 39.72 MB
VAH16.csv | 34.71 MB
VAH24.csv | 65.0 MB
VAH06.csv | 80.52 MB
VAH27.csv | 39.54 MB
VAH17.csv | 65.55 MB
VAH01.csv | 110.88 MB
VAH13.csv | 65.05 MB
VAH26.csv | 72.08 MB
VAH01_impedance.csv | 0.0 MB
VAH02_impedance.csv | 0.0 MB
VAH05_impedance.csv | 0.0 MB
VAH06_impedance.csv | 0.0 MB
VAH07_impedance.csv | 0.0 MB
VAH09_impedance.csv | 0.0 MB
VAH10_impedance.csv | 0.0 MB
VAH11_impedance.csv | 0.0 MB


In [ ]:
from pathlib import Path
import zipfile

ZIP_PATH = Path("/content/drive/MyDrive/14226830.zip")
RAW_DIR = Path("/content/evtol_raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH) as z:
    telemetry = [
        name for name in z.namelist()
        if Path(name).name.upper().startswith("VAH")
        and name.lower().endswith(".csv")
        and "_impedance" not in Path(name).name.lower()
    ]

    print("CSV telemetri ditemukan:", len(telemetry))

    for name in telemetry:
        target = RAW_DIR / Path(name).name
        if not target.exists():
            with z.open(name) as src, target.open("wb") as dst:
                while True:
                    chunk = src.read(8 * 1024 * 1024)
                    if not chunk:
                        break
                    dst.write(chunk)

print("\nFile yang berhasil diekstrak:", len(list(RAW_DIR.glob("VAH*.csv"))))
print("Folder data:", RAW_DIR)
print("Contoh:", [p.name for p in sorted(RAW_DIR.glob("VAH*.csv"))[:10]])

CSV telemetri ditemukan: 22

File yang berhasil diekstrak: 22
Folder data: /content/evtol_raw
Contoh: ['VAH01.csv', 'VAH02.csv', 'VAH05.csv', 'VAH06.csv', 'VAH07.csv', 'VAH09.csv', 'VAH10.csv', 'VAH11.csv', 'VAH12.csv', 'VAH13.csv']


In [ ]:
from pathlib import Path
import pandas as pd

RAW_DIR = Path("/content/evtol_raw")

for path in sorted(RAW_DIR.glob("VAH*.csv"))[:5]:
    df = pd.read_csv(path, nrows=5)
    print(f"\n{path.name}")
    print("Kolom:", df.columns.tolist())
    print(df.head(2).to_string(index=False))


VAH01.csv
Kolom: ['time_s', 'Ecell_V', 'I_mA', 'EnergyCharge_W_h', 'QCharge_mA_h', 'EnergyDischarge_W_h', 'QDischarge_mA_h', 'Temperature__C', 'cycleNumber', 'Ns']
 time_s  Ecell_V      I_mA  EnergyCharge_W_h  QCharge_mA_h  EnergyDischarge_W_h  QDischarge_mA_h  Temperature__C  cycleNumber  Ns
    0.0 3.538717 1491.3828          0.000000      0.000000                    0                0       21.139246            0   0
   10.0 3.545570 1492.1708          0.014693      4.144108                    0                0       21.115524            0   0

VAH02.csv
Kolom: ['time_s', 'Ecell_V', 'I_mA', 'EnergyCharge_W_h', 'QCharge_mA_h', 'EnergyDischarge_W_h', 'QDischarge_mA_h', 'Temperature__C', 'cycleNumber', 'Ns']
 time_s  Ecell_V      I_mA  EnergyCharge_W_h  QCharge_mA_h  EnergyDischarge_W_h  QDischarge_mA_h  Temperature__C  cycleNumber  Ns
    0.0 3.546661 1489.7195          0.000000      0.000000                    0                0       21.375183            0   0
   10.0 3.550481 148

In [ ]:
from pathlib import Path
import subprocess
import sys
import time

REPO = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML"
)
RAW_DIR = Path("/content/evtol_raw")
OUT_DIR = Path("/content/table1_reconstruction")

assert len(list(RAW_DIR.glob("VAH*.csv"))) == 22, "Jumlah file mentah bukan 22."
assert not OUT_DIR.exists(), (
    f"Folder output sudah ada: {OUT_DIR}. "
    "Pilih folder output baru agar hasil sebelumnya tidak tertimpa."
)

start = time.time()

result = subprocess.run(
    [
        sys.executable,
        str(REPO / "reconstruct.py"),
        "--raw-dir", str(RAW_DIR),
        "--output-dir", str(OUT_DIR),
    ],
    cwd=REPO,
    capture_output=True,
    text=True,
)

print("Return code:", result.returncode)
print("Durasi (menit):", round((time.time() - start) / 60, 2))
print("\nSTDOUT:\n", result.stdout[-12000:])
print("\nSTDERR:\n", result.stderr[-12000:])

Return code: 0
Durasi (menit): 4.34

STDOUT:
 VAH01: rows=1,091,477 missions=829 RPTs=18
VAH02: rows=1,004,761 missions=611 RPTs=14
VAH05: rows=1,015,444 missions=1,516 RPTs=30
VAH06: rows=809,016 missions=942 RPTs=20
VAH07: rows=205,487 missions=268 RPTs=6
VAH09: rows=680,849 missions=785 RPTs=23
VAH10: rows=949,302 missions=1,400 RPTs=29
VAH11: rows=1,381,245 missions=2,203 RPTs=44
VAH12: rows=1,331,202 missions=2,302 RPTs=47
VAH13: rows=647,633 missions=1,020 RPTs=21
VAH15: rows=365,851 missions=542 RPTs=11
VAH16: rows=343,414 missions=547 RPTs=11
VAH17: rows=650,381 missions=981 RPTs=21
VAH20: rows=392,502 missions=598 RPTs=13
VAH22: rows=385,429 missions=540 RPTs=11
VAH23: rows=528,761 missions=682 RPTs=15
VAH24: rows=653,075 missions=784 RPTs=17
VAH25: rows=401,297 missions=541 RPTs=10
VAH26: rows=718,345 missions=1,138 RPTs=24
VAH27: rows=391,649 missions=573 RPTs=13
VAH28: rows=770,958 missions=1,155 RPTs=24
VAH30: rows=591,239 missions=900 RPTs=19
{
  "raw_cells": 22,
  "missi

In [ ]:
from pathlib import Path

OUT_DIR = Path("/content/table1_reconstruction")

print("Folder output ada:", OUT_DIR.exists())

if OUT_DIR.exists():
    for p in sorted(OUT_DIR.iterdir()):
        if p.is_file():
            print(f"{p.name:45} {p.stat().st_size / 1e6:8.2f} MB")
        elif p.is_dir():
            print(f"{p.name}/")

Folder output ada: True
capacity_tests.csv                                0.10 MB
health_mission_features_targets.csv              11.99 MB
mission_features_targets_superset.csv            13.26 MB
reconstruction_manifest.json                      0.01 MB


In [ ]:
from pathlib import Path
import json
import pandas as pd

OUT_DIR = Path("/content/table1_reconstruction")

rpt_path = OUT_DIR / "capacity_tests.csv"
manifest_path = OUT_DIR / "reconstruction_manifest.json"

rpt = pd.read_csv(rpt_path)
manifest = json.loads(manifest_path.read_text())

print("=== Ringkasan hasil aktual ===")
print("Jumlah baris capacity_tests.csv:", len(rpt))
print("Kolom:", rpt.columns.tolist())

print("\n=== Lima baris pertama ===")
display(rpt.head())

# Cari kolom identitas sel yang tersedia.
cell_col = next(
    (c for c in ["cell_id", "cell", "cell_name", "battery_id"] if c in rpt.columns),
    None
)

if cell_col:
    counts = rpt.groupby(cell_col).size().rename("RPT aktual")
    print("\n=== Jumlah RPT per sel ===")
    display(counts.to_frame())
else:
    print("\nKolom ID sel tidak langsung dikenali.")
    print("Periksa kolom di atas untuk menentukan kolom identitas yang tepat.")

print("\n=== Manifest hasil run ===")
print(json.dumps(manifest.get("counts", {}), indent=2))

=== Ringkasan hasil aktual ===
Jumlah baris capacity_tests.csv: 441
Kolom: ['cycle_instance_id', 'raw_cycle_number', 'source_start', 'discharge_duration_s', 'c5_duration_s', 'start_discharge_voltage_V', 'final_discharge_voltage_V', 'min_discharge_voltage_V', 'max_discharge_voltage_V', 'qdis_min_mAh', 'qdis_max_mAh', 'c5_fraction', 'capacity_ah', 'voltage_span_V', 'current_ok', 'voltage_ok', 'capacity_ok', 'duration_ok', 'is_capacity_test', 'cell_id', 'mission_position', 'SOH_pct', 'rpt_order']

=== Lima baris pertama ===


,cycle_instance_id,raw_cycle_number,source_start,discharge_duration_s,c5_duration_s,start_discharge_voltage_V,final_discharge_voltage_V,min_discharge_voltage_V,max_discharge_voltage_V,qdis_min_mAh,...,voltage_span_V,current_ok,voltage_ok,capacity_ok,duration_ok,is_capacity_test,cell_id,mission_position,SOH_pct,rpt_order
0,1,0,786,19308.616917,18268.678868,4.175987,3.104852,2.500072,4.175987,0.000336,...,1.675915,True,True,True,True,True,VAH01,0,100.000000,1
1,51,1,108378,17783.223855,17723.227849,4.170198,2.500150,2.500150,4.170198,0.000337,...,1.670048,True,True,True,True,True,VAH01,50,97.101928,2
2,102,2,215128,17397.581815,17337.585808,4.168387,2.500190,2.500190,4.168387,0.000338,...,1.668197,True,True,True,True,True,VAH01,100,95.027445,3
3,153,3,325277,17137.759788,17077.763781,4.169568,2.500072,2.500072,4.169568,0.000337,...,1.669496,True,True,True,True,True,VAH01,150,93.643012,4
4,204,4,436385,16930.970018,16870.974011,4.172679,2.500111,2.500111,4.172679,0.000337,...,1.672568,True,True,True,True,True,VAH01,200,92.532341,5



=== Jumlah RPT per sel ===


,RPT aktual
cell_id,
VAH01,18
VAH02,14
VAH05,30
VAH06,20
VAH07,6
VAH09,23
VAH10,29
VAH11,44
VAH12,47



=== Manifest hasil run ===
{
  "raw_cells": 22,
  "missions_all_cells": 20857,
  "valid_rpts_all_cells": 441,
  "health_cells": 19,
  "health_supported_rows": 18608,
  "soh_m_plus_5_rows": 18513,
  "rul90_rows": 5814,
  "rul85_rows": 9595,
  "rul80_rows": 8257,
  "rul90_cells": 19,
  "rul85_cells": 16,
  "rul80_cells": 8
}


In [ ]:
from pathlib import Path
import pandas as pd

p = Path("/content/table1_reconstruction/capacity_tests.csv")
df = pd.read_csv(p)

# Audit ulang aturan yang tercantum pada reconstruction_manifest.json
checks = pd.DataFrame(index=df.index)

checks["current_ok_recomputed"] = df["c5_fraction"] >= 0.70

checks["voltage_ok_recomputed"] = (
    (df["start_discharge_voltage_V"] >= 4.10)
    & (df["min_discharge_voltage_V"] <= 2.55)
    & (df["voltage_span_V"] >= 1.40)
)

checks["capacity_ok_recomputed"] = df["capacity_ah"].between(
    1.50, 3.75, inclusive="both"
)

checks["duration_ok_recomputed"] = df["discharge_duration_s"] >= 7200

checks["all_rules_recomputed"] = checks.all(axis=1)

print("=== Jumlah RPT ===")
print("Baris output:", len(df))
print("Lolos semua aturan hasil audit:", int(checks["all_rules_recomputed"].sum()))
print("Nilai is_capacity_test pada CSV:", int(df["is_capacity_test"].sum()))

print("\n=== Konsistensi tiap kriteria ===")
for original, recomputed in [
    ("current_ok", "current_ok_recomputed"),
    ("voltage_ok", "voltage_ok_recomputed"),
    ("capacity_ok", "capacity_ok_recomputed"),
    ("duration_ok", "duration_ok_recomputed"),
]:
    mismatch = int((df[original].astype(bool) != checks[recomputed]).sum())
    print(f"{original}: {mismatch} perbedaan")

print("\n=== Nilai minimum/maksimum untuk audit ===")
for col in [
    "c5_fraction",
    "start_discharge_voltage_V",
    "min_discharge_voltage_V",
    "voltage_span_V",
    "capacity_ah",
    "discharge_duration_s",
]:
    print(f"{col}: {df[col].min():.4f} sampai {df[col].max():.4f}")

=== Jumlah RPT ===
Baris output: 441
Lolos semua aturan hasil audit: 441
Nilai is_capacity_test pada CSV: 441

=== Konsistensi tiap kriteria ===
current_ok: 0 perbedaan
voltage_ok: 0 perbedaan
capacity_ok: 0 perbedaan
duration_ok: 0 perbedaan

=== Nilai minimum/maksimum untuk audit ===
c5_fraction: 0.9302 sampai 0.9970
start_discharge_voltage_V: 4.1456 sampai 4.1835
min_discharge_voltage_V: 2.4999 sampai 2.5003
voltage_span_V: 1.6455 sampai 1.6835
capacity_ah: 2.1956 sampai 3.1397
discharge_duration_s: 13268.2797 sampai 34697.5009


In [ ]:
from pathlib import Path

repo = Path("/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML")
src = (repo / "reconstruct.py").read_text().splitlines()

for i, line in enumerate(src):
    if any(term in line.lower() for term in ["c5_dt_s", "c5_fraction", "0.39", "0.81"]):
        start = max(0, i - 3)
        end = min(len(src), i + 5)
        print(f"\n--- reconstruct.py baris {start+1}-{end} ---")
        for j in range(start, end):
            print(f"{j+1:4}: {src[j]}")


--- reconstruct.py baris 208-215 ---
 208:     raw["I_squared"] = raw["I_A"] ** 2
 209: 
 210:     discharge = raw.loc[raw["I_A"].lt(-0.05)].copy()
 211:     discharge["c5_dt_s"] = np.where(
 212:         discharge["abs_I_A"].between(0.39, 0.81, inclusive="both"),
 213:         discharge["dt_s"],
 214:         0.0,
 215:     )

--- reconstruct.py baris 209-216 ---
 209: 
 210:     discharge = raw.loc[raw["I_A"].lt(-0.05)].copy()
 211:     discharge["c5_dt_s"] = np.where(
 212:         discharge["abs_I_A"].between(0.39, 0.81, inclusive="both"),
 213:         discharge["dt_s"],
 214:         0.0,
 215:     )
 216:     dg = discharge.groupby("cycle_instance_id", sort=False, observed=True)

--- reconstruct.py baris 218-225 ---
 218:         raw_cycle_number=("cycleNumber", "first"),
 219:         source_start=("source_row", "min"),
 220:         discharge_duration_s=("dt_s", "sum"),
 221:         c5_duration_s=("c5_dt_s", "sum"),
 222:         start_discharge_voltage_V=("Ecell_V", "first"

In [ ]:
rows = [
    {
        "Criterion": "Reference current",
        "Checked Rule": ">=70% of valid discharge time at 0.39-0.81 A",
        "Purpose": "C/5 consistency around 0.6 A",
        "Implementation": (
            "abs_I_A in [0.39, 0.81]; c5_fraction >= 0.70"
        ),
        "Rows passing audit": int(audit["current_ok"].sum()),
        "Rows audited": len(df),
    },
    {
        "Criterion": "Voltage coverage",
        "Checked Rule": (
            "Start >= 4.10 V; end/minimum <= 2.55 V; span >= 1.40 V"
        ),
        "Purpose": "Full diagnostic discharge",
        "Implementation": (
            "Start voltage >= 4.10 V; minimum <= 2.55 V; "
            "span >= 1.40 V"
        ),
        "Rows passing audit": int(audit["voltage_ok"].sum()),
        "Rows audited": len(df),
    },
    {
        "Criterion": "Capacity range",
        "Checked Rule": "1.50-3.75 Ah",
        "Purpose": "Reject incomplete or implausible tests",
        "Implementation": "capacity_ah between 1.50 and 3.75 Ah",
        "Rows passing audit": int(audit["capacity_ok"].sum()),
        "Rows audited": len(df),
    },
    {
        "Criterion": "Duration",
        "Checked Rule": "Valid discharge duration >= 7200 s",
        "Purpose": "Exclude short operating cycles",
        "Implementation": "discharge_duration_s >= 7200",
        "Rows passing audit": int(audit["duration_ok"].sum()),
        "Rows audited": len(df),
    },
]

comparison = pd.DataFrame(rows)
comparison.to_csv(
    OUT_DIR / "table1_replication_comparison.csv",
    index=False
)

display(comparison)

,Criterion,Checked Rule,Purpose,Implementation,Rows passing audit,Rows audited
0,Reference current,>=70% of valid discharge time at 0.39-0.81 A,C/5 consistency around 0.6 A,"abs_I_A in [0.39, 0.81]; c5_fraction >= 0.70",441,441
1,Voltage coverage,Start >= 4.10 V; end/minimum <= 2.55 V; span >...,Full diagnostic discharge,Start voltage >= 4.10 V; minimum <= 2.55 V; sp...,441,441
2,Capacity range,1.50-3.75 Ah,Reject incomplete or implausible tests,capacity_ah between 1.50 and 3.75 Ah,441,441
3,Duration,Valid discharge duration >= 7200 s,Exclude short operating cycles,discharge_duration_s >= 7200,441,441


# **TABEL 2**

In [ ]:
# PEMERIKSAAN TERARAH TABLE 2A DAN TABLE 2B
# Tidak mengubah data atau melatih model.

print("=== OBJEK UTAMA ===")
print("T tersedia:", "T" in globals())
print("D tersedia:", "D" in globals())
print("C tersedia:", "C" in globals())
print("W tersedia:", "W" in globals())

if "T" in globals():
    print("\n=== SUMBER FUNGSI TABEL ===")
    import inspect

    for name in ["table2a", "table2b"]:
        if hasattr(T, name):
            print(f"\n--- T.{name} ---")
            try:
                print(inspect.getsource(getattr(T, name)))
            except (OSError, TypeError):
                print("Source tidak tersedia melalui inspect.")

if "D" in globals():
    print("\n=== DATA BENCHMARK ===")
    for name in ["missions_bench", "phases_bench"]:
        if name in D:
            df = D[name]
            print(f"\n{name}: shape={df.shape}")
            print("Kolom:")
            print(df.columns.tolist())

            if "cell_id" in df.columns:
                print("Jumlah sel:", df["cell_id"].nunique())
                print("Sel:", sorted(df["cell_id"].dropna().astype(str).unique()))

            if "mission_id" in df.columns:
                print("Jumlah mission_id unik:", df["mission_id"].nunique())

if "C" in globals():
    print("\n=== DAFTAR FITUR KONFIGURASI ===")
    for name in ["PHASE_FEATURES", "MISSION_FEATURES"]:
        if hasattr(C, name):
            print(f"{name}:")
            print(getattr(C, name))

=== OBJEK UTAMA ===
T tersedia: False
D tersedia: False
C tersedia: False
W tersedia: False


In [ ]:
from pathlib import Path

REPO_DIR = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML"
)

print("Folder proyek tersedia:", REPO_DIR.exists())

# Cari file Python dan notebook yang mungkin membentuk T, D, C, W
patterns = ["*.py", "*.ipynb"]
files = []

for pattern in patterns:
    files.extend(REPO_DIR.rglob(pattern))

keywords = [
    "def table2a",
    "def table2b",
    "missions_bench",
    "phases_bench",
    "build_mission_samples",
    "build_phase_samples",
    "PHASE_FEATURES",
    "MISSION_FEATURES",
]

print("\n=== FILE YANG RELEVAN ===")

matches = []

for path in files:
    if any(
        part in {"venv", ".git", "__pycache__"}
        for part in path.parts
    ):
        continue

    try:
        content = path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        continue

    found = [kw for kw in keywords if kw in content]

    if found:
        matches.append((path, found))
        print(f"\nFile: {path.relative_to(REPO_DIR)}")
        print("Kata kunci:", ", ".join(found))

if not matches:
    print(
        "\nTidak ditemukan definisi yang cocok di folder proyek. "
        "Kita perlu memeriksa lokasi notebook atau skrip yang digunakan "
        "untuk menghasilkan tabel."
    )

Folder proyek tersedia: True

=== FILE YANG RELEVAN ===

Tidak ditemukan definisi yang cocok di folder proyek. Kita perlu memeriksa lokasi notebook atau skrip yang digunakan untuk menghasilkan tabel.


In [ ]:
from pathlib import Path

ROOT = Path("/content")

print("=== 1. FILE NOTEBOOK ===")
notebooks = list(ROOT.glob("*.ipynb"))

for p in notebooks:
    print(p)

print("\n=== 2. FILE CSV TERKAIT TABLE 2 ===")

keywords = [
    "table2a",
    "table2b",
    "phase_features",
    "mission_features",
    "phase_targets",
    "mission_targets",
    "health_mission_features_targets",
]

for p in ROOT.rglob("*.csv"):
    if any(
        part in {"venv", ".git", "__pycache__"}
        for part in p.parts
    ):
        continue

    if any(k in p.name.lower() for k in keywords):
        print(p)

print("\n=== 3. FILE PYTHON DI LUAR REPOSITORI ===")

repo = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML"
)

for p in ROOT.glob("*.py"):
    print(p)

print("\nSelesai.")

=== 1. FILE NOTEBOOK ===

=== 2. FILE CSV TERKAIT TABLE 2 ===
/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML/health_mission_features_targets.csv
/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML/mission_features.csv

=== 3. FILE PYTHON DI LUAR REPOSITORI ===

Selesai.


In [ ]:
from pathlib import Path
import pandas as pd

REPO_DIR = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML"
)

FILES = {
    "mission_features": REPO_DIR / "mission_features.csv",
    "health_targets": REPO_DIR / "health_mission_features_targets.csv",
    "feature_dictionary": REPO_DIR / "feature_dictionary.csv",
}

for name, path in FILES.items():
    print("\n" + "=" * 80)
    print(f"{name}: {path.name}")
    print("=" * 80)

    if not path.exists():
        print("FILE TIDAK DITEMUKAN")
        continue

    df = pd.read_csv(path, nrows=5)
    print("Kolom:", len(df.columns))
    print("Nama kolom:")
    print(df.columns.tolist())
    print("\nContoh data:")
    display(df)

    if "cell_id" in df.columns:
        print("Jumlah sel pada sampel yang dibaca:", df["cell_id"].nunique())

# Baca ukuran penuh tanpa menampilkan seluruh isi
print("\n" + "=" * 80)
print("UKURAN DATA PENUH")
print("=" * 80)

for name in ["mission_features", "health_targets"]:
    path = FILES[name]
    if path.exists():
        df = pd.read_csv(path)
        print(
            f"{name}: baris={len(df):,}, "
            f"kolom={len(df.columns)}, "
            f"sel={df['cell_id'].nunique() if 'cell_id' in df else 'N/A'}"
        )


mission_features: mission_features.csv
Kolom: 28
Nama kolom:
['cell_id', 'mission_id', 'chg_total_dur_s', 'chg_cc_dur_s', 'chg_cv_dur_s', 'chg_energy_Wh', 'chg_cv_energy_Wh', 'chg_mean_C_rate_med', 'chg_max_C_rate_p90', 'dis_total_dur_s', 'dis_flight_Wh_abs', 'dis_mean_C_rate_med', 'dis_max_C_rate_p90', 'dis_vi_R_ohm_med', 'dis_vi_R_ohm_p90', 'dis_dSOC_sum', 'dis_dSOC_baseline_sum', 'temp_mean_C_med', 'temp_max_C_max', 'temp_th_gain_med', 'temp_th_gain_max', 'SOH_end_pct', 'RUL_med', 'chg_cv_fraction', 'cv_setpoint_V', 'chg_cv_mean_I_A_proxy', 'chg_cum_Wh', 'dis_cum_Wh']

Contoh data:


,cell_id,mission_id,chg_total_dur_s,chg_cc_dur_s,chg_cv_dur_s,chg_energy_Wh,chg_cv_energy_Wh,chg_mean_C_rate_med,chg_max_C_rate_p90,dis_total_dur_s,...,temp_max_C_max,temp_th_gain_med,temp_th_gain_max,SOH_end_pct,RUL_med,chg_cv_fraction,cv_setpoint_V,chg_cv_mean_I_A_proxy,chg_cum_Wh,dis_cum_Wh
0,VAH01,1.0,5.974510e+06,5.948405e+06,26105.098051,13126.393811,16.294201,0.397091,0.968173,40339.985674,...,61.157101,0.000147,0.002651,100.000000,249.0,0.004369,4.2,0.535009,13126.393811,49.710515
1,VAH01,2.0,6.252179e+06,6.230253e+06,21926.537740,15403.829985,14.065646,0.400618,0.968161,33079.987221,...,61.441761,0.000160,0.002696,100.000000,248.0,0.003507,4.2,0.549848,28530.223795,94.801348
2,VAH01,3.0,5.571147e+06,5.553013e+06,18134.525428,12636.932893,12.500464,0.414995,0.968129,25819.988680,...,61.188732,0.000206,0.002655,100.000000,247.0,0.003255,4.2,0.590844,41167.156689,135.312200
3,VAH01,4.0,5.573745e+06,5.554734e+06,19010.489541,10876.378674,12.618567,0.403657,0.982279,25819.988680,...,61.615715,0.000217,0.002707,98.550964,246.0,0.003411,4.2,0.568945,52043.535362,175.816704
4,VAH01,5.0,5.576161e+06,5.556811e+06,19350.331599,9049.723699,12.771543,0.409448,0.997023,25819.988788,...,61.726414,0.000215,0.002696,97.101928,245.0,0.003470,4.2,0.565729,61093.259061,216.311712


Jumlah sel pada sampel yang dibaca: 1

health_targets: health_mission_features_targets.csv
Kolom: 46
Nama kolom:
['cell_id', 'mission_id', 'cycle_instance_id', 'raw_cycle_number', 'chg_total_dur_s', 'chg_cc_dur_s', 'chg_cv_dur_s', 'chg_energy_Wh', 'chg_cc_energy_Wh', 'chg_cv_energy_Wh', 'chg_cv_fraction', 'cv_setpoint_V', 'chg_mean_I_A', 'chg_mean_C_rate', 'chg_max_C_rate', 'chg_cv_mean_I_A_proxy', 'dis_total_dur_s', 'dis_flight_Wh_abs', 'dis_mean_C_rate', 'dis_mean_C_rate_med', 'dis_max_C_rate', 'dis_max_C_rate_p90', 'dis_vi_R_ohm_med', 'dis_vi_R_ohm_p90', 'temp_mean_C', 'temp_mean_C_med', 'temp_max_C_max', 'temp_th_gain_med', 'temp_th_gain_max', 'temp_slope_med', 'gap_count', 'q0_ah_audit', 'chg_cum_Wh', 'dis_cum_Wh', 'SOH_end_pct', 'RUL90_event_mission', 'RUL90_observed', 'RUL90_missions', 'RUL85_event_mission', 'RUL85_observed', 'RUL85_missions', 'RUL80_event_mission', 'RUL80_observed', 'RUL80_missions', 'SOH_m_plus_5_pct', 'SOH_drop_5_pct']

Contoh data:


,cell_id,mission_id,cycle_instance_id,raw_cycle_number,chg_total_dur_s,chg_cc_dur_s,chg_cv_dur_s,chg_energy_Wh,chg_cc_energy_Wh,chg_cv_energy_Wh,...,RUL90_observed,RUL90_missions,RUL85_event_mission,RUL85_observed,RUL85_missions,RUL80_event_mission,RUL80_observed,RUL80_missions,SOH_m_plus_5_pct,SOH_drop_5_pct
0,VAH01,1,1,0,4955.774235,2982.268142,1973.506094,11.621741,9.458696,2.163045,...,True,321.0,638,True,637.0,NaN,False,NaN,99.652231,0.289807
1,VAH01,2,2,1,3256.370155,1426.786068,1829.584087,6.780372,4.738288,2.042083,...,True,320.0,638,True,636.0,NaN,False,NaN,99.594270,0.289807
2,VAH01,3,3,2,3068.572146,1427.290068,1641.282078,6.746749,4.739539,2.007211,...,True,319.0,638,True,635.0,NaN,False,NaN,99.536308,0.289807
3,VAH01,4,4,3,3290.862156,1437.230068,1853.632088,6.807269,4.770687,2.036582,...,True,318.0,638,True,634.0,NaN,False,NaN,99.478347,0.289807
4,VAH01,5,5,4,3278.360156,1429.334068,1849.026088,6.779168,4.745935,2.033233,...,True,317.0,638,True,633.0,NaN,False,NaN,99.420386,0.289807


Jumlah sel pada sampel yang dibaca: 1

feature_dictionary: feature_dictionary.csv
Kolom: 4
Nama kolom:
['feature', 'level', 'category', 'description']

Contoh data:


,feature,level,category,description
0,chg_is_charge_phase,phase,charging,1 if charge segment (CC/CV)
1,chg_is_CC,phase,charging,1 if CC
2,chg_is_CV,phase,charging,1 if CV
3,chg_dur_s,phase,charging,duration of charge segment
4,chg_energy_Wh,phase,charging,energy in charge segment (Wh)



UKURAN DATA PENUH
mission_features: baris=7,138, kolom=28, sel=22
health_targets: baris=18,862, kolom=46, sel=19


In [ ]:
from pathlib import Path
import pandas as pd

REPO = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML"
)

# Cari phase_features.csv di lokasi ekstraksi yang sudah digunakan sebelumnya
candidates = [
    REPO / "phase_features.csv",
    REPO / "extracted_phase_features" / "phase_features.csv",
]

phase_path = next((p for p in candidates if p.exists()), None)

# 1. Fitur misi yang direncanakan untuk Table 2B
mission_features_20 = [
    "chg_total_dur_s",
    "chg_energy_Wh",
    "chg_mean_I_A",
    "chg_mean_C_rate",
    "chg_cc_dur_s",
    "chg_cc_energy_Wh",
    "chg_cv_dur_s",
    "chg_cv_energy_Wh",
    "chg_cv_fraction",
    "cv_setpoint_V",
    "dis_total_dur_s",
    "dis_flight_Wh_abs",
    "dis_mean_C_rate",
    "dis_max_C_rate",
    "dis_vi_R_ohm_med",
    "temp_mean_C",
    "temp_max_C_max",
    "gap_count",
    "chg_cum_Wh",
    "dis_cum_Wh",
]

mission = pd.read_csv(REPO / "health_mission_features_targets.csv", nrows=1)
mission_columns = set(mission.columns)

print("=== TABLE 2B: PEMETAAN 20 FITUR MISI ===")
for i, feature in enumerate(mission_features_20, 1):
    status = "TERSEDIA" if feature in mission_columns else "TIDAK ADA"
    print(f"{i:02d}. {feature:<25} {status}")

# 2. Periksa fitur fase
print("\n=== TABLE 2A: SUMBER FITUR FASE ===")

if phase_path is None:
    print("phase_features.csv belum ditemukan pada dua lokasi yang diperiksa.")
else:
    phase = pd.read_csv(phase_path, nrows=5)
    print("Lokasi:", phase_path)
    print("Kolom:", len(phase.columns))
    print(*phase.columns.tolist(), sep="\n")

# 3. Periksa kamus fitur
dictionary_path = REPO / "feature_dictionary.csv"

if dictionary_path.exists():
    dictionary = pd.read_csv(dictionary_path)
    print("\n=== KAMUS FITUR: LEVEL PHASE ===")
    display(
        dictionary[
            dictionary["level"].astype(str).str.lower().eq("phase")
        ].reset_index(drop=True)
    )

    print("\n=== KAMUS FITUR: LEVEL MISSION ===")
    display(
        dictionary[
            dictionary["level"].astype(str).str.lower().eq("mission")
        ].reset_index(drop=True)
    )

=== TABLE 2B: PEMETAAN 20 FITUR MISI ===
01. chg_total_dur_s           TERSEDIA
02. chg_energy_Wh             TERSEDIA
03. chg_mean_I_A              TERSEDIA
04. chg_mean_C_rate           TERSEDIA
05. chg_cc_dur_s              TERSEDIA
06. chg_cc_energy_Wh          TERSEDIA
07. chg_cv_dur_s              TERSEDIA
08. chg_cv_energy_Wh          TERSEDIA
09. chg_cv_fraction           TERSEDIA
10. cv_setpoint_V             TERSEDIA
11. dis_total_dur_s           TERSEDIA
12. dis_flight_Wh_abs         TERSEDIA
13. dis_mean_C_rate           TERSEDIA
14. dis_max_C_rate            TERSEDIA
15. dis_vi_R_ohm_med          TERSEDIA
16. temp_mean_C               TERSEDIA
17. temp_max_C_max            TERSEDIA
18. gap_count                 TERSEDIA
19. chg_cum_Wh                TERSEDIA
20. dis_cum_Wh                TERSEDIA

=== TABLE 2A: SUMBER FITUR FASE ===
phase_features.csv belum ditemukan pada dua lokasi yang diperiksa.

=== KAMUS FITUR: LEVEL PHASE ===


,feature,level,category,description
0,chg_is_charge_phase,phase,charging,1 if charge segment (CC/CV)
1,chg_is_CC,phase,charging,1 if CC
2,chg_is_CV,phase,charging,1 if CV
3,chg_dur_s,phase,charging,duration of charge segment
4,chg_energy_Wh,phase,charging,energy in charge segment (Wh)
5,chg_cc_dur_s,phase,charging,CC segment duration
6,chg_cv_dur_s,phase,charging,CV segment duration
7,chg_cv_energy_Wh,phase,charging,CV segment energy (Wh)
8,chg_mean_C_rate,phase,charging,mean C-rate in charge segment
9,chg_max_C_rate,phase,charging,max C-rate in charge segment



=== KAMUS FITUR: LEVEL MISSION ===


,feature,level,category,description
0,chg_total_dur_s,mission,charging,total charge duration
1,chg_cc_dur_s,mission,charging,total CC duration
2,chg_cv_dur_s,mission,charging,total CV duration
3,chg_energy_Wh,mission,charging,total charge energy (Wh)
4,chg_cv_energy_Wh,mission,charging,total CV energy (Wh)
5,chg_cv_fraction,mission,charging,CV/(CC+CV)
6,cv_setpoint_V,mission,charging,per-cell CV setpoint (V)
7,chg_cv_mean_I_A_proxy,mission,charging,CV mean current proxy (A)
8,chg_cum_Wh,mission,charging,cumulative charge energy (Wh)
9,dis_total_dur_s,mission,discharge,total flight duration


In [ ]:
from pathlib import Path

REPO = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML"
)

print("=== FILE YANG BERKAITAN DENGAN FITUR FASE ===")

for p in REPO.rglob("*"):
    if not p.is_file():
        continue

    name = p.name.lower()

    if (
        "phase_features" in name
        or "phase" in name and p.suffix.lower() in {".rar", ".zip", ".csv", ".parquet"}
    ):
        print(f"{p}  ({p.stat().st_size:,} bytes)")

print("\n=== KANDIDAT ARSIP ===")

archives = [
    p for p in REPO.rglob("*")
    if p.is_file() and p.suffix.lower() in {".rar", ".zip"}
]

for p in archives:
    print(p)

=== FILE YANG BERKAITAN DENGAN FITUR FASE ===
/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML/mission_phase_features_importance_.png  (1,494,345 bytes)
/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML/phase_features.rar  (15,175,113 bytes)

=== KANDIDAT ARSIP ===
/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML/eda.zip
/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML/capacity_fade_overlays.zip
/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML/phase_features.rar
/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML/Baseline_mission_profiles.zip


In [ ]:
from pathlib import Path
import shutil
import subprocess
import pandas as pd

REPO = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML"
)
RAR_FILE = REPO / "phase_features.rar"
EXTRACT_DIR = REPO / "extracted_phase_features"

# 1. Pastikan alat ekstraksi tersedia
if not shutil.which("unrar") and not shutil.which("7z"):
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "unrar"], check=True)

# 2. Ekstrak jika CSV belum tersedia
phase_path = next(
    (p for p in REPO.rglob("phase_features.csv") if p.is_file()),
    None
)

if phase_path is None:
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

    if shutil.which("unrar"):
        subprocess.run(
            ["unrar", "x", "-o+", str(RAR_FILE), str(EXTRACT_DIR) + "/"],
            check=True
        )
    else:
        subprocess.run(
            ["7z", "x", str(RAR_FILE), f"-o{EXTRACT_DIR}", "-y"],
            check=True
        )

    phase_path = next(
        (p for p in EXTRACT_DIR.rglob("phase_features.csv") if p.is_file()),
        None
    )

# 3. Periksa isi aktual
if phase_path is None:
    print("CSV belum ditemukan setelah ekstraksi.")
    print("File yang berhasil diekstrak:")
    for p in EXTRACT_DIR.rglob("*"):
        if p.is_file():
            print(p)
else:
    print("CSV fase:", phase_path)

    phase = pd.read_csv(phase_path, nrows=5)
    print("Jumlah kolom:", len(phase.columns))
    print("Kolom:")
    print(*phase.columns.tolist(), sep="\n")

    print("\nContoh data:")
    display(phase)

    # Periksa apakah fitur rancangan Table 2A sudah tersedia
    phase_features_20 = [
        "ph_dur_s", "ph_energy_Wh_abs", "ph_mean_I_A",
        "ph_max_abs_I_A", "ph_mean_C_rate", "ph_max_C_rate",
        "ph_mean_P_W_abs", "ph_max_P_W_abs", "ph_V_start",
        "ph_V_end", "ph_V_mean", "ph_V_min", "ph_V_drop",
        "ph_vi_slope_ohm", "ph_T_mean_C", "ph_T_max_C",
        "ph_T_start_C", "ph_dT_C", "ph_thermal_slope_K_s",
        "ph_thermal_gain_K_W"
    ]

    actual_columns = set(phase.columns)
    print("\n=== PEMETAAN 20 FITUR TABLE 2A ===")
    for i, feature in enumerate(phase_features_20, 1):
        print(
            f"{i:02d}. {feature:<25}",
            "TERSEDIA" if feature in actual_columns else "PERLU DIPETAKAN/DIHITUNG"
        )

CSV fase: /content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML/extracted_phase_features/phase_features.csv
Jumlah kolom: 42
Kolom:
cell_id
mission_id
cycleNumber
seg_id
phase_name
phase_family
t_start_s
t_end_s
chg_is_charge_phase
chg_is_CC
chg_is_CV
chg_dur_s
chg_energy_Wh
chg_cc_dur_s
chg_cv_dur_s
chg_cv_energy_Wh
chg_mean_C_rate
chg_max_C_rate
chg_ic_centroid_V
chg_ic_spread_V
chg_ic_p1_V
chg_ic_p1_h
chg_vi_R_ohm
dis_is_flight_phase
dis_dur_s
dis_energy_Wh_abs
dis_mean_C_rate
dis_max_C_rate
dis_ic_centroid_V
dis_ic_spread_V
dis_ic_p1_V
dis_ic_p1_h
dis_vi_R_ohm
dis_dSOC
dis_dSOC_baseline
temp_mean_C
temp_max_C
temp_thermal_slope
temp_thermal_gain
SOH_mission_end_pct
RUL_missions_after_phase
RUL_missions_censored

Contoh data:


,cell_id,mission_id,cycleNumber,seg_id,phase_name,phase_family,t_start_s,t_end_s,chg_is_charge_phase,chg_is_CC,...,dis_vi_R_ohm,dis_dSOC,dis_dSOC_baseline,temp_mean_C,temp_max_C,temp_thermal_slope,temp_thermal_gain,SOH_mission_end_pct,RUL_missions_after_phase,RUL_missions_censored
0,VAH01,1.0,0,0,CC,charge,0.000000,5072.964241,1,1,...,NaN,NaN,NaN,24.565196,25.590944,0.000202,0.000035,100.0,249.0,True
1,VAH01,1.0,0,1,CV,charge,5082.964241,6947.410330,1,0,...,NaN,NaN,NaN,24.507420,25.701645,0.000637,0.000328,100.0,249.0,True
2,VAH01,1.0,0,2,CV→Rest,transition,6957.410330,7837.410372,0,0,...,NaN,NaN,NaN,22.080990,23.574633,0.001678,NaN,100.0,249.0,True
3,VAH01,1.0,0,3,Rest→TO,transition,7847.460373,11447.408544,0,0,...,NaN,NaN,NaN,23.074363,23.527191,0.000125,0.000051,100.0,249.0,True
4,VAH01,1.0,0,4,Take-off,flight,11447.410544,11477.408545,0,0,...,NaN,0.0,0.020574,23.098422,23.250443,0.005067,NaN,100.0,249.0,True



=== PEMETAAN 20 FITUR TABLE 2A ===
01. ph_dur_s                  PERLU DIPETAKAN/DIHITUNG
02. ph_energy_Wh_abs          PERLU DIPETAKAN/DIHITUNG
03. ph_mean_I_A               PERLU DIPETAKAN/DIHITUNG
04. ph_max_abs_I_A            PERLU DIPETAKAN/DIHITUNG
05. ph_mean_C_rate            PERLU DIPETAKAN/DIHITUNG
06. ph_max_C_rate             PERLU DIPETAKAN/DIHITUNG
07. ph_mean_P_W_abs           PERLU DIPETAKAN/DIHITUNG
08. ph_max_P_W_abs            PERLU DIPETAKAN/DIHITUNG
09. ph_V_start                PERLU DIPETAKAN/DIHITUNG
10. ph_V_end                  PERLU DIPETAKAN/DIHITUNG
11. ph_V_mean                 PERLU DIPETAKAN/DIHITUNG
12. ph_V_min                  PERLU DIPETAKAN/DIHITUNG
13. ph_V_drop                 PERLU DIPETAKAN/DIHITUNG
14. ph_vi_slope_ohm           PERLU DIPETAKAN/DIHITUNG
15. ph_T_mean_C               PERLU DIPETAKAN/DIHITUNG
16. ph_T_max_C                PERLU DIPETAKAN/DIHITUNG
17. ph_T_start_C              PERLU DIPETAKAN/DIHITUNG
18. ph_dT_C                  

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

REPO = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML"
)
PHASE_PATH = REPO / "extracted_phase_features" / "phase_features.csv"

df = pd.read_csv(PHASE_PATH)

# Buat tabel fitur standar sesuai rancangan Table 2A.
out = df[["cell_id", "mission_id", "seg_id", "phase_name", "phase_family"]].copy()

# Durasi berdasarkan batas waktu segmen.
out["ph_dur_s"] = (df["t_end_s"] - df["t_start_s"]).clip(lower=0)

# Energi, C-rate, dan proksi resistansi sesuai jenis fase.
is_charge = df["phase_family"].astype(str).str.lower().eq("charge")
is_flight = df["dis_is_flight_phase"].fillna(0).eq(1)

out["ph_energy_Wh_abs"] = np.nan
out.loc[is_charge, "ph_energy_Wh_abs"] = df.loc[is_charge, "chg_energy_Wh"].abs()
out.loc[is_flight, "ph_energy_Wh_abs"] = df.loc[is_flight, "dis_energy_Wh_abs"].abs()

out["ph_mean_I_A"] = np.nan
out["ph_max_abs_I_A"] = np.nan
out["ph_mean_P_W_abs"] = np.nan
out["ph_max_P_W_abs"] = np.nan
out["ph_V_start"] = np.nan
out["ph_V_end"] = np.nan
out["ph_V_mean"] = np.nan
out["ph_V_min"] = np.nan
out["ph_V_drop"] = np.nan
out["ph_T_start_C"] = np.nan
out["ph_dT_C"] = np.nan

out["ph_mean_C_rate"] = np.nan
out.loc[is_charge, "ph_mean_C_rate"] = df.loc[is_charge, "chg_mean_C_rate"]
out.loc[is_flight, "ph_mean_C_rate"] = df.loc[is_flight, "dis_mean_C_rate"]

out["ph_max_C_rate"] = np.nan
out.loc[is_charge, "ph_max_C_rate"] = df.loc[is_charge, "chg_max_C_rate"]
out.loc[is_flight, "ph_max_C_rate"] = df.loc[is_flight, "dis_max_C_rate"]

out["ph_vi_slope_ohm"] = np.nan
out.loc[is_charge, "ph_vi_slope_ohm"] = df.loc[is_charge, "chg_vi_R_ohm"]
out.loc[is_flight, "ph_vi_slope_ohm"] = df.loc[is_flight, "dis_vi_R_ohm"]

out["ph_V_mean"] = np.nan  # Tegangan mean tidak tersedia langsung.
out["ph_T_mean_C"] = df["temp_mean_C"]
out["ph_T_max_C"] = df["temp_max_C"]
out["ph_thermal_slope_K_s"] = df["temp_thermal_slope"]
out["ph_thermal_gain_K_W"] = df["temp_thermal_gain"]

feature_cols = [
    "ph_dur_s", "ph_energy_Wh_abs", "ph_mean_I_A", "ph_max_abs_I_A",
    "ph_mean_C_rate", "ph_max_C_rate", "ph_mean_P_W_abs", "ph_max_P_W_abs",
    "ph_V_start", "ph_V_end", "ph_V_mean", "ph_V_min", "ph_V_drop",
    "ph_vi_slope_ohm", "ph_T_mean_C", "ph_T_max_C", "ph_T_start_C",
    "ph_dT_C", "ph_thermal_slope_K_s", "ph_thermal_gain_K_W"
]

# Audit ketersediaan nilai untuk 20 fitur.
audit = pd.DataFrame({
    "feature": feature_cols,
    "n_non_missing": [out[c].notna().sum() for c in feature_cols],
    "n_missing": [out[c].isna().sum() for c in feature_cols],
    "status": [
        "TERSEDIA" if out[c].notna().any() else "BELUM TERSEDIA"
        for c in feature_cols
    ]
})

# Simpan hasil untuk inspeksi dan pengembangan lanjutan.
mapping_path = REPO / "table2a_feature_availability.csv"
data_path = REPO / "phase_features_20_partial.csv"

audit.to_csv(mapping_path, index=False)
out.to_csv(data_path, index=False)

print("=== AUDIT KETERSEDIAAN FITUR TABLE 2A ===")
display(audit)

print("\nFile tersimpan:")
print(mapping_path)
print(data_path)
print("\nUkuran dataset fitur:", out.shape)

=== AUDIT KETERSEDIAAN FITUR TABLE 2A ===


,feature,n_non_missing,n_missing,status
0,ph_dur_s,153556,0,TERSEDIA
1,ph_energy_Wh_abs,108378,45178,TERSEDIA
2,ph_mean_I_A,0,153556,BELUM TERSEDIA
3,ph_max_abs_I_A,0,153556,BELUM TERSEDIA
4,ph_mean_C_rate,108378,45178,TERSEDIA
5,ph_max_C_rate,108378,45178,TERSEDIA
6,ph_mean_P_W_abs,0,153556,BELUM TERSEDIA
7,ph_max_P_W_abs,0,153556,BELUM TERSEDIA
8,ph_V_start,0,153556,BELUM TERSEDIA
9,ph_V_end,0,153556,BELUM TERSEDIA



File tersimpan:
/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML/table2a_feature_availability.csv
/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML/phase_features_20_partial.csv

Ukuran dataset fitur: (153556, 25)


In [ ]:
from pathlib import Path

REPO_DIR = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML"
)

# Ekstensi file yang mungkin menyimpan data mentah atau arsip dataset
extensions = {
    ".csv", ".parquet", ".feather", ".h5", ".hdf5",
    ".mat", ".xlsx", ".xls", ".zip", ".rar", ".7z"
}

print("=== FILE DATA YANG DITEMUKAN ===")

for p in sorted(REPO_DIR.rglob("*")):
    if not p.is_file():
        continue

    # Abaikan folder lingkungan/cache agar hasil lebih relevan
    if any(part in {".git", "__pycache__", ".ipynb_checkpoints"}
           for part in p.parts):
        continue

    if p.suffix.lower() in extensions:
        print(f"{p.relative_to(REPO_DIR)} | {p.stat().st_size / 1024:.1f} KB")

=== FILE DATA YANG DITEMUKAN ===
Baseline_mission_profiles.zip | 447.8 KB
capacity_fade_overlays.zip | 1176.8 KB
capacity_tests.csv | 101.2 KB
eda.zip | 1379.3 KB
extracted_phase_features/phase_features.csv | 48077.4 KB
feature_dictionary.csv | 3.3 KB
health_mission_features_targets.csv | 11711.2 KB
mission_features.csv | 3271.7 KB
per_cycle_summary_allcells.csv | 2503.2 KB
phase_features.rar | 14819.4 KB
phase_features_20_partial.csv | 25291.9 KB
sequence_fold_metrics.csv | 5.9 KB
sequence_predictions.csv | 1248.7 KB
sequence_training_logs.csv | 36.0 KB
table2a_feature_availability.csv | 0.7 KB


In [ ]:
from pathlib import Path
import zipfile
import pandas as pd

REPO_DIR = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML"
)

# 1. Periksa isi arsip tanpa mengekstrak semua file
for archive_name in [
    "Baseline_mission_profiles.zip",
    "eda.zip",
    "capacity_fade_overlays.zip",
]:
    archive_path = REPO_DIR / archive_name
    print(f"\n{'='*70}\nARSIP: {archive_name}")

    if not archive_path.exists():
        print("File tidak ditemukan.")
        continue

    try:
        with zipfile.ZipFile(archive_path) as zf:
            entries = zf.infolist()
            print("Jumlah item:", len(entries))

            for item in entries[:50]:
                print(
                    f"{item.filename} "
                    f"({item.file_size / 1024:.1f} KB)"
                )

            if len(entries) > 50:
                print(f"... dan {len(entries)-50} item lainnya")
    except zipfile.BadZipFile:
        print("File bukan ZIP yang valid.")

# 2. Periksa struktur CSV yang relevan
for filename in [
    "capacity_tests.csv",
    "per_cycle_summary_allcells.csv",
    "health_mission_features_targets.csv",
]:
    path = REPO_DIR / filename
    print(f"\n{'='*70}\nCSV: {filename}")

    if not path.exists():
        print("File tidak ditemukan.")
        continue

    df = pd.read_csv(path, nrows=5, low_memory=False)
    print("Kolom:")
    print(df.columns.tolist())
    print("\nContoh baris:")
    print(df.head(3).to_string(index=False))


ARSIP: Baseline_mission_profiles.zip
Jumlah item: 17
Baseline_mission/ (0.0 KB)
Baseline_mission/Ns_stats___phase_labels___VAH01_csv.csv (1.7 KB)
Baseline_mission/Ns_stats___phase_labels___VAH17_csv.csv (1.7 KB)
Baseline_mission/Ns_stats___phase_labels___VAH27_csv.csv (1.7 KB)
Baseline_mission/Per-cycle_capacity___SOH___VAH01_csv.csv (27.5 KB)
Baseline_mission/Per-cycle_capacity___SOH___VAH17_csv.csv (53.1 KB)
Baseline_mission/Phase_label_map_per_file.csv (0.9 KB)
Baseline_mission/SOH_RUL_headline_summary.csv (0.2 KB)
Baseline_mission/VAH01_capacity_vs_cycle.png (87.5 KB)
Baseline_mission/VAH01_per_cycle_capacity_soh.csv (27.5 KB)
Baseline_mission/VAH01_SOH_vs_cycle.png (69.3 KB)
Baseline_mission/VAH17_capacity_vs_cycle.png (89.1 KB)
Baseline_mission/VAH17_per_cycle_capacity_soh.csv (53.1 KB)
Baseline_mission/VAH17_SOH_vs_cycle.png (69.7 KB)
Baseline_mission/VAH27_capacity_vs_cycle.png (98.6 KB)
Baseline_mission/VAH27_per_cycle_capacity_soh.csv (27.5 KB)
Baseline_mission/VAH27_SOH_vs_

In [ ]:
from pathlib import Path
import zipfile
import pandas as pd

REPO_DIR = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML"
)
archive = REPO_DIR / "Baseline_mission_profiles.zip"

with zipfile.ZipFile(archive) as zf:
    for name in zf.namelist():
        if "Ns_stats" in name or "Phase_label_map" in name:
            print(f"\n{'='*70}\nFILE: {name}")
            with zf.open(name) as f:
                df = pd.read_csv(f)
            print("Shape:", df.shape)
            print("Kolom:", df.columns.tolist())
            print(df.head(5).to_string(index=False))


FILE: Baseline_mission/Ns_stats___phase_labels___VAH01_csv.csv
Shape: (10, 13)
Kolom: ['Ns', 'n_rows', 'duration_s', 'I_mA_mean', 'I_mA_median', 'I_mA_std', 'frac_rest', 'Ecell_V_mean', 'Ecell_V_max', 'Ecell_V_min', 'dVdt_mean', 'dVdt_median', 'phase_label']
 Ns  n_rows   duration_s     I_mA_mean  I_mA_median    I_mA_std  frac_rest  Ecell_V_mean  Ecell_V_max  Ecell_V_min     dVdt_mean  dVdt_median            phase_label
  0   49762 6.349314e+06   2790.282931   2988.90670  508.461602   0.000000      3.961543     4.200089     3.181843  3.016387e-04     0.000299              charge_CV
  1   78905 6.348386e+06    678.672993    385.72446  666.975144   0.000000      4.199808     4.200207     4.198750 -3.212770e-07     0.000000              charge_CV
  2    1602 6.347401e+06      0.000000      0.00000    0.000000   1.000000      4.182638     4.197135     4.167678 -2.483335e-05    -0.000016                   rest
  3   88859 6.350121e+06   -445.113416   -609.60364  271.086423   0.270552      

In [ ]:
from pathlib import Path
import zipfile
import pandas as pd
import numpy as np

REPO_DIR = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML"
)

archive_path = REPO_DIR / "Baseline_mission_profiles.zip"

# --------------------------------------------------
# 1. Baca statistik fase dari arsip
# --------------------------------------------------
stats = []

with zipfile.ZipFile(archive_path) as zf:
    for name in zf.namelist():
        if "Ns_stats___phase_labels___" not in name:
            continue

        with zf.open(name) as f:
            df = pd.read_csv(f)

        cell = Path(name).stem.split("___")[-1].replace("_csv", "")
        df["cell_id"] = cell
        stats.append(df)

stats_df = pd.concat(stats, ignore_index=True)

print("=== RINGKASAN STATISTIK FASE ===")
print("Shape:", stats_df.shape)
print("\nJumlah baris per sel:")
print(stats_df.groupby("cell_id").size())

print("\nLabel fase:")
print(stats_df["phase_label"].value_counts(dropna=False))

print("\nDurasi fase:")
print(stats_df["duration_s"].describe())

print("\nArus rata-rata (kolom sumber I_mA_mean):")
print(stats_df["I_mA_mean"].describe())

# --------------------------------------------------
# 2. Periksa satuan dan rentang statistik arus
# --------------------------------------------------
stats_df["I_mean_A_if_mA"] = stats_df["I_mA_mean"] / 1000.0

print("\nArus rata-rata jika I_mA_mean memang dalam mA:")
print(stats_df["I_mean_A_if_mA"].describe())

# --------------------------------------------------
# 3. Bandingkan label fase dengan phase_features.csv
# --------------------------------------------------
phase_path = (
    REPO_DIR / "extracted_phase_features" / "phase_features.csv"
)

if phase_path.exists():
    phase_df = pd.read_csv(phase_path, low_memory=False)

    print("\n=== LABEL FASE DI phase_features.csv ===")
    for col in ["phase_name", "phase_family"]:
        if col in phase_df.columns:
            print(f"\n{col}:")
            print(phase_df[col].value_counts(dropna=False).head(30))

    print("\nJumlah sel yang tercakup:")
    if "cell_id" in phase_df.columns:
        print("phase_features:", phase_df["cell_id"].nunique())
        print("stats:", stats_df["cell_id"].nunique())

    # Tampilkan pasangan label yang benar-benar teramati.
    print("\nContoh fase dari phase_features.csv:")
    cols = [
        c for c in [
            "cell_id", "mission_id", "phase_name",
            "phase_family", "t_start_s", "t_end_s",
            "chg_dur_s", "dis_dur_s"
        ]
        if c in phase_df.columns
    ]
    print(phase_df[cols].head(15).to_string(index=False))
else:
    print("\nphase_features.csv tidak ditemukan di lokasi yang diharapkan.")

=== RINGKASAN STATISTIK FASE ===
Shape: (30, 14)

Jumlah baris per sel:
cell_id
VAH01    10
VAH17    10
VAH27    10
dtype: int64

Label fase:
phase_label
rest                      9
discharge_cruise_like     6
discharge_landing_like    5
charge_CV                 3
discharge_takeoff_like    3
charge_other              2
charge_CC                 2
Name: count, dtype: int64

Durasi fase:
count    3.000000e+01
mean     6.025625e+06
std      1.457975e+06
min      3.677745e+06
25%      4.224025e+06
50%      6.347893e+06
75%      7.697222e+06
max      7.705183e+06
Name: duration_s, dtype: float64

Arus rata-rata (kolom sumber I_mA_mean):
count       30.000000
mean     -3039.862591
std       6438.343289
min     -17224.757269
25%      -1885.794683
50%        -82.768268
75%          0.000000
max       2916.922604
Name: I_mA_mean, dtype: float64

Arus rata-rata jika I_mA_mean memang dalam mA:
count    30.000000
mean     -3.039863
std       6.438343
min     -17.224757
25%      -1.885795
50%     

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

REPO_DIR = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML"
)

PHASE_PATH = (
    REPO_DIR / "extracted_phase_features" / "phase_features.csv"
)

df = pd.read_csv(PHASE_PATH, low_memory=False)

print("Shape:", df.shape)
print("Jumlah sel:", df["cell_id"].nunique())
print("Jumlah misi:", df["mission_id"].nunique())

# --------------------------------------------------
# 1. Periksa kolom kandidat
# --------------------------------------------------
candidate_cols = [
    "cell_id",
    "mission_id",
    "seg_id",
    "phase_name",
    "phase_family",
    "t_start_s",
    "t_end_s",
    "chg_dur_s",
    "dis_dur_s",
    "chg_energy_Wh",
    "dis_energy_Wh_abs",
    "chg_mean_C_rate",
    "chg_max_C_rate",
    "dis_mean_C_rate",
    "dis_max_C_rate",
    "chg_vi_R_ohm",
    "dis_vi_R_ohm",
    "temp_mean_C",
    "temp_max_C",
    "temp_thermal_slope",
    "temp_thermal_gain",
    "dis_dSOC",
    "SOH_mission_end_pct",
    "RUL_missions_after_phase",
    "RUL_missions_censored",
]

available = [c for c in candidate_cols if c in df.columns]
missing = [c for c in candidate_cols if c not in df.columns]

print("\nKolom kandidat yang tersedia:")
print(available)

print("\nKolom kandidat yang tidak tersedia:")
print(missing)

# --------------------------------------------------
# 2. Persentase nilai kosong pada kolom kandidat
# --------------------------------------------------
print("\nPersentase nilai kosong:")
missing_pct = (
    df[available]
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)
print(missing_pct.round(2).to_string())

# --------------------------------------------------
# 3. Ringkasan per jenis fase
# --------------------------------------------------
numeric_candidates = [
    c for c in [
        "chg_dur_s",
        "dis_dur_s",
        "chg_energy_Wh",
        "dis_energy_Wh_abs",
        "chg_mean_C_rate",
        "dis_mean_C_rate",
        "chg_vi_R_ohm",
        "dis_vi_R_ohm",
        "temp_mean_C",
        "temp_max_C",
        "temp_thermal_slope",
        "temp_thermal_gain",
    ]
    if c in df.columns
]

print("\nRingkasan median per phase_family:")
print(
    df.groupby("phase_family")[numeric_candidates]
      .median()
      .round(4)
      .to_string()
)

# --------------------------------------------------
# 4. Jumlah nilai valid untuk fitur charge/discharge
# --------------------------------------------------
for family in ["charge", "flight", "transition", "rest"]:
    sub = df[df["phase_family"] == family]

    print(f"\n--- {family}: {len(sub)} baris ---")

    cols = [
        c for c in [
            "phase_name",
            "chg_energy_Wh",
            "dis_energy_Wh_abs",
            "chg_mean_C_rate",
            "dis_mean_C_rate",
            "chg_vi_R_ohm",
            "dis_vi_R_ohm",
            "temp_mean_C",
            "temp_max_C",
        ]
        if c in sub.columns
    ]

    if cols:
        print(sub[cols].head(5).to_string(index=False))

# --------------------------------------------------
# 5. Validasi batas waktu
# --------------------------------------------------
if {"t_start_s", "t_end_s"}.issubset(df.columns):
    df["duration_from_bounds_s"] = df["t_end_s"] - df["t_start_s"]

    print("\nDurasi berdasarkan batas waktu:")
    print(df["duration_from_bounds_s"].describe().round(4))

    print(
        "\nJumlah durasi negatif:",
        int((df["duration_from_bounds_s"] < 0).sum())
    )

    print(
        "Jumlah durasi nol:",
        int((df["duration_from_bounds_s"] == 0).sum())
    )

    if "chg_dur_s" in df.columns:
        mask = (
            df["phase_family"].eq("charge")
            & df["chg_dur_s"].notna()
            & df["duration_from_bounds_s"].notna()
        )

        print(
            "\nPerbandingan durasi charge dari batas waktu "
            "dan kolom chg_dur_s:"
        )
        print(
            df.loc[
                mask,
                ["phase_name", "duration_from_bounds_s", "chg_dur_s"]
            ].head(10).to_string(index=False)
        )

Shape: (153556, 42)
Jumlah sel: 22
Jumlah misi: 500

Kolom kandidat yang tersedia:
['cell_id', 'mission_id', 'seg_id', 'phase_name', 'phase_family', 't_start_s', 't_end_s', 'chg_dur_s', 'dis_dur_s', 'chg_energy_Wh', 'dis_energy_Wh_abs', 'chg_mean_C_rate', 'chg_max_C_rate', 'dis_mean_C_rate', 'dis_max_C_rate', 'chg_vi_R_ohm', 'dis_vi_R_ohm', 'temp_mean_C', 'temp_max_C', 'temp_thermal_slope', 'temp_thermal_gain', 'dis_dSOC', 'SOH_mission_end_pct', 'RUL_missions_after_phase', 'RUL_missions_censored']

Kolom kandidat yang tidak tersedia:
[]

Persentase nilai kosong:
chg_vi_R_ohm                86.05
dis_vi_R_ohm                72.33
chg_max_C_rate              71.43
chg_mean_C_rate             71.43
dis_max_C_rate              57.99
dis_mean_C_rate             57.99
dis_dSOC                    57.99
temp_thermal_gain           29.37
mission_id                   0.00
RUL_missions_after_phase     0.00
chg_dur_s                    0.00
t_end_s                      0.00
t_start_s              

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

REPO_DIR = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML"
)
PHASE_PATH = REPO_DIR / "extracted_phase_features" / "phase_features.csv"

df = pd.read_csv(PHASE_PATH, low_memory=False)

df["duration_bounds_s"] = df["t_end_s"] - df["t_start_s"]

# Pilih kolom durasi sumber sesuai kelompok fase.
df["duration_source_s"] = np.select(
    [
        df["phase_family"].eq("charge"),
        df["phase_family"].eq("flight"),
    ],
    [
        df["chg_dur_s"],
        df["dis_dur_s"],
    ],
    default=np.nan,
)

df["duration_ratio"] = (
    df["duration_source_s"]
    / df["duration_bounds_s"].replace(0, np.nan)
)

# Baris yang berbeda lebih dari 10 kali.
suspect = df[
    df["duration_ratio"].gt(10)
    | df["duration_ratio"].lt(0.1)
].copy()

cols = [
    "cell_id", "mission_id", "seg_id", "phase_name",
    "phase_family", "t_start_s", "t_end_s",
    "duration_bounds_s", "duration_source_s", "duration_ratio",
]

print("Jumlah baris yang perlu diaudit:", len(suspect))
print(
    suspect[cols]
    .sort_values("duration_ratio", ascending=False)
    .head(30)
    .to_string(index=False)
)

print("\nRingkasan rasio durasi per kelompok fase:")
print(
    df.groupby("phase_family")["duration_ratio"]
      .describe()
      .round(3)
)

Jumlah baris yang perlu diaudit: 14194
cell_id  mission_id  seg_id phase_name phase_family    t_start_s      t_end_s  duration_bounds_s  duration_source_s  duration_ratio
  VAH09         1.0      80         CC       charge 5.256684e+06 5.256744e+06          60.000003       9.816863e+05    16361.436780
  VAH11       255.0      14         CC       charge 1.603314e+07 1.603417e+07        1034.875198       4.844320e+06     4681.066526
  VAH11       325.0      14         CC       charge 1.665257e+07 1.665362e+07        1045.277190       4.884352e+06     4672.781548
  VAH11       268.0      14         CC       charge 1.614243e+07 1.614347e+07        1038.745195       4.851347e+06     4670.391372
  VAH11       333.0      14         CC       charge 1.672116e+07 1.672220e+07        1048.215188       4.889350e+06     4664.452707
  VAH11       332.0      14         CC       charge 1.671260e+07 1.671365e+07        1049.231187       4.888753e+06     4659.366379
  VAH11       253.0      14         C

In [ ]:
from pathlib import Path

REPO_DIR = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML"
)

keywords = [
    "chg_dur_s",
    "dis_dur_s",
    "dis_dSOC",
    "temp_thermal_slope",
    "temp_thermal_gain",
    "chg_vi_R_ohm",
    "dis_vi_R_ohm",
]

extensions = {".py", ".md", ".json", ".yaml", ".yml"}

for keyword in keywords:
    print(f"\n{'=' * 20} {keyword} {'=' * 20}")
    matches = 0

    for path in REPO_DIR.rglob("*"):
        if not path.is_file() or path.suffix.lower() not in extensions:
            continue

        if any(
            part in {"extracted_phase_features", ".git", "__pycache__"}
            for part in path.parts
        ):
            continue

        try:
            lines = path.read_text(
                encoding="utf-8", errors="ignore"
            ).splitlines()
        except Exception:
            continue

        for line_no, line in enumerate(lines, start=1):
            if keyword.lower() in line.lower():
                print(
                    f"{path.relative_to(REPO_DIR)}:{line_no}: "
                    f"{line.strip()[:220]}"
                )
                matches += 1

                if matches >= 10:
                    print("Hasil dibatasi hingga 10 kecocokan.")
                    break

        if matches >= 10:
            break

    if matches == 0:
        print("Tidak ditemukan.")


==================== chg_dur_s ====================
Tidak ditemukan.

==================== dis_dur_s ====================
Dataset_ processing stage.py:136: f = pd.DataFrame(feats, columns=['cycleNumber','frac_flight','dis_dur_s','quasi_cc','QDischarge_mAh','vmin','vdrop'])
Dataset_ processing stage.py:139: mission_dur_med = f.loc[f['frac_flight'] > 0.2, 'dis_dur_s'].median()
Dataset_ processing stage.py:145: S2 = (f['dis_dur_s'] > max(1200.0, 1.8 * mission_dur_med))
Dataset_ processing stage.py:158: out = cycle_stats.merge(f[['cycleNumber','is_capacity_test','score','dis_dur_s','frac_flight']],
Dataset_ processing stage.py:287: top = cs.sort_values('score', ascending=False).head(3)[['cycleNumber','score','dis_dur_s','frac_flight','QDischarge_mAh']]

==================== dis_dSOC ====================
preprocessing_and_targets.md:197: For a phase, `dis_dSOC = SOC_end - SOC_start`. Capacity and SOH used for this
preprocessing_and_targets.md:199: allow-list. Its mission aggregate, `dis_dS

In [ ]:
from pathlib import Path

REPO_DIR = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML"
)

def show_lines(filename, start, end):
    path = REPO_DIR / filename

    print(f"\n{'=' * 80}")
    print(f"FILE: {filename} | BARIS {start}-{end}")
    print("=" * 80)

    if not path.exists():
        print("File tidak ditemukan:", path)
        return

    lines = path.read_text(
        encoding="utf-8", errors="ignore"
    ).splitlines()

    for i in range(start, min(end, len(lines)) + 1):
        print(f"{i:4d}: {lines[i - 1]}")

# Dokumentasi definisi target dan window SOC
show_lines("preprocessing_and_targets.md", 185, 230)

# Cara menghitung agregat fitur misi
show_lines("reconstruct.py", 120, 190)

# Bagian awal pipeline rekonstruksi
show_lines("reconstruct.py", 1, 120)

# Pencarian asal pembentukan fitur fase
for filename in [
    "Feature engineering stage(features_extractions).py",
    "Phase labeling Stage.py",
    "Dataset_ processing stage.py",
]:
    path = REPO_DIR / filename
    print(f"\n{'=' * 80}\nFILE: {filename}")

    if not path.exists():
        print("File tidak ditemukan.")
        continue

    lines = path.read_text(
        encoding="utf-8", errors="ignore"
    ).splitlines()

    hits = [
        i for i, line in enumerate(lines)
        if any(
            key in line.lower()
            for key in [
                "duration_s",
                "chg_dur",
                "thermal_slope",
                "thermal_gain",
                "vi_r_ohm",
                "phase_features",
            ]
        )
    ]

    if not hits:
        print("Tidak ada istilah yang cocok.")
        continue

    shown = set()
    for i in hits[:12]:
        for j in range(max(0, i - 3), min(len(lines), i + 4)):
            if j not in shown:
                print(f"{j + 1:4d}: {lines[j]}")
                shown.add(j)
        print("-" * 40)


FILE: preprocessing_and_targets.md | BARIS 185-230
 185: from being trained as though failure occurred at data collection end.
 186: 
 187: ## 6. SOC labels
 188: 
 189: Each mission begins fully charged under the published protocol. Telemetry SOC
 190: is coulomb-counted using the mission-specific interpolated capacity:
 191: 
 192: \[
 193: SOC_i=\operatorname{clip}_{[0,100]}\left(100+
 194: 100\frac{\sum_{k\le i} I_kdt_k/3600}{Q_m}\right).
 195: \]
 196: 
 197: For a phase, `dis_dSOC = SOC_end - SOC_start`. Capacity and SOH used for this
 198: calculation are target-construction data and are excluded from the predictor
 199: allow-list. Its mission aggregate, `dis_dSOC_sum`, is also labelled as target
 200: context and cannot enter the SOH or RUL predictor matrix. The separate
 201: `dis_dSOC_baseline` uses beginning-of-life capacity and remains an admissible
 202: physics-based predictor.
 203: 
 204: ## 7. Sliding-window samples
 205: 
 206: All windows use stride 1, left padding

In [ ]:
from pathlib import Path
import ast

REPO_DIR = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML"
)

reconstruct_path = REPO_DIR / "reconstruct.py"

assert reconstruct_path.exists(), (
    f"File tidak ditemukan: {reconstruct_path}"
)

source = reconstruct_path.read_text(encoding="utf-8")
tree = ast.parse(source)

print("FILE:", reconstruct_path)
print("\nFungsi yang tersedia:")

for node in tree.body:
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
        print(f"- {node.name} (baris {node.lineno})")

print("\nKolom telemetri yang diwajibkan:")

for node in tree.body:
    if isinstance(node, ast.Assign):
        for target in node.targets:
            if isinstance(target, ast.Name) and target.id == "REQUIRED":
                print(ast.literal_eval(node.value))

print("\nFungsi pembentukan fitur fase ditemukan:",
      "_phase_table" in source)

print("Fungsi pembentukan fitur misi ditemukan:",
      "_mission_row" in source)

FILE: /content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML/reconstruct.py

Fungsi yang tersedia:
- sha256 (baris 38)
- ordered_mission (baris 46)
- _safe_percentile (baris 59)
- _phase_table (baris 64)
- _mission_row (baris 119)
- process_cell (baris 181)
- run (baris 391)
- main (baris 456)

Kolom telemetri yang diwajibkan:
['time_s', 'Ecell_V', 'I_mA', 'QCharge_mA_h', 'QDischarge_mA_h', 'Temperature__C', 'cycleNumber', 'Ns']

Fungsi pembentukan fitur fase ditemukan: True
Fungsi pembentukan fitur misi ditemukan: True


In [ ]:
from pathlib import Path

REPO_DIR = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML"
)

def show_lines(filename, start, end):
    path = REPO_DIR / filename

    print(f"\n{'=' * 90}")
    print(f"FILE: {filename} | BARIS {start}-{end}")
    print("=" * 90)

    if not path.exists():
        print(f"File tidak ditemukan: {path}")
        return

    lines = path.read_text(
        encoding="utf-8",
        errors="replace"
    ).splitlines()

    for i in range(start, min(end, len(lines)) + 1):
        print(f"{i:4d}: {lines[i - 1]}")


# Periksa pembentukan siklus, batas fase, dan fitur
show_lines("reconstruct.py", 181, 300)

# Periksa pembentukan target dan keluaran pipeline
show_lines("reconstruct.py", 301, 390)

# Periksa konfigurasi, argumen, dan eksekusi pipeline
show_lines("reconstruct.py", 391, 490)


FILE: reconstruct.py | BARIS 181-300
 181: def process_cell(path: Path) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
 182:     cell_id = path.stem.upper()
 183:     raw = pd.read_csv(path, usecols=REQUIRED, low_memory=False)
 184:     for column in REQUIRED:
 185:         raw[column] = pd.to_numeric(raw[column], errors="raise")
 186:     raw["source_row"] = np.arange(len(raw), dtype=np.int64)
 187: 
 188:     new_cycle = (
 189:         raw["cycleNumber"].ne(raw["cycleNumber"].shift())
 190:         | raw["time_s"].lt(raw["time_s"].shift())
 191:     )
 192:     raw["cycle_instance_id"] = new_cycle.cumsum().astype(np.int64)
 193:     new_phase = new_cycle | raw["Ns"].ne(raw["Ns"].shift())
 194:     raw["phase_run_id"] = new_phase.cumsum().astype(np.int64)
 195: 
 196:     dt_raw = raw["time_s"].diff()
 197:     first_phase_row = raw["phase_run_id"].ne(raw["phase_run_id"].shift())
 198:     raw["large_gap"] = (~first_phase_row) & dt_raw.gt(60.0)
 199:     raw["dt_s"] = dt_raw.mask(first

In [ ]:
from pathlib import Path
import pandas as pd

REPO_DIR = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML"
)

# Cari file mentah VAH*.csv di beberapa lokasi yang mungkin.
search_roots = [
    REPO_DIR,
    REPO_DIR / "data",
    REPO_DIR / "dataset",
    Path("/content"),
]

raw_files = {}

for root in search_roots:
    if root.exists():
        for path in root.glob("VAH*.csv"):
            raw_files[str(path.resolve())] = path.resolve()

print("Jumlah file VAH*.csv ditemukan:", len(raw_files))

for path in sorted(raw_files.values())[:30]:
    print(path)

Jumlah file VAH*.csv ditemukan: 0


In [ ]:
from pathlib import Path

REPO_DIR = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML"
)

for root in [REPO_DIR, Path("/content")]:
    print(f"\n=== {root} ===")
    if not root.exists():
        print("Folder tidak ditemukan")
        continue

    for pattern in ["*.zip", "*.rar", "*.csv"]:
        matches = list(root.glob(pattern))
        print(f"\n{pattern}: {len(matches)} file")
        for p in matches[:50]:
            print(" ", p)


=== /content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML ===

*.zip: 3 file
  /content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML/eda.zip
  /content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML/capacity_fade_overlays.zip
  /content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML/Baseline_mission_profiles.zip

*.rar: 1 file
  /content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML/phase_features.rar

*.csv: 10 file
  /content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML/sequence_fold_metrics.csv
  /content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML/health_mission_features_targets.csv
  /content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML/capacity_tests.csv
  /content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML/mission_features.csv
  /content/Mission-Phase-Feature-Learning

In [ ]:
from pathlib import Path
import pandas as pd

REPO_DIR = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML"
)

files = [
    "table2a_feature_availability.csv",
    "phase_features_20_partial.csv",
    "feature_dictionary.csv",
]

for filename in files:
    path = REPO_DIR / filename

    print("\n" + "=" * 90)
    print("FILE:", filename)
    print("=" * 90)

    if not path.exists():
        print("File tidak ditemukan.")
        continue

    df = pd.read_csv(path, low_memory=False)

    print("Shape:", df.shape)
    print("\nKolom:")
    print(df.columns.tolist())

    print("\nLima baris pertama:")
    display(df.head())

    print("\nTipe data:")
    print(df.dtypes)


FILE: table2a_feature_availability.csv
Shape: (20, 4)

Kolom:
['feature', 'n_non_missing', 'n_missing', 'status']

Lima baris pertama:


,feature,n_non_missing,n_missing,status
0,ph_dur_s,153556,0,TERSEDIA
1,ph_energy_Wh_abs,108378,45178,TERSEDIA
2,ph_mean_I_A,0,153556,BELUM TERSEDIA
3,ph_max_abs_I_A,0,153556,BELUM TERSEDIA
4,ph_mean_C_rate,108378,45178,TERSEDIA



Tipe data:
feature          object
n_non_missing     int64
n_missing         int64
status           object
dtype: object

FILE: phase_features_20_partial.csv
Shape: (153556, 25)

Kolom:
['cell_id', 'mission_id', 'seg_id', 'phase_name', 'phase_family', 'ph_dur_s', 'ph_energy_Wh_abs', 'ph_mean_I_A', 'ph_max_abs_I_A', 'ph_mean_P_W_abs', 'ph_max_P_W_abs', 'ph_V_start', 'ph_V_end', 'ph_V_mean', 'ph_V_min', 'ph_V_drop', 'ph_T_start_C', 'ph_dT_C', 'ph_mean_C_rate', 'ph_max_C_rate', 'ph_vi_slope_ohm', 'ph_T_mean_C', 'ph_T_max_C', 'ph_thermal_slope_K_s', 'ph_thermal_gain_K_W']

Lima baris pertama:


,cell_id,mission_id,seg_id,phase_name,phase_family,ph_dur_s,ph_energy_Wh_abs,ph_mean_I_A,ph_max_abs_I_A,ph_mean_P_W_abs,...,ph_V_drop,ph_T_start_C,ph_dT_C,ph_mean_C_rate,ph_max_C_rate,ph_vi_slope_ohm,ph_T_mean_C,ph_T_max_C,ph_thermal_slope_K_s,ph_thermal_gain_K_W
0,VAH01,1.0,0,CC,charge,5072.964241,8.202658,NaN,NaN,NaN,...,NaN,NaN,NaN,0.482842,0.483708,NaN,24.565196,25.590944,0.000202,0.000035
1,VAH01,1.0,1,CV,charge,1864.446089,1.014242,NaN,NaN,NaN,...,NaN,NaN,NaN,0.149744,0.471464,0.000021,24.507420,25.701645,0.000637,0.000328
2,VAH01,1.0,2,CV→Rest,transition,880.000042,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,22.080990,23.574633,0.001678,NaN
3,VAH01,1.0,3,Rest→TO,transition,3599.948171,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,23.074363,23.527191,0.000125,0.000051
4,VAH01,1.0,4,Take-off,flight,29.998001,0.002305,NaN,NaN,NaN,...,NaN,NaN,NaN,0.024689,0.097201,NaN,23.098422,23.250443,0.005067,NaN



Tipe data:
cell_id                  object
mission_id              float64
seg_id                    int64
phase_name               object
phase_family             object
ph_dur_s                float64
ph_energy_Wh_abs        float64
ph_mean_I_A             float64
ph_max_abs_I_A          float64
ph_mean_P_W_abs         float64
ph_max_P_W_abs          float64
ph_V_start              float64
ph_V_end                float64
ph_V_mean               float64
ph_V_min                float64
ph_V_drop               float64
ph_T_start_C            float64
ph_dT_C                 float64
ph_mean_C_rate          float64
ph_max_C_rate           float64
ph_vi_slope_ohm         float64
ph_T_mean_C             float64
ph_T_max_C              float64
ph_thermal_slope_K_s    float64
ph_thermal_gain_K_W     float64
dtype: object

FILE: feature_dictionary.csv
Shape: (55, 4)

Kolom:
['feature', 'level', 'category', 'description']

Lima baris pertama:


,feature,level,category,description
0,chg_is_charge_phase,phase,charging,1 if charge segment (CC/CV)
1,chg_is_CC,phase,charging,1 if CC
2,chg_is_CV,phase,charging,1 if CV
3,chg_dur_s,phase,charging,duration of charge segment
4,chg_energy_Wh,phase,charging,energy in charge segment (Wh)



Tipe data:
feature        object
level          object
category       object
description    object
dtype: object


In [ ]:
from pathlib import Path
import pandas as pd

REPO_DIR = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML"
)

# 1. Audit seluruh 20 fitur Table 2A
audit = pd.read_csv(
    REPO_DIR / "table2a_feature_availability.csv"
)

print("=== AUDIT TABLE 2A ===")
display(
    audit.sort_values(
        ["status", "n_missing"],
        ascending=[True, False]
    ).to_string(index=False)
)

print("\nRingkasan status:")
print(audit["status"].value_counts(dropna=False).to_string())

# 2. Hitung ulang missing value dari tabel fitur parsial
phase_path = REPO_DIR / "phase_features_20_partial.csv"
phase = pd.read_csv(phase_path, low_memory=False)

feature_cols = [
    c for c in phase.columns
    if c not in [
        "cell_id", "mission_id", "seg_id",
        "phase_name", "phase_family"
    ]
]

missing = pd.DataFrame({
    "feature": feature_cols,
    "n_non_missing": phase[feature_cols].notna().sum().values,
    "n_missing": phase[feature_cols].isna().sum().values,
    "missing_pct": (
        phase[feature_cols].isna().mean().values * 100
    ).round(2)
})

print("\n=== MISSING VALUE AKTUAL ===")
display(
    missing.sort_values("missing_pct", ascending=False)
    .to_string(index=False)
)

# 3. Cari skrip rekonstruksi dan referensi ke file telemetry mentah
print("\n=== SKRIP YANG RELEVAN ===")

for p in REPO_DIR.rglob("*.py"):
    name = p.name.lower()
    if any(k in name for k in [
        "reconstruct", "feature", "phase", "dataset", "pipeline"
    ]):
        print(p)

print("\n=== ARSIP DATA YANG TERSEDIA ===")
for pattern in ["*.zip", "*.rar", "*.7z", "*.tar", "*.gz"]:
    for p in REPO_DIR.rglob(pattern):
        print(p)

=== AUDIT TABLE 2A ===


'             feature  n_non_missing  n_missing         status\n         ph_mean_I_A              0     153556 BELUM TERSEDIA\n      ph_max_abs_I_A              0     153556 BELUM TERSEDIA\n     ph_mean_P_W_abs              0     153556 BELUM TERSEDIA\n      ph_max_P_W_abs              0     153556 BELUM TERSEDIA\n          ph_V_start              0     153556 BELUM TERSEDIA\n            ph_V_end              0     153556 BELUM TERSEDIA\n           ph_V_mean              0     153556 BELUM TERSEDIA\n            ph_V_min              0     153556 BELUM TERSEDIA\n           ph_V_drop              0     153556 BELUM TERSEDIA\n        ph_T_start_C              0     153556 BELUM TERSEDIA\n             ph_dT_C              0     153556 BELUM TERSEDIA\n     ph_vi_slope_ohm          63911      89645       TERSEDIA\n    ph_energy_Wh_abs         108378      45178       TERSEDIA\n      ph_mean_C_rate         108378      45178       TERSEDIA\n       ph_max_C_rate         108378      45178       T


Ringkasan status:
status
BELUM TERSEDIA    11
TERSEDIA           9

=== MISSING VALUE AKTUAL ===


'             feature  n_non_missing  n_missing  missing_pct\n      ph_max_abs_I_A              0     153556       100.00\n         ph_mean_I_A              0     153556       100.00\n      ph_max_P_W_abs              0     153556       100.00\n     ph_mean_P_W_abs              0     153556       100.00\n             ph_dT_C              0     153556       100.00\n            ph_V_min              0     153556       100.00\n          ph_V_start              0     153556       100.00\n            ph_V_end              0     153556       100.00\n           ph_V_mean              0     153556       100.00\n        ph_T_start_C              0     153556       100.00\n           ph_V_drop              0     153556       100.00\n     ph_vi_slope_ohm          63911      89645        58.38\n      ph_mean_C_rate         108378      45178        29.42\n    ph_energy_Wh_abs         108378      45178        29.42\n       ph_max_C_rate         108378      45178        29.42\n ph_thermal_gain_K_W   


=== SKRIP YANG RELEVAN ===
/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML/Feature engineering stage(features_extractions).py
/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML/build_reproducible_pipeline.py
/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML/reconstruct.py
/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML/Phase labeling Stage.py
/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML/pipeline.py
/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML/Dataset_ processing stage.py
/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML/Capacity_test_pipeline.py

=== ARSIP DATA YANG TERSEDIA ===
/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML/eda.zip
/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML/capacity_fade_overlays.zip
/content/Miss

In [ ]:
from pathlib import Path

RAW_DIR = Path("/content/evtol_raw")

raw_files = sorted(
    p for p in RAW_DIR.glob("VAH*.csv")
    if "_impedance" not in p.name.lower()
)

print("Jumlah file telemetry mentah:", len(raw_files))
print("Contoh file:", [p.name for p in raw_files[:10]])

if raw_files:
    import pandas as pd
    sample = pd.read_csv(raw_files[0], nrows=5)
    print("File contoh:", raw_files[0])
    print("Kolom:", sample.columns.tolist())
    display(sample)

Jumlah file telemetry mentah: 0
Contoh file: []


In [ ]:
if phase is not None:
    id_cols = {'cell_id', 'mission_id', 'cycleNumber', 'seg_id'}
    completeness_rows = []
    for col in phase.columns:
        if col in id_cols:
            continue
        present = int(phase[col].notna().sum())
        missing = int(len(phase) - present)
        completeness_rows.append({
            'Feature': col,
            'Rows': len(phase),
            'Non-missing': present,
            'Missing': missing,
            'Non-missing (%)': round(100 * present / len(phase), 2) if len(phase) else 0
        })
    phase_completeness = pd.DataFrame(completeness_rows)
    display(phase_completeness.sort_values('Non-missing (%)'))
else:
    phase_completeness = pd.DataFrame()
    print('phase_features.csv tidak tersedia; audit kelengkapan dilewati.')

,Feature,Rows,Non-missing,Missing,Non-missing (%)
15,chg_ic_spread_V,153556,21387,132169,13.93
14,chg_ic_centroid_V,153556,21387,132169,13.93
17,chg_ic_p1_h,153556,21387,132169,13.93
16,chg_ic_p1_V,153556,21387,132169,13.93
18,chg_vi_R_ohm,153556,21418,132138,13.95
28,dis_vi_R_ohm,153556,42493,111063,27.67
13,chg_max_C_rate,153556,43866,109690,28.57
12,chg_mean_C_rate,153556,43866,109690,28.57
27,dis_ic_p1_h,153556,63237,90319,41.18
24,dis_ic_centroid_V,153556,63237,90319,41.18


In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display, HTML, Markdown

REPO_DIR = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main"
)
OUT_DIR = REPO_DIR / "paper_tables"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# TABLE 2A
# =========================
table2a = pd.DataFrame([
    {
        "Component": "Phase predictors (20)",
        "Specification": (
            "Phase indicators; duration; energy; mean/max current and C-rate; "
            "start/end/minimum voltage and drop; voltage-current slope; "
            "mean/max/rise/slope temperature; BOL-capacity SOC decrement; "
            "gap count"
        )
    },
    {
        "Component": "Mission predictors (20)",
        "Specification": (
            "Charge duration/energy/current/C-rate; CC/CV duration and energy; "
            "CV fraction/setpoint; discharge duration/energy/C-rate/slope; "
            "temperature; gap count; cumulative charge/discharge energy"
        )
    }
])

# =========================
# TABLE 2B
# =========================
table2b = pd.DataFrame([
    {
        "Feature family": "Phase identity and timing",
        "Level": "Phase",
        "Descriptors": "Take-off, cruise, and landing indicators; duration; gap count",
        "Status and physical role": "Available by phase end"
    },
    {
        "Feature family": "Electrical load",
        "Level": "Phase",
        "Descriptors": "Energy; mean/max current; mean/max C-rate",
        "Status and physical role": "Local power stress"
    },
    {
        "Feature family": "Voltage and resistance response",
        "Level": "Phase",
        "Descriptors": "Start/end/minimum voltage; voltage drop; voltage-current slope",
        "Status and physical role": "Voltage-sag/impedance proxy"
    },
    {
        "Feature family": "Thermal magnitude and response",
        "Level": "Phase",
        "Descriptors": "Mean/max/rise/slope temperature",
        "Status and physical role": "Electro-thermal exposure"
    },
    {
        "Feature family": "SOC decrement proxy",
        "Level": "Phase",
        "Descriptors": "Coulombic decrement normalized by beginning-of-life capacity",
        "Status and physical role": "Observed SOC label excluded"
    },
    {
        "Feature family": "Charging exposure",
        "Level": "Mission",
        "Descriptors": "Charge duration, energy, current/C-rate; CC/CV duration and energy; CV fraction/setpoint",
        "Status and physical role": "Charge acceptance and high-SOC exposure"
    },
    {
        "Feature family": "Flight discharge",
        "Level": "Mission",
        "Descriptors": "Discharge duration, energy, C-rate, and slope",
        "Status and physical role": "Mission energy demand"
    },
    {
        "Feature family": "Accumulated operation",
        "Level": "Mission",
        "Descriptors": "Cumulative charge/discharge energy; temperature; gap count",
        "Status and physical role": "Accumulated stress and data quality"
    }
])

# Tampilkan seperti susunan paper: 2A di atas, 2B di bawah
display(HTML("""
<h3 style="text-align:center;">Table 2.</h3>
<p style="text-align:center;">
(a) Predictor and benchmark model specification;
(b) leakage-safe mission-phase feature taxonomy and physical interpretation.
</p>
"""))

display(HTML("<h4 style='text-align:center;'>(a)</h4>"))
display(table2a)

display(HTML("<hr><h4 style='text-align:center;'>(b)</h4>"))
display(table2b)

display(Markdown("**BOL** = beginning of life."))

# Simpan hanya Table 2A dan Table 2B
table2a.to_csv(OUT_DIR / "table2a.csv", index=False, encoding="utf-8-sig")
table2b.to_csv(OUT_DIR / "table2b.csv", index=False, encoding="utf-8-sig")

print("\nTable 2A dan Table 2B berhasil ditampilkan.")
print("Output:", OUT_DIR)

,Component,Specification
0,Phase predictors (20),Phase indicators; duration; energy; mean/max c...
1,Mission predictors (20),Charge duration/energy/current/C-rate; CC/CV d...


,Feature family,Level,Descriptors,Status and physical role
0,Phase identity and timing,Phase,"Take-off, cruise, and landing indicators; dura...",Available by phase end
1,Electrical load,Phase,Energy; mean/max current; mean/max C-rate,Local power stress
2,Voltage and resistance response,Phase,Start/end/minimum voltage; voltage drop; volta...,Voltage-sag/impedance proxy
3,Thermal magnitude and response,Phase,Mean/max/rise/slope temperature,Electro-thermal exposure
4,SOC decrement proxy,Phase,Coulombic decrement normalized by beginning-of...,Observed SOC label excluded
5,Charging exposure,Mission,"Charge duration, energy, current/C-rate; CC/CV...",Charge acceptance and high-SOC exposure
6,Flight discharge,Mission,"Discharge duration, energy, C-rate, and slope",Mission energy demand
7,Accumulated operation,Mission,Cumulative charge/discharge energy; temperatur...,Accumulated stress and data quality


**BOL** = beginning of life.


Table 2A dan Table 2B berhasil ditampilkan.
Output: /content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main/paper_tables


# TABEL **3**

In [ ]:
from pathlib import Path
import sys
import subprocess
import importlib.util

REPO_DIR = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main"
)

assert REPO_DIR.exists(), f"Repository tidak ditemukan: {REPO_DIR}"

# Cari skrip model dan benchmark yang tersedia
candidate_files = [
    "Attn_lstm_MoE.py",
    "Attn_lstm.py",
    "sequence_benchmark.py",
    "baseline_gate.py",
    "battery_common.py",
    "pipeline.json",
    "health_mission_features_targets.csv",
    "mission_features.csv",
    "phase_features.csv",
]

print("=" * 70)
print("AUDIT AWAL IMPLEMENTASI TABLE 3")
print("=" * 70)

for filename in candidate_files:
    path = REPO_DIR / filename
    print(f"{filename:42} {'ADA' if path.exists() else 'TIDAK ADA'}")

print("\nVersi Python:", sys.version.split()[0])

for package in ["torch", "numpy", "pandas", "sklearn"]:
    spec = importlib.util.find_spec(package)
    print(f"Paket {package:12}:", "TERSEDIA" if spec else "BELUM TERSEDIA")

# Cari semua skrip Python terkait model sekuens dan pelatihan
print("\nSkrip Python terkait model/pelatihan:")
for path in sorted(REPO_DIR.glob("*.py")):
    name = path.name.lower()
    if any(key in name for key in [
        "lstm", "transformer", "sequence", "baseline", "train", "benchmark"
    ]):
        print(" -", path.name)

AUDIT AWAL IMPLEMENTASI TABLE 3
Attn_lstm_MoE.py                           ADA
Attn_lstm.py                               ADA
sequence_benchmark.py                      ADA
baseline_gate.py                           ADA
battery_common.py                          ADA
pipeline.json                              ADA
health_mission_features_targets.csv        ADA
mission_features.csv                       ADA
phase_features.csv                         TIDAK ADA

Versi Python: 3.12.13
Paket torch       : TERSEDIA
Paket numpy       : TERSEDIA
Paket pandas      : TERSEDIA
Paket sklearn     : TERSEDIA

Skrip Python terkait model/pelatihan:
 - Attn_lstm.py
 - Attn_lstm_MoE.py
 - aggregate_benchmark.py
 - baseline_gate.py
 - baseline_tress.py
 - sequence_benchmark.py
 - sequential_baseline.py


In [ ]:
import sys
from pathlib import Path
import importlib.util
import torch

REPO_DIR = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main"
)

assert (REPO_DIR / "sequence_benchmark.py").exists()
sys.path.insert(0, str(REPO_DIR))

# Muat skrip sebagai modul tanpa menjalankan main()
spec = importlib.util.spec_from_file_location(
    "sequence_benchmark",
    REPO_DIR / "sequence_benchmark.py"
)
sb = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = sb
spec.loader.exec_module(sb)

print("Parameter model untuk 20 fitur per misi:")
for name in [
    "attention_lstm_moe",
    "mission_phase_transformer",
]:
    model = sb.make_model(name, d_in=20, nonnegative=False)
    n_params = sum(
        p.numel() for p in model.parameters()
        if p.requires_grad
    )
    print(f"{name}: {n_params:,}")

Parameter model untuk 20 fitur per misi:
attention_lstm_moe: 186,156
mission_phase_transformer: 160,417


In [ ]:
from pathlib import Path

REPO_DIR = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main"
)

script_path = REPO_DIR / "sequence_benchmark.py"
backup_path = REPO_DIR / "sequence_benchmark.py.before_adamw.bak"

source = script_path.read_text(encoding="utf-8")

old = "optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)"
new = "optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)"

if old not in source:
    print("⚠️ Baris asli tidak ditemukan. Tidak ada perubahan yang dilakukan.")
elif "optimizer = torch.optim.AdamW(model.parameters()" in source:
    print("ℹ️ Optimizer tampaknya sudah menggunakan AdamW.")
else:
    backup_path.write_text(source, encoding="utf-8")
    script_path.write_text(source.replace(old, new, 1), encoding="utf-8")
    print("✅ Optimizer berhasil diubah menjadi AdamW.")
    print("✅ Backup tersimpan:", backup_path)

✅ Optimizer berhasil diubah menjadi AdamW.
✅ Backup tersimpan: /content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main/sequence_benchmark.py.before_adamw.bak


In [ ]:
from pathlib import Path

script_path = REPO_DIR / "sequence_benchmark.py"
lines = script_path.read_text(encoding="utf-8").splitlines()

for i, line in enumerate(lines, start=1):
    if "torch.optim.Adam" in line:
        print(f"Baris {i}: {line.strip()}")

Baris 296: optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)


In [ ]:
import json
import pandas as pd
from pathlib import Path

FOLD_DIR = Path(
    "/content/drive/MyDrive/eVTOL_benchmark_reproduction_v2/"
    "table3_single_fold_rul90_attention_lstm_moe/"
    "rul90/attention_lstm_moe/VAH01"
)

print("File hasil:")
for p in sorted(FOLD_DIR.iterdir()):
    print(f"- {p.name} ({p.stat().st_size:,} bytes)")

log_path = FOLD_DIR / "training_log.csv"
pred_path = FOLD_DIR / "predictions.csv"
record_path = FOLD_DIR / "fold_record.json"

if log_path.exists():
    log = pd.read_csv(log_path)
    print("\n=== LOG TRAINING ===")
    display(log)
else:
    print("\ntraining_log.csv tidak ditemukan.")

if pred_path.exists():
    pred = pd.read_csv(pred_path)
    print("\n=== RINGKASAN PREDIKSI ===")
    display(pred[["y_true", "y_pred", "error"]].describe())

    print("\nLima prediksi pertama:")
    display(pred.head())

    print("\nKorelasi target vs prediksi:",
          pred["y_true"].corr(pred["y_pred"]))
else:
    print("\npredictions.csv tidak ditemukan.")

if record_path.exists():
    print("\n=== FOLD RECORD ===")
    print(json.dumps(
        json.loads(record_path.read_text()),
        indent=2
    ))

File hasil:
- best.pt (754,937 bytes)
- fold_record.json (747 bytes)
- predictions.csv (11,446 bytes)
- training_log.csv (708 bytes)

=== LOG TRAINING ===


,epoch,train_huber,val_mae,val_rmse,learning_rate
0,1,0.069389,45.011124,61.983425,0.0020
1,2,0.006286,39.853786,53.300457,0.0020
2,3,0.004797,43.163445,56.053696,0.0020
3,4,0.002914,42.479359,54.670921,0.0020
4,5,0.003015,40.706158,52.484802,0.0020
5,6,0.002379,41.346142,53.370094,0.0010
6,7,0.001993,42.466724,54.220577,0.0010
7,8,0.002066,45.008682,56.792347,0.0010
8,9,0.002079,41.337082,52.010017,0.0010
9,10,0.002005,41.462475,52.253300,0.0005



=== RINGKASAN PREDIKSI ===


,y_true,y_pred,error
count,322.000000,322.000000,322.000000
mean,160.500000,71.680957,-88.819044
std,93.097619,19.170837,73.978413
min,0.000000,40.384100,-212.047820
25%,80.250000,53.982947,-152.813010
50%,160.500000,71.885387,-88.614613
75%,240.750000,88.276916,-26.689928
max,321.000000,117.580560,40.384100



Lima prediksi pertama:


,cell_id,mission_id,y_true,y_pred,error
0,VAH01,1,321.0,117.58056,-203.41943
1,VAH01,2,320.0,111.53083,-208.46918
2,VAH01,3,319.0,109.62294,-209.37706
3,VAH01,4,318.0,106.50862,-211.49138
4,VAH01,5,317.0,105.02785,-211.97215



Korelasi target vs prediksi: 0.9978606607921089

=== FOLD RECORD ===
{
  "best_epoch": 2,
  "best_val_mae": 39.85378646850586,
  "epochs_run": 10,
  "training_seconds": 36.51962925099997,
  "parameter_count": 186156,
  "target_mean": 157.33279418945312,
  "target_std": 97.02947998046875,
  "task": "rul90",
  "model": "attention_lstm_moe",
  "test_cell": "VAH01",
  "n_test": 322,
  "MAE": 95.06487274169922,
  "RMSE": 115.51896667480469,
  "elapsed_seconds": 36.63933806799923,
  "train_cells": [
    "VAH05",
    "VAH10",
    "VAH12",
    "VAH13",
    "VAH16",
    "VAH17",
    "VAH20",
    "VAH22",
    "VAH23",
    "VAH26",
    "VAH27",
    "VAH30"
  ],
  "validation_cells": [
    "VAH02",
    "VAH11",
    "VAH15",
    "VAH24",
    "VAH25",
    "VAH28"
  ],
  "monotonic_penalty": "batch-local, lambda=0.02"
}


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

pred_path = FOLD_DIR / "predictions.csv"

print("File prediksi tersedia:", pred_path.exists())

if pred_path.exists():
    saved = pd.read_csv(pred_path)
    print("\nKolom CSV:", saved.columns.tolist())
    print("Shape CSV:", saved.shape)
    print("\nLima baris pertama:")
    display(saved.head())

    print("\n=== KOLOM IDENTITAS DAN TARGET ===")
    for col in ["cell_id", "mission_id", "y_true", "y_pred",
                "true", "pred", "target", "prediction"]:
        if col in saved.columns:
            print(col, "contoh:", saved[col].head(5).tolist())

    print("\n=== RINGKASAN KOLOM NUMERIK ===")
    display(saved.select_dtypes(include="number").describe().T)

File prediksi tersedia: True

Kolom CSV: ['cell_id', 'mission_id', 'y_true', 'y_pred', 'error']
Shape CSV: (322, 5)

Lima baris pertama:


,cell_id,mission_id,y_true,y_pred,error
0,VAH01,1,321.0,117.58056,-203.41943
1,VAH01,2,320.0,111.53083,-208.46918
2,VAH01,3,319.0,109.62294,-209.37706
3,VAH01,4,318.0,106.50862,-211.49138
4,VAH01,5,317.0,105.02785,-211.97215



=== KOLOM IDENTITAS DAN TARGET ===
cell_id contoh: ['VAH01', 'VAH01', 'VAH01', 'VAH01', 'VAH01']
mission_id contoh: [1, 2, 3, 4, 5]
y_true contoh: [321.0, 320.0, 319.0, 318.0, 317.0]
y_pred contoh: [117.58056, 111.53083, 109.62294, 106.50862, 105.02785]

=== RINGKASAN KOLOM NUMERIK ===


,count,mean,std,min,25%,50%,75%,max
mission_id,322.0,161.500000,93.097619,1.00000,81.250000,161.500000,241.750000,322.00000
y_true,322.0,160.500000,93.097619,0.00000,80.250000,160.500000,240.750000,321.00000
y_pred,322.0,71.680957,19.170837,40.38410,53.982947,71.885387,88.276916,117.58056
error,322.0,-88.819044,73.978413,-212.04782,-152.813010,-88.614613,-26.689928,40.38410


In [ ]:
# Normalisasi tipe metadata agar bisa dibandingkan
record_train_cells = record["train_cells"]
record_val_cells = record["validation_cells"]

if isinstance(record_train_cells, str):
    record_train_cells = record_train_cells.split(";")

if isinstance(record_val_cells, str):
    record_val_cells = record_val_cells.split(";")

print("Train cells cocok:",
      sorted(train_cells) == sorted(record_train_cells))
print("Validation cells cocok:",
      sorted(val_cells) == sorted(record_val_cells))
print("Jumlah sampel test:", len(test_idx), len(saved))

print("MAE hasil scaler rekonstruksi:",
      float(np.mean(np.abs(pred - true))))
print("MAE CSV:",
      float(np.mean(np.abs(saved["y_pred"].to_numpy()
                           - saved["y_true"].to_numpy()))))
print("Korelasi prediksi baru vs CSV:",
      float(np.corrcoef(pred, saved["y_pred"].to_numpy())[0, 1]))
print("MAE selisih prediksi:",
      float(np.mean(np.abs(pred - saved["y_pred"].to_numpy()))))

Train cells cocok: True
Validation cells cocok: True
Jumlah sampel test: 322 322
MAE hasil scaler rekonstruksi: 15658.8837890625
MAE CSV: 95.06487519875778
Korelasi prediksi baru vs CSV: 0.9999999999999964
MAE selisih prediksi: 1.1555827835720142e-06


In [ ]:
print("=== AUDIT TARGET VAH01 ===")
print("Target dari predict:")
print("min =", float(np.min(true)))
print("max =", float(np.max(true)))
print("mean =", float(np.mean(true)))

print("\nTarget dari CSV:")
print("min =", float(saved["y_true"].min()))
print("max =", float(saved["y_true"].max()))
print("mean =", float(saved["y_true"].mean()))

print("\nSelisih target maksimum:",
      float(np.max(np.abs(
          np.asarray(true).reshape(-1)
          - saved["y_true"].to_numpy()
      ))))

print("\nPrediksi dari predict:")
print("min =", float(np.min(pred)))
print("max =", float(np.max(pred)))
print("mean =", float(np.mean(pred)))

=== AUDIT TARGET VAH01 ===
Target dari predict:
min = 157.33279418945312
max = 31303.794921875
mean = 15730.5634765625

Target dari CSV:
min = 0.0
max = 321.0
mean = 160.5

Selisih target maksimum: 30982.794921875

Prediksi dari predict:
min = 40.38410186767578
max = 117.58055877685547
mean = 71.68096160888672


In [ ]:
from pathlib import Path
import json

OUT_ROOT = Path(
    "/content/drive/MyDrive/eVTOL_benchmark_reproduction_v2"
)
REPO_DIR = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main"
)

print("=== 1. FOLDER EKSPERIMEN TABLE 3 ===")
for p in sorted(OUT_ROOT.glob("*table3*")):
    print(p.name, "| folder:", p.is_dir())

print("\n=== 2. SEMUA FOLD RUL90 ATTENTION-LSTM-MOE ===")
matches = list(
    OUT_ROOT.glob("**/rul90/attention_lstm_moe/*/predictions.csv")
)
print("Jumlah predictions.csv:", len(matches))
for p in matches:
    print("-", p.parent.name, "|", p.parent)

print("\n=== 3. FILE SCRIPT BENCHMARK ===")
for name in ["sequence_benchmark_table3.py", "sequence_benchmark.py"]:
    p = REPO_DIR / name
    print(f"\n{name}: ditemukan =", p.exists())
    if p.exists():
        print("Ukuran:", p.stat().st_size, "bytes")

print("\n=== 4. KONFIGURASI FOLD / CELL ===")
for p in sorted(OUT_ROOT.glob("**/fold_record.json")):
    try:
        d = json.loads(p.read_text(encoding="utf-8"))
        print(
            p.parent.name,
            "| task:", d.get("task"),
            "| model:", d.get("model"),
            "| test_cell:", d.get("test_cell"),
            "| best_epoch:", d.get("best_epoch"),
        )
    except Exception as e:
        print("Gagal membaca", p, repr(e))

=== 1. FOLDER EKSPERIMEN TABLE 3 ===
table3_rul90_attention_lstm_moe_fold_audit.csv | folder: False
table3_single_fold_rul90_attention_lstm_moe | folder: True

=== 2. SEMUA FOLD RUL90 ATTENTION-LSTM-MOE ===
Jumlah predictions.csv: 1
- VAH01 | /content/drive/MyDrive/eVTOL_benchmark_reproduction_v2/table3_single_fold_rul90_attention_lstm_moe/rul90/attention_lstm_moe/VAH01

=== 3. FILE SCRIPT BENCHMARK ===

sequence_benchmark_table3.py: ditemukan = True
Ukuran: 24565 bytes

sequence_benchmark.py: ditemukan = True
Ukuran: 21993 bytes

=== 4. KONFIGURASI FOLD / CELL ===
VAH01 | task: soh | model: mission_phase_transformer | test_cell: VAH01 | best_epoch: 9
VAH01 | task: rul90 | model: attention_lstm_moe | test_cell: VAH01 | best_epoch: 2


In [ ]:
from pathlib import Path
import pandas as pd

REPO_DIR = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main"
)

INPUT_CSV = REPO_DIR / "health_mission_features_targets.csv"
SCRIPT = REPO_DIR / "sequence_benchmark_table3.py"
OUTPUT_DIR = Path(
    "/content/drive/MyDrive/eVTOL_benchmark_reproduction_v2/table3_full_run"
)

print("Script tersedia:", SCRIPT.exists(), SCRIPT)
print("Input tersedia:", INPUT_CSV.exists(), INPUT_CSV)

if not SCRIPT.exists():
    raise FileNotFoundError(SCRIPT)

if not INPUT_CSV.exists():
    raise FileNotFoundError(INPUT_CSV)

df = pd.read_csv(INPUT_CSV)

print("\nUkuran input:", df.shape)
print("Kolom cell_id tersedia:", "cell_id" in df.columns)
print("Kolom mission_id tersedia:", "mission_id" in df.columns)

if "cell_id" in df.columns:
    print("Jumlah cell:", df["cell_id"].nunique())
    print("Jumlah misi:", len(df))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("\nDirektori output:", OUTPUT_DIR)
print("Direktori output siap:", OUTPUT_DIR.is_dir())

Script tersedia: True /content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main/sequence_benchmark_table3.py
Input tersedia: True /content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main/health_mission_features_targets.csv

Ukuran input: (18862, 46)
Kolom cell_id tersedia: True
Kolom mission_id tersedia: True
Jumlah cell: 19
Jumlah misi: 18862

Direktori output: /content/drive/MyDrive/eVTOL_benchmark_reproduction_v2/table3_full_run
Direktori output siap: True


In [ ]:
from pathlib import Path
import pandas as pd

OUT = Path(
    "/content/drive/MyDrive/eVTOL_benchmark_reproduction_v2/table3_full_run"
)

tasks = ["soh", "rul90", "rul85", "rul80"]
models = ["attention_lstm_moe", "mission_phase_transformer"]
expected_cells = 19

rows = []

for task in tasks:
    for model in models:
        root = OUT / task / model
        fold_dirs = sorted(
            p for p in root.iterdir() if p.is_dir()
        ) if root.exists() else []

        complete = 0
        predictions = 0
        logs = 0

        for fold in fold_dirs:
            required = [
                fold / "best.pt",
                fold / "fold_record.json",
                fold / "predictions.csv",
                fold / "training_log.csv",
                fold / "scaler.json",
            ]

            if all(p.exists() and p.stat().st_size > 0 for p in required):
                complete += 1

            if (fold / "predictions.csv").exists():
                predictions += 1
            if (fold / "training_log.csv").exists():
                logs += 1

        rows.append({
            "task": task,
            "model": model,
            "fold_dirs": len(fold_dirs),
            "complete_folds": complete,
            "prediction_files": predictions,
            "training_logs": logs,
            "expected_folds": expected_cells,
        })

summary = pd.DataFrame(rows)
display(summary)

print("\nFile ringkasan:")
for name in [
    "sequence_metrics.json",
    "sequence_fold_metrics.csv",
    "sequence_predictions.csv",
    "sequence_training_logs.csv",
]:
    p = OUT / name
    print(name, "->", "ADA" if p.exists() else "BELUM ADA")

,task,model,fold_dirs,complete_folds,prediction_files,training_logs,expected_folds
0,soh,attention_lstm_moe,19,19,19,19,19
1,soh,mission_phase_transformer,6,5,5,5,19
2,rul90,attention_lstm_moe,0,0,0,0,19
3,rul90,mission_phase_transformer,0,0,0,0,19
4,rul85,attention_lstm_moe,0,0,0,0,19
5,rul85,mission_phase_transformer,0,0,0,0,19
6,rul80,attention_lstm_moe,0,0,0,0,19
7,rul80,mission_phase_transformer,0,0,0,0,19



File ringkasan:
sequence_metrics.json -> BELUM ADA
sequence_fold_metrics.csv -> BELUM ADA
sequence_predictions.csv -> BELUM ADA
sequence_training_logs.csv -> BELUM ADA


In [ ]:
import sys
import subprocess
from pathlib import Path

REPO_DIR = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main"
)
SCRIPT = REPO_DIR / "sequence_benchmark_table3.py"
INPUT_CSV = REPO_DIR / "health_mission_features_targets.csv"
OUTPUT_DIR = Path(
    "/content/drive/MyDrive/eVTOL_benchmark_reproduction_v2/table3_full_run"
)

cmd = [
    sys.executable,
    "-u",
    str(SCRIPT),
    "--input", str(INPUT_CSV),
    "--output-dir", str(OUTPUT_DIR),
    "--manifest", "manuscript20",
    "--tasks", "soh", "rul90", "rul85", "rul80",
    "--models", "attention_lstm_moe", "mission_phase_transformer",
]

print("Benchmark dimulai dengan resume dari hasil tersimpan...\n", flush=True)

process = subprocess.Popen(
    cmd,
    cwd=str(REPO_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

try:
    for line in process.stdout:
        print(line, end="", flush=True)
finally:
    process.stdout.close()

return_code = process.wait()
print(f"\nBenchmark selesai dengan exit code: {return_code}")

Benchmark dimulai dengan resume dari hasil tersimpan...

Prepared soh: N=18513 cells=19
soh attention_lstm_moe 01/19 VAH01: resumed cached fold
soh attention_lstm_moe 02/19 VAH02: resumed cached fold
soh attention_lstm_moe 03/19 VAH05: resumed cached fold
soh attention_lstm_moe 04/19 VAH10: resumed cached fold
soh attention_lstm_moe 05/19 VAH11: resumed cached fold
soh attention_lstm_moe 06/19 VAH12: resumed cached fold
soh attention_lstm_moe 07/19 VAH13: resumed cached fold
soh attention_lstm_moe 08/19 VAH15: resumed cached fold
soh attention_lstm_moe 09/19 VAH16: resumed cached fold
soh attention_lstm_moe 10/19 VAH17: resumed cached fold
soh attention_lstm_moe 11/19 VAH20: resumed cached fold
soh attention_lstm_moe 12/19 VAH22: resumed cached fold
soh attention_lstm_moe 13/19 VAH23: resumed cached fold
soh attention_lstm_moe 14/19 VAH24: resumed cached fold
soh attention_lstm_moe 15/19 VAH25: resumed cached fold
soh attention_lstm_moe 16/19 VAH26: resumed cached fold
soh attention_ls

In [ ]:
from pathlib import Path
import json
import pandas as pd

OUT = Path(
    "/content/drive/MyDrive/eVTOL_benchmark_reproduction_v2/table3_full_run"
)

print("=" * 75)
print("AUDIT OUTPUT BENCHMARK TABLE 3")
print("=" * 75)

# A. Periksa empat file hasil utama
files = [
    "sequence_metrics.json",
    "sequence_fold_metrics.csv",
    "sequence_predictions.csv",
    "sequence_training_logs.csv",
]

for name in files:
    p = OUT / name
    print(
        f"{name}:",
        f"ADA | {p.stat().st_size:,} bytes"
        if p.exists() else "BELUM ADA"
    )

# B. Baca metrik agregat
metrics_path = OUT / "sequence_metrics.json"

if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))

    print("\n=== METRIK AGREGAT ===")
    print(json.dumps(metrics, indent=2, ensure_ascii=False))
else:
    print("\nFile sequence_metrics.json belum ditemukan.")

# C. Periksa struktur CSV
for name in [
    "sequence_fold_metrics.csv",
    "sequence_predictions.csv",
    "sequence_training_logs.csv",
]:
    p = OUT / name

    if p.exists():
        df = pd.read_csv(p)
        print(f"\n=== {name} ===")
        print("Ukuran:", df.shape)
        print("Kolom:", df.columns.tolist())
        print(df.head(3).to_string(index=False))

# D. Hitung jumlah fold yang mempunyai hasil tersimpan
print("\n=== JUMLAH FOLD LENGKAP ===")

for task in ["soh", "rul90", "rul85", "rul80"]:
    for model in ["attention_lstm_moe", "mission_phase_transformer"]:
        root = OUT / task / model
        n_complete = 0

        if root.exists():
            for fold in root.iterdir():
                if not fold.is_dir():
                    continue

                required = [
                    fold / "predictions.csv",
                    fold / "fold_record.json",
                    fold / "training_log.csv",
                ]

                if all(p.exists() and p.stat().st_size > 0 for p in required):
                    n_complete += 1

        print(f"{task:5s} | {model:27s} | {n_complete} fold")

AUDIT OUTPUT BENCHMARK TABLE 3
sequence_metrics.json: ADA | 10,397 bytes
sequence_fold_metrics.csv: ADA | 35,812 bytes
sequence_predictions.csv: ADA | 6,507,982 bytes
sequence_training_logs.csv: ADA | 225,266 bytes

=== METRIK AGREGAT ===
[
  {
    "task": "soh",
    "model": "attention_lstm_moe",
    "cells": 19,
    "n": 18513,
    "mae": 1.4344824449376028,
    "rmse": 2.1470795695500673,
    "macro_mae": 1.4881368535798314,
    "p90_cell_mae": 2.932262664527869,
    "worst_cell_mae": 5.427888414320388,
    "features": [
      "chg_total_dur_s",
      "chg_energy_Wh",
      "chg_mean_I_A",
      "chg_mean_C_rate",
      "chg_cc_dur_s",
      "chg_cc_energy_Wh",
      "chg_cv_dur_s",
      "chg_cv_energy_Wh",
      "chg_cv_fraction",
      "cv_setpoint_V",
      "dis_total_dur_s",
      "dis_flight_Wh_abs",
      "dis_mean_C_rate",
      "dis_max_C_rate",
      "dis_vi_R_ohm_med",
      "temp_mean_C",
      "temp_max_C_max",
      "gap_count",
      "chg_cum_Wh",
      "dis_cum_Wh"
 

In [ ]:
from pathlib import Path

REPO = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main"
)
SCRIPT = REPO / "sequence_benchmark_table3.py"
lines = SCRIPT.read_text(
    encoding="utf-8",
    errors="replace"
).splitlines()

def show_lines(start, end):
    print(f"\n{'=' * 85}")
    print(f"BARIS {start}-{end}")
    print("=" * 85)

    for i in range(start, min(end, len(lines)) + 1):
        print(f"{i:4d}: {lines[i - 1]}")

# Pembagian inner fold dan preprocessing fitur
show_lines(93, 180)

# Arsitektur kedua model
show_lines(180, 280)

# Imputasi, scaling, optimizer, scheduler, dan training
show_lines(300, 345)

# Early stopping dan pemilihan checkpoint
show_lines(405, 445)


BARIS 93-180
  93: def balanced_inner_folds(cells: np.ndarray, train_indices: np.ndarray) -> dict[str, int]:
  94:     counts = pd.Series(cells[train_indices]).value_counts().to_dict()
  95:     totals = [0, 0, 0]
  96:     assignment: dict[str, int] = {}
  97:     for cell, count in sorted(counts.items(), key=lambda item: (-item[1], item[0])):
  98:         fold = min(range(3), key=lambda value: (totals[value], value))
  99:         assignment[str(cell)] = fold
 100:         totals[fold] += int(count)
 101:     return assignment
 102: 
 103: 
 104: class FoldScaler:
 105:     def fit(self, x: np.ndarray, mask: np.ndarray) -> "FoldScaler":
 106:         valid = x[mask]
 107:         self.median = np.nanmedian(valid, axis=0)
 108:         self.median = np.where(np.isfinite(self.median), self.median, 0.0)
 109:         filled = np.where(np.isfinite(valid), valid, self.median)
 110:         self.mean = filled.mean(axis=0)
 111:         self.std = filled.std(axis=0)
 112:         self.std

In [ ]:
from pathlib import Path
import pandas as pd

REPO_DIR = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main"
)

SCRIPT = REPO_DIR / "sequence_benchmark_table3.py"
METRICS = Path(
    "/content/drive/MyDrive/eVTOL_benchmark_reproduction_v2/"
    "table3_full_run/sequence_fold_metrics.csv"
)

lines = SCRIPT.read_text(
    encoding="utf-8",
    errors="replace"
).splitlines()

def show_lines(start, end):
    print(f"\n{'=' * 85}")
    print(f"BARIS {start}-{end}")
    print("=" * 85)

    for i in range(start, min(end, len(lines)) + 1):
        print(f"{i:4}: {lines[i - 1]}")

# Periksa pembentukan fitur, sequence, scaler, dan pemisahan data.
show_lines(445, 530)

# Periksa proses LOCO, pemilihan cell, dan pemanggilan scaler.
show_lines(530, 600)

# Verifikasi jumlah parameter aktual pada hasil eksperimen.
if METRICS.exists():
    df = pd.read_csv(METRICS)

    print("\n" + "=" * 85)
    print("JUMLAH PARAMETER AKTUAL")
    print("=" * 85)

    print(
        df.groupby("model")["parameter_count"]
        .agg(["min", "max", "nunique"])
        .to_string()
    )

    print("\nContoh fold:")
    cols = [
        c for c in [
            "task", "model", "fold", "test_cell",
            "parameter_count", "train_n", "validation_n", "test_n"
        ]
        if c in df.columns
    ]
    print(df[cols].head(10).to_string(index=False))
else:
    print("\nFile sequence_fold_metrics.csv tidak ditemukan:", METRICS)


BARIS 445-530
 445:     samples: dict[str, np.ndarray],
 446:     features: list[str],
 447:     task: str,
 448:     model_name: str,
 449:     output_dir: Path,
 450: ) -> tuple[dict, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
 451:     x_raw, mask, y = samples["x"], samples["mask"], samples["y"]
 452:     cells, mission_ids = samples["cell"], samples["mission_id"]
 453:     _, _, upper_bound = task_target(task)
 454:     all_index = np.arange(len(y))
 455:     prediction_rows, fold_rows, log_rows = [], [], []
 456:     for fold_number, test_cell in enumerate(sorted(np.unique(cells)), start=1):
 457:         fold_dir = output_dir / task / model_name / test_cell
 458:         fold_predictions_path = fold_dir / "predictions.csv"
 459:         fold_record_path = fold_dir / "fold_record.json"
 460:         fold_log_path = fold_dir / "training_log.csv"
 461:         if (
 462:             fold_predictions_path.exists()
 463:             and fold_record_path.exists()
 464:             and

In [ ]:
from pathlib import Path
import json
import pandas as pd

REPO_DIR = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main"
)

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/eVTOL_benchmark_reproduction_v2/"
    "table3_full_run"
)

SCRIPT = REPO_DIR / "sequence_benchmark_table3.py"
lines = SCRIPT.read_text(encoding="utf-8", errors="replace").splitlines()

def show_lines(start, end):
    print(f"\n{'=' * 85}")
    print(f"BARIS {start}-{end}")
    print("=" * 85)
    for i in range(start, min(end, len(lines)) + 1):
        print(f"{i:4}: {lines[i - 1]}")

# A. Lihat cara input dibaca dan fitur dipilih.
show_lines(600, 690)

# B. Periksa apakah konfigurasi manifest dan target terdokumentasi.
for filename in ["baseline_gate.py", "pipeline.json"]:
    path = REPO_DIR / filename
    print(f"\n{'=' * 85}\nFILE: {filename}")
    if not path.exists():
        print("Tidak ditemukan:", path)
        continue

    content = path.read_text(encoding="utf-8", errors="replace").splitlines()
    keywords = (
        ["MANIFESTS", "manuscript20", "health_mission_features_targets.csv"]
        if filename == "baseline_gate.py"
        else ["manuscript20", "features", "history", "target", "rul_thresholds_pct"]
    )

    hits = [
        i for i, line in enumerate(content)
        if any(k.lower() in line.lower() for k in keywords)
    ]

    shown = set()
    for i in hits[:12]:
        for j in range(max(0, i - 3), min(len(content), i + 5)):
            if j not in shown:
                print(f"{j + 1:4}: {content[j]}")
                shown.add(j)
        print("-" * 45)

# C. Ringkasan cache per task/model.
print("\n" + "=" * 85)
print("AUDIT CACHE")
print("=" * 85)

for task in ["soh", "rul90", "rul85", "rul80"]:
    for model in ["attention_lstm_moe", "mission_phase_transformer"]:
        base = OUTPUT_DIR / task / model
        if not base.exists():
            print(f"{task:5} | {model:25} | folder tidak ditemukan")
            continue

        fold_dirs = sorted(p for p in base.iterdir() if p.is_dir())
        complete = 0
        incomplete = []

        for fold_dir in fold_dirs:
            required = [
                fold_dir / "predictions.csv",
                fold_dir / "fold_record.json",
                fold_dir / "training_log.csv",
            ]
            if all(p.exists() for p in required):
                complete += 1
            else:
                incomplete.append(fold_dir.name)

        print(
            f"{task:5} | {model:25} | "
            f"fold_dirs={len(fold_dirs):2} | complete_cache={complete:2} | "
            f"incomplete={incomplete[:5]}"
        )


BARIS 600-690
 600: def main() -> None:
 601:     parser = argparse.ArgumentParser(description=__doc__)
 602:     parser.add_argument("--input", type=Path, required=True)
 603:     parser.add_argument("--output-dir", type=Path, required=True)
 604:     parser.add_argument("--manifest", choices=sorted(MANIFESTS), default="manuscript20")
 605:     parser.add_argument(
 606:         "--tasks", nargs="+", choices=["soh", "rul90", "rul85", "rul80"], default=["soh"]
 607:     )
 608:     parser.add_argument(
 609:         "--models",
 610:         nargs="+",
 611:         choices=["attention_lstm_moe", "mission_phase_transformer"],
 612:         default=["attention_lstm_moe", "mission_phase_transformer"],
 613:     )
 614:     args = parser.parse_args()
 615:     set_seed()
 616:     missions = pd.read_csv(args.input)
 617:     features = MANIFESTS[args.manifest]
 618:     args.output_dir.mkdir(parents=True, exist_ok=True)
 619:     all_metrics, all_predictions, all_folds, all_logs = [], []

In [ ]:
from pathlib import Path
import json
import pandas as pd
import importlib.util

REPO_DIR = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main"
)

INPUT = REPO_DIR / "health_mission_features_targets.csv"
OUTPUT_DIR = Path(
    "/content/drive/MyDrive/eVTOL_benchmark_reproduction_v2/"
    "table3_full_run"
)

# Muat manifest langsung dari baseline_gate.py
spec = importlib.util.spec_from_file_location(
    "baseline_gate_audit", REPO_DIR / "baseline_gate.py"
)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

features = module.MANIFESTS["manuscript20"]
df = pd.read_csv(INPUT)

print("=" * 80)
print("AUDIT MANIFEST")
print("=" * 80)
print("Jumlah fitur:", len(features))
for i, feature in enumerate(features, start=1):
    print(f"{i:02d}. {feature}")

missing = [f for f in features if f not in df.columns]
print("\nFitur tidak ditemukan:", missing)
print("Ukuran dataset:", df.shape)

print("\n" + "=" * 80)
print("AUDIT TARGET DAN JUMLAH SAMPEL")
print("=" * 80)

for col in [
    "SOH_m_plus_5_pct",
    "RUL90_missions",
    "RUL85_missions",
    "RUL80_missions",
]:
    if col not in df.columns:
        print(f"{col}: TIDAK DITEMUKAN")
        continue

    valid = df[col].notna()
    cells = df.loc[valid, "cell_id"].nunique()
    print(
        f"{col}: valid_rows={int(valid.sum())}, "
        f"cells={cells}, "
        f"min={df.loc[valid, col].min():.4f}, "
        f"max={df.loc[valid, col].max():.4f}"
    )

print("\n" + "=" * 80)
print("AUDIT METRIK AGREGAT TERSIMPAN")
print("=" * 80)

metrics_path = OUTPUT_DIR / "sequence_metrics.json"
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    for item in metrics:
        print(
            f"{item['task']:5} | {item['model']:25} | "
            f"N={item['n']:5} | cells={item['cells']:2} | "
            f"MAE={item['mae']:.4f} | RMSE={item['rmse']:.4f}"
        )
else:
    print("sequence_metrics.json tidak ditemukan.")

AUDIT MANIFEST
Jumlah fitur: 20
01. chg_total_dur_s
02. chg_energy_Wh
03. chg_mean_I_A
04. chg_mean_C_rate
05. chg_cc_dur_s
06. chg_cc_energy_Wh
07. chg_cv_dur_s
08. chg_cv_energy_Wh
09. chg_cv_fraction
10. cv_setpoint_V
11. dis_total_dur_s
12. dis_flight_Wh_abs
13. dis_mean_C_rate
14. dis_max_C_rate
15. dis_vi_R_ohm_med
16. temp_mean_C
17. temp_max_C_max
18. gap_count
19. chg_cum_Wh
20. dis_cum_Wh

Fitur tidak ditemukan: []
Ukuran dataset: (18862, 46)

AUDIT TARGET DAN JUMLAH SAMPEL
SOH_m_plus_5_pct: valid_rows=18513, cells=19, min=71.9969, max=99.7337
RUL90_missions: valid_rows=5814, cells=19, min=0.0000, max=399.0000
RUL85_missions: valid_rows=9595, cells=16, min=0.0000, max=810.0000
RUL80_missions: valid_rows=8257, cells=8, min=0.0000, max=1276.0000

AUDIT METRIK AGREGAT TERSIMPAN
soh   | attention_lstm_moe        | N=18513 | cells=19 | MAE=1.4345 | RMSE=2.1471
soh   | mission_phase_transformer | N=18513 | cells=19 | MAE=1.3744 | RMSE=2.2537
rul90 | attention_lstm_moe        | N= 5

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

REPO_DIR = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main"
)

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/eVTOL_benchmark_reproduction_v2/"
    "table3_full_run"
)

DATA_PATH = REPO_DIR / "health_mission_features_targets.csv"
df = pd.read_csv(DATA_PATH)

TARGET = "SOH_m_plus_5_pct"

for model in ["attention_lstm_moe", "mission_phase_transformer"]:
    fold_dir = OUTPUT_DIR / "soh" / model / "VAH01"
    record = json.loads(
        (fold_dir / "fold_record.json").read_text(encoding="utf-8")
    )
    pred = pd.read_csv(fold_dir / "predictions.csv")

    train_cells = record["train_cells"].split(";")
    val_cells = record["validation_cells"].split(";")

    train_df = df[df["cell_id"].isin(train_cells)]
    val_df = df[df["cell_id"].isin(val_cells)]
    test_df = df[df["cell_id"] == "VAH01"]

    print("\n" + "=" * 80)
    print("MODEL:", model)
    print("=" * 80)

    print("\nDistribusi target SOH:")
    for name, part in [
        ("Train cells", train_df),
        ("Validation cells", val_df),
        ("Test VAH01", test_df),
    ]:
        s = part[TARGET].dropna()
        print(
            f"{name:18} n={len(s):5} "
            f"mean={s.mean():8.3f} median={s.median():8.3f} "
            f"min={s.min():8.3f} max={s.max():8.3f}"
        )

    print("\nDistribusi error menurut kelompok misi VAH01:")
    pred = pred.sort_values("mission_id").copy()
    pred["abs_error"] = pred["error"].abs()

    pred["mission_group"] = pd.cut(
        pred["mission_id"],
        bins=[-np.inf, 20, 100, 250, 500, np.inf],
        labels=["1-20", "21-100", "101-250", "251-500", "501+"],
    )

    summary = pred.groupby("mission_group", observed=True).agg(
        n=("error", "size"),
        mean_true=("y_true", "mean"),
        mean_pred=("y_pred", "mean"),
        mean_error=("error", "mean"),
        mae=("abs_error", "mean"),
    )
    print(summary.round(3).to_string())

    print("\nContoh fold metadata:")
    print(
        "Target mean training =", record["target_mean"],
        "| Target std training =", record["target_std"],
    )


MODEL: attention_lstm_moe

Distribusi target SOH:
Train cells        n=11744 mean=  86.693 median=  86.659 min=  71.997 max=  99.734
Validation cells   n= 5945 mean=  85.902 median=  86.149 min=  73.041 max=  99.730
Test VAH01         n=  824 mean=  88.838 median=  87.971 min=  81.477 max=  99.652

Distribusi error menurut kelompok misi VAH01:
                 n  mean_true  mean_pred  mean_error     mae
mission_group                                               
1-20            20     99.102     85.241     -13.861  13.861
21-100          80     96.523     84.515     -12.008  12.008
101-250        150     92.979     83.214      -9.765   9.765
251-500        250     88.810     84.585      -4.225   4.225
501+           324     84.412     82.209      -2.203   2.203

Contoh fold metadata:
Target mean training = 86.6927261352539 | Target std training = 6.234044075012207

MODEL: mission_phase_transformer

Distribusi target SOH:
Train cells        n=11744 mean=  86.693 median=  86.659 min=  

In [ ]:
from pathlib import Path
import pandas as pd

REPO_DIR = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main"
)

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/eVTOL_benchmark_reproduction_v2/"
    "table3_full_run"
)

df = pd.read_csv(REPO_DIR / "health_mission_features_targets.csv")

for model in ["attention_lstm_moe", "mission_phase_transformer"]:
    pred_path = OUTPUT_DIR / "soh" / model / "VAH01" / "predictions.csv"
    pred = pd.read_csv(pred_path)

    print("\n" + "=" * 75)
    print("MODEL:", model)
    print("=" * 75)

    print("\nKolom predictions.csv:")
    print(pred.columns.tolist())

    print("\nLima baris pertama predictions.csv:")
    print(pred.head().to_string(index=False))

print("\nKolom dataset sumber:")
print(df.columns.tolist())

print("\nLima baris pertama dataset sumber:")
print(df[["cell_id", "mission_id", "SOH_m_plus_5_pct"]].head().to_string(index=False))


MODEL: attention_lstm_moe

Kolom predictions.csv:
['task', 'model', 'test_cell', 'mission_id', 'y_true', 'y_pred', 'error']

Lima baris pertama predictions.csv:
task              model test_cell  mission_id   y_true   y_pred      error
 soh attention_lstm_moe     VAH01           1 99.65223 85.79429 -13.857941
 soh attention_lstm_moe     VAH01           2 99.59427 85.40152 -14.192749
 soh attention_lstm_moe     VAH01           3 99.53631 85.47368 -14.062630
 soh attention_lstm_moe     VAH01           4 99.47835 85.36861 -14.109741
 soh attention_lstm_moe     VAH01           5 99.42039 85.36841 -14.051979

MODEL: mission_phase_transformer

Kolom predictions.csv:
['task', 'model', 'test_cell', 'mission_id', 'y_true', 'y_pred', 'error']

Lima baris pertama predictions.csv:
task                     model test_cell  mission_id   y_true   y_pred      error
 soh mission_phase_transformer     VAH01           1 99.65223 84.59803 -15.054199
 soh mission_phase_transformer     VAH01           2 99

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

path = Path(
    "/content/drive/MyDrive/eVTOL_raw_telemetry/VAH01.csv"
)

df = pd.read_csv(
    path,
    usecols=["Ecell_V", "I_mA", "Ns"],
    nrows=100_000
)

df["V_V"] = pd.to_numeric(df["Ecell_V"], errors="coerce")
df["I_A"] = pd.to_numeric(df["I_mA"], errors="coerce") / 1000
df["Ns"] = pd.to_numeric(df["Ns"], errors="coerce")
df = df.dropna(subset=["V_V", "I_A", "Ns"])

def calculate_slope(group):
    # Gunakan sampel arus negatif; diagnostik, bukan pengganti
    # filter validitas fase di pipeline asli.
    g = group[group["I_A"] < -0.05]

    if len(g) < 12:
        return pd.Series({
            "n": len(g),
            "slope_V_vs_I_ohm": np.nan,
            "negative_slope_ohm": np.nan
        })

    i = g["I_A"].to_numpy(float)
    v = g["V_V"].to_numpy(float)

    denominator = np.sum((i - i.mean()) ** 2)
    if denominator <= 1e-12:
        slope = np.nan
    else:
        slope = np.sum((i - i.mean()) * (v - v.mean())) / denominator

    return pd.Series({
        "n": len(g),
        "slope_V_vs_I_ohm": slope,
        "negative_slope_ohm": -slope
    })

result = (
    df[df["Ns"].isin([4, 5, 6])]
    .groupby("Ns")
    .apply(calculate_slope, include_groups=False)
)

print(result.to_string())

          n  slope_V_vs_I_ohm  negative_slope_ohm
Ns                                               
4    3527.0          0.028657           -0.028657
5   43871.0          0.006097           -0.006097
6    4801.0          0.051306           -0.051306


In [ ]:
import json
from pathlib import Path

p = Path("/content/drive/MyDrive/eVTOL_benchmark_reproduction_v2/table3_full_run/sequence_metrics.json")
rows = json.loads(p.read_text())

for r in rows:
    print(
        f"{r['task']:6} | {r['model']:24} | "
        f"N={r['n']:5} | cells={r['cells']:2} | "
        f"MAE={r['mae']:.4f} | RMSE={r['rmse']:.4f}"
    )

soh    | attention_lstm_moe       | N=18513 | cells=19 | MAE=1.4345 | RMSE=2.1471
soh    | mission_phase_transformer | N=18513 | cells=19 | MAE=1.3744 | RMSE=2.2537
rul90  | attention_lstm_moe       | N= 5814 | cells=19 | MAE=37.1696 | RMSE=55.1689
rul90  | mission_phase_transformer | N= 5814 | cells=19 | MAE=36.8694 | RMSE=67.2868
rul85  | attention_lstm_moe       | N= 9595 | cells=16 | MAE=67.4785 | RMSE=89.4009
rul85  | mission_phase_transformer | N= 9595 | cells=16 | MAE=73.4760 | RMSE=95.4749
rul80  | attention_lstm_moe       | N= 8257 | cells= 8 | MAE=180.6363 | RMSE=222.3230
rul80  | mission_phase_transformer | N= 8257 | cells= 8 | MAE=175.4670 | RMSE=233.3264


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown

# Lokasi hasil eksperimen yang sudah selesai
RESULT_DIR = Path(
    "/content/drive/MyDrive/eVTOL_benchmark_reproduction_v2/"
    "table3_full_run"
)
METRICS_FILE = RESULT_DIR / "sequence_metrics.json"

if not METRICS_FILE.exists():
    raise FileNotFoundError(f"Hasil Table 3 tidak ditemukan: {METRICS_FILE}")

with METRICS_FILE.open("r", encoding="utf-8") as f:
    results = json.load(f)

# Validasi: harus ada 8 hasil untuk 4 task × 2 model
expected = {
    ("soh", "attention_lstm_moe"),
    ("soh", "mission_phase_transformer"),
    ("rul90", "attention_lstm_moe"),
    ("rul90", "mission_phase_transformer"),
    ("rul85", "attention_lstm_moe"),
    ("rul85", "mission_phase_transformer"),
    ("rul80", "attention_lstm_moe"),
    ("rul80", "mission_phase_transformer"),
}
actual = {(r["task"], r["model"]) for r in results}

if actual != expected:
    raise ValueError(
        f"Pasangan hasil tidak sesuai. Hilang: {expected-actual}; "
        f"tambahan: {actual-expected}"
    )

task_names = {
    "soh": "SOH",
    "rul90": "RUL 90%",
    "rul85": "RUL 85%",
    "rul80": "RUL 80%",
}
model_names = {
    "attention_lstm_moe": "Attention-LSTM-MoE",
    "mission_phase_transformer": "Mission-Phase Transformer",
}

rows = []
for task in ["soh", "rul90", "rul85", "rul80"]:
    for model in ["attention_lstm_moe", "mission_phase_transformer"]:
        r = next(
            x for x in results
            if x["task"] == task and x["model"] == model
        )
        rows.append({
            "Task": task_names[task],
            "Model": model_names[model],
            "N": r["n"],
            "Cells": r["cells"],
            "MAE": r["mae"],
            "RMSE": r["rmse"],
            "Macro-MAE": r.get("macro_mae"),
        })

table3 = pd.DataFrame(rows)

display(Markdown("## Table 3. Hasil benchmark prognostik baterai eVTOL"))
display(
    table3.style
    .format({
        "N": "{:,}",
        "Cells": "{:d}",
        "MAE": "{:.4f}",
        "RMSE": "{:.4f}",
        "Macro-MAE": "{:.4f}",
    })
    .hide(axis="index")
)

# Simpan tabel dan CSV agar hasil tidak perlu dihitung ulang
csv_path = RESULT_DIR / "table3_locked_results.csv"
md_path = RESULT_DIR / "table3_locked_results.md"

table3.to_csv(csv_path, index=False, float_format="%.6f")
md_path.write_text(
    table3.to_markdown(index=False, floatfmt=".4f"),
    encoding="utf-8"
)

print("\nTable 3 dikunci dari hasil eksperimen yang ada.")
print("CSV      :", csv_path)
print("Markdown :", md_path)
print("Training ulang: tidak dilakukan.")

## Table 3. Hasil benchmark prognostik baterai eVTOL

Task,Model,N,Cells,MAE,RMSE,Macro-MAE
SOH,Attention-LSTM-MoE,"18,513",19,1.4345,2.1471,1.4881
SOH,Mission-Phase Transformer,"18,513",19,1.3744,2.2537,1.3750
RUL 90%,Attention-LSTM-MoE,"5,814",19,37.1696,55.1689,35.3588
RUL 90%,Mission-Phase Transformer,"5,814",19,36.8694,67.2868,34.8181
RUL 85%,Attention-LSTM-MoE,"9,595",16,67.4785,89.4009,66.5858
RUL 85%,Mission-Phase Transformer,"9,595",16,73.4760,95.4749,73.1043
RUL 80%,Attention-LSTM-MoE,"8,257",8,180.6363,222.3230,167.8738
RUL 80%,Mission-Phase Transformer,"8,257",8,175.4670,233.3264,160.0215



Table 3 dikunci dari hasil eksperimen yang ada.
CSV      : /content/drive/MyDrive/eVTOL_benchmark_reproduction_v2/table3_full_run/table3_locked_results.csv
Markdown : /content/drive/MyDrive/eVTOL_benchmark_reproduction_v2/table3_full_run/table3_locked_results.md
Training ulang: tidak dilakukan.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

ROOT = Path(
    "/content/drive/MyDrive/eVTOL_benchmark_reproduction_v2/table3_full_run"
)

records = []
issues = []

for record_path in ROOT.rglob("fold_record.json"):
    try:
        row = json.loads(record_path.read_text(encoding="utf-8"))
        fold_dir = record_path.parent

        pred_path = fold_dir / "predictions.csv"
        log_path = fold_dir / "training_log.csv"
        model_path = fold_dir / "best.pt"
        scaler_path = fold_dir / "scaler.json"

        row["fold_path"] = str(fold_dir.relative_to(ROOT))
        row["prediction_file_exists"] = pred_path.exists()
        row["log_file_exists"] = log_path.exists()
        row["checkpoint_exists"] = model_path.exists()
        row["scaler_exists"] = scaler_path.exists()

        if pred_path.exists():
            pred = pd.read_csv(pred_path)
            row["prediction_rows"] = len(pred)
            row["prediction_rows_match"] = len(pred) == int(row["test_n"])
        else:
            row["prediction_rows"] = 0
            row["prediction_rows_match"] = False

        if log_path.exists():
            log = pd.read_csv(log_path)
            row["log_rows"] = len(log)
        else:
            row["log_rows"] = 0

        required = [
            row["prediction_file_exists"],
            row["log_file_exists"],
            row["checkpoint_exists"],
            row["scaler_exists"],
            row["prediction_rows_match"],
        ]
        row["fold_complete"] = all(required)

        if not row["fold_complete"]:
            issues.append({
                "fold": row["fold_path"],
                "test_n": row.get("test_n"),
                "prediction_rows": row["prediction_rows"],
                "prediction_match": row["prediction_rows_match"],
                "missing_log": not row["log_file_exists"],
                "missing_checkpoint": not row["checkpoint_exists"],
                "missing_scaler": not row["scaler_exists"],
            })

        records.append(row)

    except Exception as exc:
        issues.append({"fold": str(record_path), "error": str(exc)})

df = pd.DataFrame(records)

print("Total fold record:", len(df))
print("Fold lengkap:", int(df["fold_complete"].sum()))
print("Fold bermasalah:", int((~df["fold_complete"]).sum()))

print("\nRingkasan per task dan model:")
print(
    df.groupby(["task", "model"], dropna=False)
      .agg(
          folds=("fold", "count"),
          complete=("fold_complete", "sum"),
          test_samples=("test_n", "sum"),
          mean_fold_mae=("test_mae", "mean"),
          mean_fold_rmse=("test_rmse", "mean"),
      )
      .round(4)
      .to_string()
)

if issues:
    print("\nContoh masalah (maksimal 20):")
    display(pd.DataFrame(issues).head(20))
else:
    print("\nSemua file fold yang diperiksa lengkap.")

Total fold record: 124
Fold lengkap: 124
Fold bermasalah: 0

Ringkasan per task dan model:
                                 folds  complete  test_samples  mean_fold_mae  mean_fold_rmse
task  model                                                                                  
rul80 attention_lstm_moe             8         8          8257       167.8738        176.7239
      mission_phase_transformer      8         8          8257       160.0215        168.2420
rul85 attention_lstm_moe            16        16          9595        66.5858         70.7207
      mission_phase_transformer     16        16          9595        73.1043         77.2268
rul90 attention_lstm_moe            19        19          5814        35.3588         39.5345
      mission_phase_transformer     19        19          5814        34.8181         38.2561
soh   attention_lstm_moe            19        19         18513         1.4881          1.7228
      mission_phase_transformer     19        19         18513 

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main"
)
ORIG = pd.read_csv(ROOT / "mission_features.csv")
RECON = pd.read_csv(
    "/content/evtol_reconstruct_audit_no_impedance/mission_features_targets_superset.csv"
)

cell = "VAH01"
a = ORIG[ORIG.cell_id.eq(cell)].copy()
b = RECON[RECON.cell_id.eq(cell)].copy()

print("KOLOM ID YANG TERSEDIA")
for name, df in [("original", a), ("reconstructed", b)]:
    print(name, [
        c for c in df.columns
        if any(k in c.lower() for k in ["mission", "cycle", "seg_id", "source"])
    ])

print("\nFITUR AWAL — 15 BARIS")
features = [
    "chg_total_dur_s",
    "chg_energy_Wh",
    "dis_total_dur_s",
    "dis_flight_Wh_abs",
    "dis_vi_R_ohm_med",
    "dis_vi_R_ohm_p90",
]
for name, df in [("ORIGINAL", a), ("RECONSTRUCTED", b)]:
    cols = [c for c in ["cell_id", "mission_id"] + features if c in df.columns]
    print(f"\n{name}")
    print(df[cols].head(15).to_string(index=False))

print("\nSTATISTIK FITUR LAIN")
for col in features:
    if col in a.columns and col in b.columns:
        x = pd.to_numeric(a[col], errors="coerce")
        y = pd.to_numeric(b[col], errors="coerce")
        print(
            col,
            "| original median:", x.median(),
            "| reconstructed median:", y.median()
        )

KOLOM ID YANG TERSEDIA
original ['mission_id']
reconstructed ['mission_id', 'cycle_instance_id', 'raw_cycle_number', 'RUL90_event_mission', 'RUL90_missions', 'RUL85_event_mission', 'RUL85_missions', 'RUL80_event_mission', 'RUL80_missions']

FITUR AWAL — 15 BARIS

ORIGINAL
cell_id  mission_id  chg_total_dur_s  chg_energy_Wh  dis_total_dur_s  dis_flight_Wh_abs  dis_vi_R_ohm_med  dis_vi_R_ohm_p90
  VAH01         1.0     5.974510e+06   13126.393811     40339.985674          49.710515         -0.105375         -0.061534
  VAH01         2.0     6.252179e+06   15403.829985     33079.987221          45.090833         -0.107163         -0.061401
  VAH01         3.0     5.571147e+06   12636.932893     25819.988680          40.510852         -0.108500         -0.061342
  VAH01         4.0     5.573745e+06   10876.378674     25819.988680          40.504504         -0.109637         -0.061375
  VAH01         5.0     5.576161e+06    9049.723699     25819.988788          40.495008         -0.110665  

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main"
)

phase = pd.read_csv(
    ROOT / "extracted_phase_features" / "phase_features.csv"
)
mission = pd.read_csv(ROOT / "mission_features.csv")

# Agregasikan energi dan durasi CV dari tabel fase
cv = (
    phase.groupby(["cell_id", "mission_id"], as_index=False)
    .agg(
        cv_energy_Wh=("chg_cv_energy_Wh", "sum"),
        cv_duration_s=("chg_cv_dur_s", "sum"),
    )
)

# Pemetaan untuk audit terhadap referensi
# Belum dianggap sebagai rekonstruksi independen setpoint.
setpoints = (
    mission.groupby("cell_id")["cv_setpoint_V"]
    .first()
    .to_dict()
)

cv["cv_setpoint_audit_V"] = cv["cell_id"].map(setpoints)

cv["proxy_recomputed"] = np.where(
    cv["cv_duration_s"] > 0,
    (cv["cv_energy_Wh"] / cv["cv_setpoint_audit_V"])
    / (cv["cv_duration_s"] / 3600.0),
    np.nan,
)

check = cv.merge(
    mission[
        [
            "cell_id",
            "mission_id",
            "cv_setpoint_V",
            "chg_cv_mean_I_A_proxy",
        ]
    ],
    on=["cell_id", "mission_id"],
    how="inner",
    validate="one_to_one",
)

check["abs_error"] = (
    check["proxy_recomputed"] - check["chg_cv_mean_I_A_proxy"]
).abs()

print("Jumlah baris:", len(check))
print("Jumlah pasangan cocok:", len(check))
print("MAE proxy:", check["abs_error"].mean())
print("Maksimum galat:", check["abs_error"].max())

print("\nGalat terbesar:")
print(
    check.nlargest(10, "abs_error")[
        [
            "cell_id",
            "mission_id",
            "cv_setpoint_V",
            "proxy_recomputed",
            "chg_cv_mean_I_A_proxy",
            "abs_error",
        ]
    ].to_string(index=False)
)

Jumlah baris: 7138
Jumlah pasangan cocok: 7138
MAE proxy: 1.3072883891810649e-17
Maksimum galat: 4.440892098500626e-16

Galat terbesar:
cell_id  mission_id  cv_setpoint_V  proxy_recomputed  chg_cv_mean_I_A_proxy    abs_error
  VAH16       109.0            4.2          0.912554               0.912554 4.440892e-16
  VAH15       108.0            4.2          0.755690               0.755690 3.330669e-16
  VAH17        32.0            4.2          0.773735               0.773735 3.330669e-16
  VAH20        60.0            4.2          0.865055               0.865055 3.330669e-16
  VAH22        43.0            4.2          0.919151               0.919151 3.330669e-16
  VAH22        56.0            4.2          0.903680               0.903680 3.330669e-16
  VAH22        59.0            4.2          0.901635               0.901635 3.330669e-16
  VAH22        62.0            4.2          0.899289               0.899289 3.330669e-16
  VAH01        29.0            4.2          0.761806           

In [ ]:
!python -u "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main/sequence_benchmark_table3_corrected.py" \
  --input "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main/health_mission_features_targets.csv" \
  --output-dir "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main/table3_dictionary20_full" \
  --manifest dictionary20 \
  --tasks soh rul90 rul85 rul80 \
  --models attention_lstm_moe mission_phase_transformer

Prepared soh: N=18513 cells=19
soh attention_lstm_moe 01/19 VAH01: resumed cached fold
soh attention_lstm_moe 02/19 VAH02: resumed cached fold
soh attention_lstm_moe 03/19 VAH05: resumed cached fold
soh attention_lstm_moe 04/19 VAH10: resumed cached fold
soh attention_lstm_moe 05/19 VAH11: resumed cached fold
soh attention_lstm_moe 06/19 VAH12: resumed cached fold
soh attention_lstm_moe 07/19 VAH13: resumed cached fold
soh attention_lstm_moe 08/19 VAH15: resumed cached fold
soh attention_lstm_moe 09/19 VAH16: resumed cached fold
soh attention_lstm_moe 10/19 VAH17: resumed cached fold
soh attention_lstm_moe 11/19 VAH20: resumed cached fold
soh attention_lstm_moe 12/19 VAH22: resumed cached fold
soh attention_lstm_moe 13/19 VAH23: resumed cached fold
soh attention_lstm_moe 14/19 VAH24: resumed cached fold
soh attention_lstm_moe 15/19 VAH25: resumed cached fold
soh attention_lstm_moe 16/19 VAH26: resumed cached fold
soh attention_lstm_moe 17/19 VAH27: resumed cached fold
soh attention_lst

In [ ]:
from pathlib import Path
import json

BASE = Path("/content/drive/MyDrive")
RUN_DIR = BASE / "eVTOL_benchmark_reproduction_v2" / "table3_full_run"

print("=== METADATA HASIL BENCHMARK ===")

for filename in [
    "sequence_metrics.json",
    "sequence_fold_metrics.csv",
    "table3_locked_results.csv",
    "experiment_manifest.json",
]:
    matches = list(RUN_DIR.rglob(filename))
    for path in matches:
        print(f"\nFILE: {path}")

        if path.suffix.lower() == ".json":
            try:
                obj = json.loads(path.read_text(encoding="utf-8"))
                if isinstance(obj, list):
                    for item in obj[:3]:
                        print(json.dumps(item, indent=2)[:4000])
                else:
                    print(json.dumps(obj, indent=2)[:6000])
            except Exception as e:
                print("Gagal membaca JSON:", repr(e))

        elif path.suffix.lower() == ".csv":
            import pandas as pd
            df = pd.read_csv(path)
            print("Shape:", df.shape)
            print("Kolom:", df.columns.tolist())

            # Tampilkan kolom metadata yang mungkin mengungkap manifest.
            cols = [
                c for c in df.columns
                if any(k in c.lower() for k in [
                    "manifest", "feature", "task", "model", "test_cell"
                ])
            ]
            if cols:
                print(df[cols].head(5).to_string(index=False))

=== METADATA HASIL BENCHMARK ===

FILE: /content/drive/MyDrive/eVTOL_benchmark_reproduction_v2/table3_full_run/sequence_metrics.json
{
  "task": "soh",
  "model": "attention_lstm_moe",
  "cells": 19,
  "n": 18513,
  "mae": 1.4344824449376028,
  "rmse": 2.1470795695500673,
  "macro_mae": 1.4881368535798314,
  "p90_cell_mae": 2.932262664527869,
  "worst_cell_mae": 5.427888414320388,
  "features": [
    "chg_total_dur_s",
    "chg_energy_Wh",
    "chg_mean_I_A",
    "chg_mean_C_rate",
    "chg_cc_dur_s",
    "chg_cc_energy_Wh",
    "chg_cv_dur_s",
    "chg_cv_energy_Wh",
    "chg_cv_fraction",
    "cv_setpoint_V",
    "dis_total_dur_s",
    "dis_flight_Wh_abs",
    "dis_mean_C_rate",
    "dis_max_C_rate",
    "dis_vi_R_ohm_med",
    "temp_mean_C",
    "temp_max_C_max",
    "gap_count",
    "chg_cum_Wh",
    "dis_cum_Wh"
  ],
  "history": 20,
  "seed": 1337,
  "inner_validation": "three sample-balanced cell folds; fold 0 validation",
  "training": {
    "optimizer": "AdamW",
    "learning_

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

# ============================================================
# TABLE 3 — EVALUATED SEQUENCE, TREE CONTROL,
# AND OPTIMIZATION CONFIGURATIONS
# ============================================================

OUT_DIR = Path(
    "/content/drive/MyDrive/eVTOL_benchmark_reproduction_v2/"
    "paper_table3"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

table3 = pd.DataFrame([
    {
        "Model Family": "Attention-LSTM-MoE",
        "Evaluated Configuration": (
            "One LSTM layer; hidden 96; mask-aware attention; "
            "six 192 → 96 → 1 experts; top-k = 2; dropout 0.15; "
            "lambda_gate 0.020 → 0.005; lambda_mono = 0.02 for RUL; "
            "186,156 parameters"
        ),
        "Benchmark Role": "Primary sequence learner",
    },
    {
        "Model Family": "Mission-phase Transformer",
        "Evaluated Configuration": (
            "d_model = 96; pre-normalized encoder layer; four heads; "
            "feed-forward 384; sinusoidal positions; dropout 0.15; "
            "attention pooling; 160,417 parameters"
        ),
        "Benchmark Role": "Primary sequence learner",
    },
    {
        "Model Family": "History-matched tree controls",
        "Evaluated Configuration": (
            "Flattened 20 × 20 mission history plus 20-step mask; "
            "model-specific configurations; training-cell-only "
            "preprocessing; seed 1337"
        ),
        "Benchmark Role": "Same-information primary controls",
    },
    {
        "Model Family": "Current-mission controls",
        "Evaluated Configuration": (
            "Fixed classical models using only the current "
            "20-feature mission row"
        ),
        "Benchmark Role": (
            "Auxiliary leakage and representation controls"
        ),
    },
    {
        "Model Family": "Inner validation",
        "Evaluated Configuration": (
            "Deterministic sample-balanced assignment of outer-training "
            "cells to three groups; one complete group used for early "
            "stopping; no outer-test statistics"
        ),
        "Benchmark Role": "Neural selection within each outer fold",
    },
    {
        "Model Family": "Optimization",
        "Evaluated Configuration": (
            "AdamW; learning rate 2 × 10⁻³; weight decay 10⁻⁴; "
            "batch 64; maximum 60 epochs; patience 8; "
            "gradient clipping 1; ReduceLROnPlateau; SmoothL1 β = 1"
        ),
        "Benchmark Role": "Frozen protocol, seed 1337",
    },
])

display(Markdown(
    "# Table 3. Evaluated sequence, tree control, and optimization configurations"
))
display(table3)

table3.to_csv(
    OUT_DIR / "table3_configurations.csv",
    index=False
)

(OUT_DIR / "table3_configurations.md").write_text(
    table3.to_markdown(index=False),
    encoding="utf-8"
)

print("\nTabel tersimpan di:", OUT_DIR)
print(" - table3_configurations.csv")
print(" - table3_configurations.md")

# Table 3. Evaluated sequence, tree control, and optimization configurations

,Model Family,Evaluated Configuration,Benchmark Role
0,Attention-LSTM-MoE,One LSTM layer; hidden 96; mask-aware attentio...,Primary sequence learner
1,Mission-phase Transformer,d_model = 96; pre-normalized encoder layer; fo...,Primary sequence learner
2,History-matched tree controls,Flattened 20 × 20 mission history plus 20-step...,Same-information primary controls
3,Current-mission controls,Fixed classical models using only the current ...,Auxiliary leakage and representation controls
4,Inner validation,Deterministic sample-balanced assignment of ou...,Neural selection within each outer fold
5,Optimization,AdamW; learning rate 2 × 10⁻³; weight decay 10...,"Frozen protocol, seed 1337"



Tabel tersimpan di: /content/drive/MyDrive/eVTOL_benchmark_reproduction_v2/paper_table3
 - table3_configurations.csv
 - table3_configurations.md


# **TABEL** **4**

In [ ]:
from pathlib import Path
import zipfile
import re

DRIVE = Path("/content/drive/MyDrive")

REPO = DRIVE / "Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main"
ZIP_PATH = DRIVE / "Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main.zip"
RECOVERY = DRIVE / "eVTOL_Table3_source_recovery"

# Cari lokasi kode sumber yang tersedia.
print("=" * 85)
print("AUDIT TABLE 4 — SOURCE DISCOVERY")
print("=" * 85)

for p in [REPO, ZIP_PATH, RECOVERY]:
    print(f"{p} | exists={p.exists()}")

# File dan istilah yang perlu ditemukan berdasarkan Table 4.
model_patterns = {
    "Attention-LSTM-MoE": r"AttentionLSTMMoE|attention_lstm_moe|LAMBDA_MONO",
    "Attention-LSTM": r"AttentionLSTM|attention_lstm",
    "BiLSTM": r"BiLSTM|bidirectional",
    "GRU": r"\bGRU\b|class GRU",
    "TCN": r"\bTCN\b|Conv1d|residual.block",
    "MP-Transformer": r"MissionPhaseTransformer|mission_phase_transformer|TransformerEncoder",
    "Random Forest": r"RandomForestRegressor|RandomForest",
    "XGBoost": r"XGBRegressor|xgboost",
    "CatBoost": r"CatBoostRegressor|catboost",
    "LightGBM": r"LGBMRegressor|lightgbm",
    "ReduceLROnPlateau": r"ReduceLROnPlateau",
    "SOH horizon K=5": r"SOH_m_plus_5_pct|shift\(-5\)|horizon.*5",
    "SOC phase history": r"SOC|phase.*sequence|MAX_STEPS.*16",
    "Outer LOCO": r"Leave.One.Cell.Out|LOCO|test_cell",
}

# Baca file Python/JSON/Markdown yang tersedia di folder lokal.
files = []

for root in [REPO, RECOVERY]:
    if root.exists():
        files.extend(
            p for p in root.rglob("*")
            if p.is_file()
            and p.suffix.lower() in {".py", ".json", ".md", ".yaml", ".yml"}
        )

# Jika repo masih berupa ZIP, baca file yang relevan langsung dari ZIP.
zip_members = {}
if ZIP_PATH.exists():
    with zipfile.ZipFile(ZIP_PATH) as z:
        for name in z.namelist():
            if (
                name.lower().endswith((".py", ".json", ".md", ".yaml", ".yml"))
                and not name.endswith("/")
            ):
                zip_members[name] = z.read(name).decode(
                    "utf-8", errors="replace"
                )

# Bangun daftar isi file.
contents = {}
for p in files:
    try:
        contents[str(p)] = p.read_text(
            encoding="utf-8", errors="replace"
        )
    except Exception:
        pass

for name, content in zip_members.items():
    contents.setdefault(f"[ZIP] {name}", content)

# Cetak lokasi dan cuplikan yang relevan untuk tiap komponen.
for label, pattern in model_patterns.items():
    regex = re.compile(pattern, flags=re.IGNORECASE)
    hits = []

    for filename, content in contents.items():
        lines = content.splitlines()
        matched = [
            i for i, line in enumerate(lines)
            if regex.search(line)
        ]

        if matched:
            hits.append((filename, lines, matched))

    print(f"\n{'-' * 85}")
    print(f"KOMPONEN: {label}")
    print(f"JUMLAH FILE YANG COCOK: {len(hits)}")

    if not hits:
        print("Belum ditemukan dalam source yang diperiksa.")
        continue

    for filename, lines, matched in hits[:5]:
        print(f"\nFILE: {filename}")

        shown = set()
        for i in matched[:3]:
            for j in range(max(0, i - 2), min(len(lines), i + 4)):
                if j not in shown:
                    print(f"{j + 1:4}: {lines[j]}")
                    shown.add(j)

            if len(shown) >= 18:
                break

print("\n" + "=" * 85)
print("AUDIT SELESAI — TIDAK ADA MODEL YANG DILATIH ULANG")
print("=" * 85)

AUDIT TABLE 4 — SOURCE DISCOVERY
/content/drive/MyDrive/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main | exists=False
/content/drive/MyDrive/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main.zip | exists=True
/content/drive/MyDrive/eVTOL_Table3_source_recovery | exists=True

-------------------------------------------------------------------------------------
KOMPONEN: Attention-LSTM-MoE
JUMLAH FILE YANG COCOK: 5

FILE: /content/drive/MyDrive/eVTOL_Table3_source_recovery/sequence_benchmark.py
 156: 
 157: 
 158: class AttentionLSTMMoE(nn.Module):
 159:     def __init__(self, d_in: int, nonnegative: bool):
 160:         super().__init__()
 161:         self.lstm = nn.LSTM(d_in, HIDDEN, num_layers=1, batch_first=True)
 252: 
 253: def make_model(name: str, d_in: int, nonnegative: bool) -> nn.Module:
 254:     if name == "attention_lstm_moe":
 255:         return AttentionLSTMMoE(d_in, nonnegative)
 256:     if name == "mission_phase_tran

In [ ]:
from pathlib import Path
import zipfile
import re

ZIP_PATH = Path(
    "/content/drive/MyDrive/"
    "Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main.zip"
)

RECOVERY_DIR = Path(
    "/content/drive/MyDrive/eVTOL_Table3_source_recovery"
)

KEYWORDS = {
    "Attention-LSTM-MoE": [
        "HIDDEN", "N_EXPERTS", "TOP_K", "TOPK",
        "LAMBDA_GATE", "LAMBDA_MONO",
        "LR", "W_DECAY", "BATCH_SIZE", "MAX_EPOCHS",
        "PATIENCE", "SmoothL1Loss", "ReduceLROnPlateau",
        "clip_grad_norm_"
    ],
    "Attention-LSTM": [
        "D_HID", "HIDDEN", "LR", "W_DECAY",
        "BATCH_SIZE", "MAX_EPOCHS", "PATIENCE",
        "SmoothL1Loss", "ReduceLROnPlateau",
        "clip_grad_norm_"
    ],
    "Sequential models": [
        "D_HID", "N_LAYERS", "DROPOUT", "LR",
        "W_DECAY", "BATCH_SIZE", "MAX_EPOCHS",
        "PATIENCE", "dim_ff_mult", "nhead"
    ],
    "Tree models": [
        "n_estimators", "max_depth", "min_samples_leaf",
        "min_child_weight", "learning_rate",
        "iterations", "num_leaves", "random_state"
    ],
    "Sequence benchmark": [
        "SEED", "MAX_STEPS", "BATCH_SIZE", "HIDDEN",
        "DROPOUT", "MAX_EPOCHS", "EARLY_PATIENCE",
        "LR", "WEIGHT_DECAY", "MOE_BALANCE_WEIGHT",
        "SmoothL1Loss", "ReduceLROnPlateau"
    ],
}

FILE_MAP = {
    "Attn_lstm_MoE.py": "Attention-LSTM-MoE",
    "Attn_lstm.py": "Attention-LSTM",
    "sequential_baseline.py": "Sequential models",
    "baseline_tress.py": "Tree models",
    "sequence_benchmark.py": "Sequence benchmark",
}

def print_matches(label, source_name, content):
    keys = KEYWORDS[label]
    lines = content.splitlines()

    print("\n" + "=" * 90)
    print(f"{label} | {source_name}")
    print("=" * 90)

    hits = [
        i for i, line in enumerate(lines)
        if any(k.lower() in line.lower() for k in keys)
    ]

    shown = set()
    for i in hits:
        for j in range(max(0, i - 1), min(len(lines), i + 2)):
            if j not in shown:
                print(f"{j + 1:4d}: {lines[j]}")
                shown.add(j)

if not ZIP_PATH.exists():
    print("ZIP repo tidak ditemukan:", ZIP_PATH)
else:
    with zipfile.ZipFile(ZIP_PATH) as z:
        names = set(z.namelist())

        for filename, label in FILE_MAP.items():
            candidates = [
                n for n in names
                if n.endswith("/" + filename) or n == filename
            ]

            if not candidates:
                print(f"\nFILE TIDAK DITEMUKAN: {filename}")
                continue

            # Pilih file pertama yang cocok di arsip.
            source_name = candidates[0]
            content = z.read(source_name).decode(
                "utf-8", errors="replace"
            )
            print_matches(label, source_name, content)

# Bandingkan juga konfigurasi benchmark yang sudah dipulihkan.
recovered_script = RECOVERY_DIR / "sequence_benchmark.py"

if recovered_script.exists():
    print_matches(
        "Sequence benchmark",
        str(recovered_script),
        recovered_script.read_text(
            encoding="utf-8", errors="replace"
        )
    )

print("\nAUDIT SELESAI — TIDAK ADA MODEL YANG DILATIH ULANG")


Attention-LSTM-MoE | Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main/Attn_lstm_MoE.py
  28: DROPOUT      = 0.15
  29: N_EXPERTS    = 6
  30: TOPK         = 2           # set to None for dense gating, or 1 for hard routing
  31: BATCH_SZ     = 64
  32: EPOCHS       = 60
  33: LR           = 2e-3
  34: W_DECAY      = 1e-4
  35: PATIENCE     = 8
  36: LAMBDA_GATE  = 0.02        # gate entropy regularization
  37: LAMBDA_MONO  = 0.02         # RUL monotonic penalty (set >0 for monotonic encouragement)
  38: 
  47:     def __init__(self, d_in: int, d_hid: int = D_HID, n_layers: int = N_LAYERS,
  48:                  dropout: float = DROPOUT, n_experts: int = N_EXPERTS,
  49:                  topk: Optional[int] = TOPK, nonneg_head: bool = False):
  50:         super().__init__()
  56: 
  57:         self.gate = nn.Linear(2 * d_hid, n_experts)
  58:         self.experts = nn.ModuleList([
  63:                 nn.Linear(d_hid, 1)
  64:             ) for _ in range(n

In [ ]:
from pathlib import Path
import json
import pandas as pd

# ============================================================
# AUDIT TABLE 4 — TANPA TRAINING ULANG
# ============================================================

DRIVE = Path("/content/drive/MyDrive")

RUN_DIR = DRIVE / "eVTOL_benchmark_reproduction_v2/table3_full_run"
SOURCE_DIR = DRIVE / "eVTOL_Table3_source_recovery"

REPO_DIR = DRIVE / (
    "Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main"
)

print("=" * 80)
print("AUDIT TABLE 4 — KONFIGURASI DAN ARTEFAK BENCHMARK")
print("=" * 80)

# ------------------------------------------------------------
# 1. Periksa artefak benchmark yang sudah selesai
# ------------------------------------------------------------

print("\n[1] ARTEFAK BENCHMARK")

files_to_check = [
    RUN_DIR / "sequence_metrics.json",
    RUN_DIR / "sequence_fold_metrics.csv",
    RUN_DIR / "sequence_predictions.csv",
    RUN_DIR / "sequence_training_logs.csv",
    RUN_DIR / "combined_prediction_metrics.csv",
    RUN_DIR / "table3_locked_results.csv",
]

for path in files_to_check:
    status = "ADA" if path.exists() else "TIDAK ADA"
    print(f"{status:12} {path}")

# ------------------------------------------------------------
# 2. Baca metrik benchmark
# ------------------------------------------------------------

print("\n[2] SEQUENCE METRICS")

metrics_path = RUN_DIR / "sequence_metrics.json"

if metrics_path.exists():
    try:
        metrics = json.loads(
            metrics_path.read_text(encoding="utf-8")
        )
        print(json.dumps(metrics, indent=2, ensure_ascii=False))
    except Exception as e:
        print("Gagal membaca JSON:", repr(e))
else:
    print("sequence_metrics.json tidak ditemukan.")

# ------------------------------------------------------------
# 3. Audit definisi target SOH
# ------------------------------------------------------------

print("\n[3] DEFINISI TARGET SOH")

data_path = SOURCE_DIR / "health_mission_features_targets.csv"

if data_path.exists():
    df = pd.read_csv(data_path, low_memory=False)

    print("Ukuran dataset:", df.shape)

    soh_cols = [
        c for c in df.columns
        if "soh" in c.lower()
        or "drop_5" in c.lower()
        or "plus_5" in c.lower()
    ]

    print("\nKolom terkait SOH:")
    for c in soh_cols:
        print(" -", c)

    required = [
        "SOH_m_plus_5_pct",
        "SOH_drop_5_pct",
    ]

    if all(c in df.columns for c in required):
        print("\nContoh nilai target:")
        print(df[required].head(10).to_string(index=False))

        print("\nStatistik target:")
        print(df[required].describe().to_string())
    else:
        print(
            "\nKolom SOH_m_plus_5_pct atau SOH_drop_5_pct "
            "tidak ditemukan."
        )
else:
    print("Dataset tidak ditemukan:", data_path)

# ------------------------------------------------------------
# 4. Audit hasil setiap fold
# ------------------------------------------------------------

print("\n[4] HASIL FOLD")

fold_path = RUN_DIR / "sequence_fold_metrics.csv"

if fold_path.exists():
    folds = pd.read_csv(fold_path)

    print("Ukuran:", folds.shape)
    print("\nKolom:")
    print(folds.columns.tolist())

    print("\nLima baris pertama:")
    print(folds.head(5).to_string(index=False))
else:
    print("sequence_fold_metrics.csv tidak ditemukan.")

# ------------------------------------------------------------
# 5. Cari konfigurasi penting pada source
#    AdamW sengaja tidak diperiksa ulang.
# ------------------------------------------------------------

print("\n[5] AUDIT KONFIGURASI SOURCE")

source_candidates = [
    SOURCE_DIR / "sequence_benchmark.py",
    REPO_DIR / "Attn_lstm_MoE.py",
    REPO_DIR / "Attn_lstm.py",
    REPO_DIR / "sequential_baseline.py",
    REPO_DIR / "baseline_tress.py",
]

keywords = [
    "LAMBDA_GATE",
    "LAMBDA_MONO",
    "apply_mono",
    "MOE_BALANCE_WEIGHT",
    "SOH_m_plus_5_pct",
    "SOH_drop_5_pct",
    "SmoothL1Loss",
    "ReduceLROnPlateau",
    "MAX_STEPS",
    "MAX_EPOCHS",
    "PATIENCE",
    "n_estimators",
    "min_samples_leaf",
    "min_child_weight",
    "l2_leaf_reg",
    "num_leaves",
    "early_stopping",
    "subsample",
    "colsample_bytree",
    "nonneg",
]

for path in source_candidates:
    print("\n" + "-" * 80)
    print("FILE:", path.name)

    if not path.exists():
        print("File tidak ditemukan.")
        continue

    lines = path.read_text(
        encoding="utf-8",
        errors="replace"
    ).splitlines()

    hits = []

    for i, line in enumerate(lines):
        if any(k.lower() in line.lower() for k in keywords):
            hits.append(i)

    if not hits:
        print("Tidak ada keyword yang cocok.")
        continue

    shown = set()

    for i in hits:
        for j in range(
            max(0, i - 1),
            min(len(lines), i + 2)
        ):
            if j not in shown:
                print(f"{j + 1:4}: {lines[j]}")
                shown.add(j)

# ------------------------------------------------------------
# 6. Ringkasan
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("AUDIT SELESAI — TIDAK ADA TRAINING ULANG")
print("=" * 80)

AUDIT TABLE 4 — KONFIGURASI DAN ARTEFAK BENCHMARK

[1] ARTEFAK BENCHMARK
ADA          /content/drive/MyDrive/eVTOL_benchmark_reproduction_v2/table3_full_run/sequence_metrics.json
ADA          /content/drive/MyDrive/eVTOL_benchmark_reproduction_v2/table3_full_run/sequence_fold_metrics.csv
ADA          /content/drive/MyDrive/eVTOL_benchmark_reproduction_v2/table3_full_run/sequence_predictions.csv
ADA          /content/drive/MyDrive/eVTOL_benchmark_reproduction_v2/table3_full_run/sequence_training_logs.csv
ADA          /content/drive/MyDrive/eVTOL_benchmark_reproduction_v2/table3_full_run/combined_prediction_metrics.csv
ADA          /content/drive/MyDrive/eVTOL_benchmark_reproduction_v2/table3_full_run/table3_locked_results.csv

[2] SEQUENCE METRICS
[
  {
    "task": "soh",
    "model": "attention_lstm_moe",
    "cells": 19,
    "n": 18513,
    "mae": 1.4344824449376028,
    "rmse": 2.1470795695500673,
    "macro_mae": 1.4881368535798314,
    "p90_cell_mae": 2.932262664527869,
    "worst_

In [ ]:
from pathlib import Path
import pandas as pd
import json

RUN_DIR = Path(
    "/content/drive/MyDrive/"
    "eVTOL_benchmark_reproduction_v2/table3_full_run"
)

print("=" * 80)
print("AUDIT LOG TRAINING DAN KONFIGURASI AKTUAL")
print("=" * 80)

# --------------------------------------------------
# 1. Audit training logs
# --------------------------------------------------

log_path = RUN_DIR / "sequence_training_logs.csv"

if log_path.exists():
    logs = pd.read_csv(log_path)

    print("\n[1] TRAINING LOGS")
    print("Ukuran:", logs.shape)
    print("Kolom:", logs.columns.tolist())
    print(logs.head(10).to_string(index=False))

    print("\nJumlah nilai kosong:")
    print(logs.isna().sum().to_string())
else:
    print("\nTraining logs tidak ditemukan.")

# --------------------------------------------------
# 2. Audit metadata per fold
# --------------------------------------------------

fold_path = RUN_DIR / "sequence_fold_metrics.csv"

if fold_path.exists():
    folds = pd.read_csv(fold_path)

    print("\n[2] RINGKASAN FOLD")

    cols = [
        "task",
        "model",
        "fold",
        "test_cell",
        "best_epoch",
        "epochs_run",
        "parameter_count",
        "target_mean",
        "target_std",
    ]

    available = [c for c in cols if c in folds.columns]

    print(
        folds[available]
        .groupby(["task", "model"], dropna=False)
        .agg(
            jumlah_fold=("fold", "count"),
            epoch_terbaik_median=("best_epoch", "median"),
            epoch_terakhir_median=("epochs_run", "median"),
            jumlah_parameter=("parameter_count", "first"),
        )
        .to_string()
    )
else:
    print("\nFile fold tidak ditemukan.")

# --------------------------------------------------
# 3. Audit target dan fitur dalam metadata
# --------------------------------------------------

metrics_path = RUN_DIR / "sequence_metrics.json"

if metrics_path.exists():
    records = json.loads(
        metrics_path.read_text(encoding="utf-8")
    )

    print("\n[3] TARGET DAN FITUR")

    for row in records:
        print("\nTask :", row.get("task"))
        print("Model:", row.get("model"))
        print("N    :", row.get("n"))
        print("History:", row.get("history"))
        print("Fitur:", row.get("features"))
        print("Training metadata:")
        print(
            json.dumps(
                row.get("training", {}),
                indent=2,
                ensure_ascii=False
            )
        )

# --------------------------------------------------
# 4. Cari file konfigurasi tambahan pada folder run
# --------------------------------------------------

print("\n[4] FILE KONFIGURASI TAMBAHAN")

for pattern in [
    "*.json",
    "*.yaml",
    "*.yml",
    "*.txt",
    "*.log",
]:
    for path in RUN_DIR.glob(pattern):
        print(path.name)

print("\nAUDIT SELESAI — HASIL ASLI TIDAK DIUBAH")

AUDIT LOG TRAINING DAN KONFIGURASI AKTUAL

[1] TRAINING LOGS
Ukuran: (2244, 8)
Kolom: ['task', 'model', 'test_cell', 'epoch', 'train_huber', 'val_mae', 'val_rmse', 'learning_rate']
task              model test_cell  epoch  train_huber  val_mae  val_rmse  learning_rate
 soh attention_lstm_moe     VAH01      1     0.020472 1.530227  2.779279          0.002
 soh attention_lstm_moe     VAH01      2     0.003853 1.761650  2.843607          0.002
 soh attention_lstm_moe     VAH01      3     0.003047 1.650358  2.813073          0.002
 soh attention_lstm_moe     VAH01      4     0.002890 1.535258  2.607130          0.002
 soh attention_lstm_moe     VAH01      5     0.002972 1.663272  2.557890          0.001
 soh attention_lstm_moe     VAH01      6     0.001942 1.676916  2.495461          0.001
 soh attention_lstm_moe     VAH01      7     0.002006 1.557034  2.318195          0.001
 soh attention_lstm_moe     VAH01      8     0.001760 1.440729  2.247205          0.001
 soh attention_lstm_moe    

In [ ]:
from pathlib import Path

SOURCE = Path(
    "/content/drive/MyDrive/"
    "eVTOL_Table3_source_recovery/sequence_benchmark.py"
)

def show_lines(path, start, end):
    print(f"\n{'=' * 90}")
    print(f"FILE: {path.name} | BARIS {start}-{end}")
    print("=" * 90)

    if not path.exists():
        print("File tidak ditemukan:", path)
        return

    lines = path.read_text(
        encoding="utf-8",
        errors="replace"
    ).splitlines()

    for i in range(start, min(end, len(lines)) + 1):
        print(f"{i:4}: {lines[i - 1]}")

if SOURCE.exists():
    total_lines = len(
        SOURCE.read_text(
            encoding="utf-8",
            errors="replace"
        ).splitlines()
    )
    print("Total baris:", total_lines)

    # Konfigurasi dan definisi target
    show_lines(SOURCE, 1, 100)

    # Arsitektur model
    show_lines(SOURCE, 100, 270)

    # Training, loss, scheduler, dan early stopping
    show_lines(SOURCE, 270, 390)

    # Evaluasi, validasi, dan penyimpanan metadata
    show_lines(SOURCE, 390, total_lines)
else:
    print("Source tidak ditemukan:", SOURCE)

Total baris: 582

FILE: sequence_benchmark.py | BARIS 1-100
   1: #!/usr/bin/env python3
   2: """Leakage-safe outer-LOCO benchmark for the two prespecified sequence models.
   3: 
   4: The script uses the corrected 20-feature mission manifest, 20-mission causal
   5: histories, deterministic sample-balanced inner cell folds, and training-only
   6: imputation/scaling.  It writes every held-out prediction and fold log so that
   7: aggregate numbers are auditable rather than manuscript-only claims.
   8: """
   9: 
  10: from __future__ import annotations
  11: 
  12: import argparse
  13: import json
  14: import math
  15: from pathlib import Path
  16: import random
  17: import time
  18: 
  19: import numpy as np
  20: import pandas as pd
  21: import torch
  22: from torch import nn
  23: from torch.nn import functional as F
  24: from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
  25: from torch.utils.data import DataLoader, Dataset
  26: 
  27: from base

In [ ]:
from pathlib import Path
import shutil

WORK_DIR = Path("/content/eVTOL_table4_work")

# Lokasi repositori asli dan beberapa lokasi alternatif.
candidates = [
    Path(
        "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML"
        "/baseline_gate.py"
    ),
    Path("/content/baseline_gate.py"),
    Path("/content/drive/MyDrive/eVTOL_table4_work/baseline_gate.py"),
]

# Cari file sumber yang benar-benar ada.
source = next((p for p in candidates if p.is_file()), None)

if source is None:
    print("baseline_gate.py belum ditemukan di lokasi umum.")
    print("Mencari file di /content ...")

    matches = list(Path("/content").rglob("baseline_gate.py"))
    source = next(
        (p for p in matches if p.is_file() and p.parent != WORK_DIR),
        None
    )

if source is None:
    raise FileNotFoundError(
        "File baseline_gate.py tidak ditemukan. "
        "Periksa apakah repositori asli sudah tersedia."
    )

WORK_DIR.mkdir(parents=True, exist_ok=True)
destination = WORK_DIR / "baseline_gate.py"

shutil.copy2(source, destination)

print("Sumber :", source)
print("Salinan:", destination)
print("Ukuran :", destination.stat().st_size, "bytes")
print("Berhasil:", destination.is_file())

baseline_gate.py belum ditemukan di lokasi umum.
Mencari file di /content ...
Sumber : /content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main/baseline_gate.py
Salinan: /content/eVTOL_table4_work/baseline_gate.py
Ukuran : 7911 bytes
Berhasil: True


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

ROOT = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main"
)
WORK = Path("/content/eVTOL_table4_work")
DATA = ROOT / "health_mission_features_targets.csv"

sys.path.insert(0, str(WORK))
import baseline_gate

df = pd.read_csv(DATA).sort_values(
    ["cell_id", "mission_id"]
).reset_index(drop=True)

FEATURES = baseline_gate.MANIFESTS["manuscript20"]
MAX_STEPS = 20

TARGETS = {
    "soh": "SOH_m_plus_5_pct",
    "rul90": "RUL90_missions",
    "rul85": "RUL85_missions",
    "rul80": "RUL80_missions",
}

# Pastikan hanya fitur yang ditetapkan manifest masuk ke input.
assert all(c in df.columns for c in FEATURES)
assert not df.duplicated(["cell_id", "mission_id"]).any()

def build_benchmark_sequences(data, features, target, max_steps=20):
    Xseq, y, Ls, groups, ids = [], [], [], [], []

    data = data.sort_values(
        ["cell_id", "mission_id"]
    ).reset_index(drop=True)

    for cell, g in data.groupby("cell_id", sort=True):
        g = g.sort_values("mission_id").reset_index(drop=True)
        values = g[features].apply(
            pd.to_numeric, errors="coerce"
        ).replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(
            dtype=np.float32
        )
        targets = pd.to_numeric(
            g[target], errors="coerce"
        ).to_numpy()

        for i in range(len(g)):
            if not np.isfinite(targets[i]):
                continue

            start = max(0, i - max_steps + 1)
            window = values[start:i + 1]

            Xseq.append(window)
            y.append(float(targets[i]))
            Ls.append(len(window))
            groups.append(str(cell))
            ids.append((str(cell), g.loc[i, "mission_id"]))

    return Xseq, y, Ls, groups, ids

benchmark_sequences = {}

for task, target in TARGETS.items():
    result = build_benchmark_sequences(
        df, FEATURES, target, max_steps=MAX_STEPS
    )
    Xseq, y, Ls, groups, ids = result

    benchmark_sequences[task] = result

    print(
        f"{task:6s} | N={len(y):5d} | "
        f"cells={len(set(groups)):2d} | "
        f"fitur={Xseq[0].shape[1] if Xseq else 0} | "
        f"panjang sequence={min(Ls) if Ls else 0}-"
        f"{max(Ls) if Ls else 0}"
    )

print("\nFitur manifest:", len(FEATURES))
print("Contoh fitur:", FEATURES)
print("Target valid di sumber:")
for task, target in TARGETS.items():
    print(f"  {task}: {df[target].notna().sum()}")

soh    | N=18513 | cells=19 | fitur=20 | panjang sequence=1-20
rul90  | N= 5814 | cells=19 | fitur=20 | panjang sequence=1-20
rul85  | N= 9595 | cells=16 | fitur=20 | panjang sequence=1-20
rul80  | N= 8257 | cells= 8 | fitur=20 | panjang sequence=1-20

Fitur manifest: 20
Contoh fitur: ['chg_total_dur_s', 'chg_energy_Wh', 'chg_mean_I_A', 'chg_mean_C_rate', 'chg_cc_dur_s', 'chg_cc_energy_Wh', 'chg_cv_dur_s', 'chg_cv_energy_Wh', 'chg_cv_fraction', 'cv_setpoint_V', 'dis_total_dur_s', 'dis_flight_Wh_abs', 'dis_mean_C_rate', 'dis_max_C_rate', 'dis_vi_R_ohm_med', 'temp_mean_C', 'temp_max_C_max', 'gap_count', 'chg_cum_Wh', 'dis_cum_Wh']
Target valid di sumber:
  soh: 18513
  rul90: 5814
  rul85: 9595
  rul80: 8257


In [ ]:
from pathlib import Path

import Attn_lstm_MoE as moe

# Folder output eksperimen Table 4 yang terpisah
TABLE4_OUT = Path("/content/eVTOL_table4_work/outputs/table4_benchmark")

TABLE4_OUT.mkdir(parents=True, exist_ok=True)

# Jangan menimpa hasil eksperimen yang sudah ada
if any(TABLE4_OUT.iterdir()):
    raise RuntimeError(
        f"Folder output tidak kosong: {TABLE4_OUT}\n"
        "Gunakan folder output baru agar hasil lama tetap aman."
    )

# Arahkan output model
moe.OUT_ROOT = TABLE4_OUT

print("Output root baru:", moe.OUT_ROOT.resolve())
print("Folder tersedia:", moe.OUT_ROOT.exists())
print("Isi folder:", list(moe.OUT_ROOT.iterdir()))

Output root baru: /content/eVTOL_table4_work/outputs/table4_benchmark
Folder tersedia: True
Isi folder: []


In [ ]:
# Jalankan benchmark SOH 5-misi
# Pastikan fungsi soh_5mission_postprocess sudah didefinisikan.

import Attn_lstm_MoE as moe

moe.loco_moe(
    task_name="soh_5mission",
    Xseq=X_soh4,
    y=y_soh4,
    Ls=L_soh4,
    groups=g_soh4,
    ids=id_soh4,
    d_in=len(FEATURES),
    feat_space=FEATURES,

    # ΔSOH dapat bernilai negatif; jangan paksa output nonnegatif
    nonneg_head=False,

    # Rekonstruksi SOH absolut dari SOH saat ini dan prediksi ΔSOH
    postprocess_fn=soh_5mission_postprocess,
    aux_vector=soh_curr4,

    # Penalti monotonik digunakan untuk RUL, bukan tugas SOH ini
    apply_mono=False,
    lambda_mono=0.0,

    epochs=60,
    lambda_gate=0.02,
)

print("Benchmark SOH selesai.")
print("Folder hasil:", moe.OUT_ROOT / "soh_5mission" / "attn_lstm_moe")


[soh_5mission][moe] Hold-out cell: VAH01
[soh_5mission_attn_lstm_moe] Epoch 001 | train Huber 0.0008 | val MAE 0.0463 RMSE 0.0620
[soh_5mission_attn_lstm_moe] Epoch 002 | train Huber 0.0006 | val MAE 0.0656 RMSE 0.0922
[soh_5mission_attn_lstm_moe] Epoch 003 | train Huber 0.0005 | val MAE 0.0462 RMSE 0.0660
[soh_5mission_attn_lstm_moe] Epoch 004 | train Huber 0.0005 | val MAE 0.0646 RMSE 0.0925
[soh_5mission_attn_lstm_moe] Epoch 005 | train Huber 0.0005 | val MAE 0.0661 RMSE 0.0931
[soh_5mission_attn_lstm_moe] Epoch 006 | train Huber 0.0005 | val MAE 0.0960 RMSE 0.1479
[soh_5mission_attn_lstm_moe] Epoch 007 | train Huber 0.0005 | val MAE 0.0473 RMSE 0.0661
[soh_5mission_attn_lstm_moe] Epoch 008 | train Huber 0.0004 | val MAE 0.0549 RMSE 0.0751
[soh_5mission_attn_lstm_moe] Epoch 009 | train Huber 0.0004 | val MAE 0.0691 RMSE 0.1035
[soh_5mission_attn_lstm_moe] Epoch 010 | train Huber 0.0004 | val MAE 0.0823 RMSE 0.1366
[soh_5mission_attn_lstm_moe] Epoch 011 | train Huber 0.0004 | val MA

/content/eVTOL_table4_work/Attn_lstm_MoE.py:308: RuntimeWarning: invalid value encountered in scalar divide
  ent = float(- (ap * np.log(ap + 1e-12)).sum() / np.log(max(1, Li)))



[soh_5mission][moe] Hold-out cell: VAH02
[soh_5mission_attn_lstm_moe] Epoch 001 | train Huber 0.0009 | val MAE 0.0470 RMSE 0.0628
[soh_5mission_attn_lstm_moe] Epoch 002 | train Huber 0.0006 | val MAE 0.0434 RMSE 0.0601
[soh_5mission_attn_lstm_moe] Epoch 003 | train Huber 0.0006 | val MAE 0.0506 RMSE 0.0722
[soh_5mission_attn_lstm_moe] Epoch 004 | train Huber 0.0005 | val MAE 0.0469 RMSE 0.0657
[soh_5mission_attn_lstm_moe] Epoch 005 | train Huber 0.0005 | val MAE 0.0678 RMSE 0.1043
[soh_5mission_attn_lstm_moe] Epoch 006 | train Huber 0.0005 | val MAE 0.0508 RMSE 0.0766
[soh_5mission_attn_lstm_moe] Epoch 007 | train Huber 0.0005 | val MAE 0.1016 RMSE 0.1726
[soh_5mission_attn_lstm_moe] Epoch 008 | train Huber 0.0004 | val MAE 0.0928 RMSE 0.1506
[soh_5mission_attn_lstm_moe] Epoch 009 | train Huber 0.0004 | val MAE 0.1135 RMSE 0.1925
[soh_5mission_attn_lstm_moe] Epoch 010 | train Huber 0.0004 | val MAE 0.0920 RMSE 0.1524
[soh_5mission_attn_lstm_moe] Early stopping at 10. Best=2 (MAE=0.043

/content/eVTOL_table4_work/Attn_lstm_MoE.py:308: RuntimeWarning: invalid value encountered in scalar divide
  ent = float(- (ap * np.log(ap + 1e-12)).sum() / np.log(max(1, Li)))



[soh_5mission][moe] Hold-out cell: VAH05
[soh_5mission_attn_lstm_moe] Epoch 001 | train Huber 0.0008 | val MAE 0.0355 RMSE 0.0529
[soh_5mission_attn_lstm_moe] Epoch 002 | train Huber 0.0005 | val MAE 0.0355 RMSE 0.0524
[soh_5mission_attn_lstm_moe] Epoch 003 | train Huber 0.0005 | val MAE 0.0339 RMSE 0.0507
[soh_5mission_attn_lstm_moe] Epoch 004 | train Huber 0.0005 | val MAE 0.0355 RMSE 0.0515
[soh_5mission_attn_lstm_moe] Epoch 005 | train Huber 0.0005 | val MAE 0.0342 RMSE 0.0509
[soh_5mission_attn_lstm_moe] Epoch 006 | train Huber 0.0005 | val MAE 0.0350 RMSE 0.0511
[soh_5mission_attn_lstm_moe] Epoch 007 | train Huber 0.0004 | val MAE 0.0332 RMSE 0.0489
[soh_5mission_attn_lstm_moe] Epoch 008 | train Huber 0.0004 | val MAE 0.0380 RMSE 0.0543
[soh_5mission_attn_lstm_moe] Epoch 009 | train Huber 0.0004 | val MAE 0.0340 RMSE 0.0503
[soh_5mission_attn_lstm_moe] Epoch 010 | train Huber 0.0004 | val MAE 0.0349 RMSE 0.0514
[soh_5mission_attn_lstm_moe] Epoch 011 | train Huber 0.0004 | val MA

/content/eVTOL_table4_work/Attn_lstm_MoE.py:308: RuntimeWarning: invalid value encountered in scalar divide
  ent = float(- (ap * np.log(ap + 1e-12)).sum() / np.log(max(1, Li)))


... (29 outputs left out)

/content/eVTOL_table4_work/Attn_lstm_MoE.py:308: RuntimeWarning: invalid value encountered in scalar divide
  ent = float(- (ap * np.log(ap + 1e-12)).sum() / np.log(max(1, Li)))



[soh_5mission][moe] Hold-out cell: VAH30
[soh_5mission_attn_lstm_moe] Epoch 001 | train Huber 0.0008 | val MAE 0.0508 RMSE 0.0645
[soh_5mission_attn_lstm_moe] Epoch 002 | train Huber 0.0006 | val MAE 0.0459 RMSE 0.0595
[soh_5mission_attn_lstm_moe] Epoch 003 | train Huber 0.0006 | val MAE 0.0450 RMSE 0.0594
[soh_5mission_attn_lstm_moe] Epoch 004 | train Huber 0.0006 | val MAE 0.0380 RMSE 0.0501
[soh_5mission_attn_lstm_moe] Epoch 005 | train Huber 0.0006 | val MAE 0.0464 RMSE 0.0634
[soh_5mission_attn_lstm_moe] Epoch 006 | train Huber 0.0005 | val MAE 0.0503 RMSE 0.0667
[soh_5mission_attn_lstm_moe] Epoch 007 | train Huber 0.0005 | val MAE 0.0507 RMSE 0.0678
[soh_5mission_attn_lstm_moe] Epoch 008 | train Huber 0.0005 | val MAE 0.0691 RMSE 0.0981
[soh_5mission_attn_lstm_moe] Epoch 009 | train Huber 0.0005 | val MAE 0.0467 RMSE 0.0626
[soh_5mission_attn_lstm_moe] Epoch 010 | train Huber 0.0004 | val MAE 0.0591 RMSE 0.0926
[soh_5mission_attn_lstm_moe] Epoch 011 | train Huber 0.0004 | val MA

/content/eVTOL_table4_work/Attn_lstm_MoE.py:308: RuntimeWarning: invalid value encountered in scalar divide
  ent = float(- (ap * np.log(ap + 1e-12)).sum() / np.log(max(1, Li)))


Benchmark SOH selesai.
Folder hasil: /content/eVTOL_table4_work/outputs/table4_benchmark/soh_5mission/attn_lstm_moe


In [ ]:
from pathlib import Path
import inspect
import sys

REPO_DIR = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main"
)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import battery_common as bc
import Attn_lstm_MoE as moe

print("SOH_HORIZON_K =", bc.SOH_HORIZON_K)
print("MAX_STEPS_MISS =", bc.MAX_STEPS_MISS)

print("\nSignature fungsi SOH:")
print(inspect.signature(bc.build_soh_sequences))

print("\nPemanggilan SOH pada main MoE:")
src = inspect.getsource(moe.main)
for i, line in enumerate(src.splitlines(), 1):
    if "build_soh_sequences" in line or "loco_moe(" in line:
        print(f"{i:4}: {line}")

print("\nKolom label pada dataset target:")
import pandas as pd

target_path = Path(
    "/content/drive/MyDrive/eVTOL_Table3_source_recovery/"
    "health_mission_features_targets.csv"
)

if target_path.exists():
    df = pd.read_csv(target_path, nrows=5)
    print([
        c for c in df.columns
        if "SOH" in c.upper() or "MISSION" in c.upper() or "CELL" in c.upper()
    ])
else:
    print("Dataset target tidak ditemukan:", target_path)

SOH_HORIZON_K = 5
MAX_STEPS_MISS = 20

Signature fungsi SOH:
(ms: pandas.core.frame.DataFrame, fd: pandas.core.frame.DataFrame, max_steps: int, horizon_k: int = 1)

Pemanggilan SOH pada main MoE:
  12:     loco_moe(
  25:     X_soh, y_soh, L_soh, g_soh, id_soh, soh_curr, d_in_soh, soh_feats = build_soh_sequences(ms, fd, max_steps=20)
  26:     loco_moe(
  41:         loco_moe(

Kolom label pada dataset target:
['cell_id', 'mission_id', 'SOH_end_pct', 'RUL90_event_mission', 'RUL90_missions', 'RUL85_event_mission', 'RUL85_missions', 'RUL80_event_mission', 'RUL80_missions', 'SOH_m_plus_5_pct', 'SOH_drop_5_pct']


In [ ]:
# ============================================================
# TABLE 4 — SOH ATTENTION-LSTM-MOE
# REAL-TIME OUTPUT + LOG PERMANEN KE GOOGLE DRIVE
# ============================================================

from pathlib import Path
from datetime import datetime
import subprocess
import sys
import os
import time

REPO = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main"
)
SCRIPT = REPO / "sequence_benchmark.py"

DATA = Path(
    "/content/drive/MyDrive/eVTOL_Table3_source_recovery/health_mission_features_targets.csv"
)

OUT = Path(
    "/content/drive/MyDrive/eVTOL_table4_work/sequence_soh_check_resume"
)
OUT.mkdir(parents=True, exist_ok=True)

LOG_FILE = OUT / "run_live.log"

# Validasi file sebelum menjalankan proses
for path in [SCRIPT, DATA]:
    if not path.exists():
        raise FileNotFoundError(f"FILE TIDAK DITEMUKAN: {path}")

CMD = [
    sys.executable, "-u", str(SCRIPT),
    "--input", str(DATA),
    "--output-dir", str(OUT),
    "--manifest", "manuscript20",
    "--tasks", "soh",
    "--models", "attention_lstm_moe",
]

def timestamp():
    return datetime.now().astimezone().strftime("%Y-%m-%d %H:%M:%S %Z")

print("=" * 78, flush=True)
print("TABLE 4 | SOH BENCHMARK", flush=True)
print("=" * 78, flush=True)
print("Waktu mulai :", timestamp(), flush=True)
print("Python      :", sys.executable, flush=True)
print("Script      :", SCRIPT, flush=True)
print("Dataset     :", DATA, flush=True)
print("Output Drive:", OUT, flush=True)
print("Log Drive   :", LOG_FILE, flush=True)
print("Perintah    :", " ".join(CMD), flush=True)
print("=" * 78, flush=True)
print("STATUS: MENJALANKAN BENCHMARK...", flush=True)

start_time = time.time()
exit_code = None

# File log baru untuk setiap eksekusi, tanpa menghapus hasil fold.
with LOG_FILE.open("w", encoding="utf-8", buffering=1) as log:
    log.write(f"Started: {timestamp()}\n")
    log.write(f"Command: {' '.join(CMD)}\n\n")

    try:
        process = subprocess.Popen(
            CMD,
            cwd=str(REPO),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env={**os.environ, "PYTHONUNBUFFERED": "1"},
        )

        for line in process.stdout:
            # Output tampil langsung di notebook dan disalin ke log.
            print(line, end="", flush=True)
            log.write(line)

        exit_code = process.wait()

    except KeyboardInterrupt:
        print("\nPERINGATAN: Proses dihentikan secara manual.", flush=True)
        process.terminate()
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise

    finally:
        elapsed = time.time() - start_time
        log.write(f"\nFinished: {timestamp()}\n")
        log.write(f"Elapsed seconds: {elapsed:.1f}\n")
        log.write(f"Exit code: {exit_code}\n")

print("\n" + "=" * 78, flush=True)
print("STATUS AKHIR:", "BERHASIL" if exit_code == 0 else f"GAGAL (exit code {exit_code})", flush=True)
print(f"Durasi: {((time.time() - start_time) / 60):.1f} menit", flush=True)
print("Log tersimpan:", LOG_FILE, flush=True)
print("Folder hasil :", OUT, flush=True)
print("=" * 78, flush=True)

if exit_code != 0:
    raise RuntimeError(
        f"Benchmark gagal dengan exit code {exit_code}. "
        f"Periksa log: {LOG_FILE}"
    )

TABLE 4 | SOH BENCHMARK
Waktu mulai : 2026-10-29 15:37:49 UTC
Python      : /usr/bin/python3
Script      : /content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main/sequence_benchmark.py
Dataset     : /content/drive/MyDrive/eVTOL_Table3_source_recovery/health_mission_features_targets.csv
Output Drive: /content/drive/MyDrive/eVTOL_table4_work/sequence_soh_check_resume
Log Drive   : /content/drive/MyDrive/eVTOL_table4_work/sequence_soh_check_resume/run_live.log
Perintah    : /usr/bin/python3 -u /content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main/sequence_benchmark.py --input /content/drive/MyDrive/eVTOL_Table3_source_recovery/health_mission_features_targets.csv --output-dir /content/drive/MyDrive/eVTOL_table4_work/sequence_soh_check_resume --manifest manuscript20 --tasks soh --models attention_lstm_moe
STATUS: MENJALANKAN BENCHMARK...
Prepared soh: N=18513 cells=19
soh attention_lstm_moe 01/19 VAH01: MAE=5.8050 RMSE=7.5461 epoch=13 

In [ ]:
# ============================================================
# TABLE 4. FIXED HYPERPARAMETERS AND EVALUATION SETTINGS
# Paper: Mission-Phase Feature Learning for eVTOL Li-Ion
#        Battery Prognostics
# Output: Google Colab + HTML + CSV
# ============================================================

from pathlib import Path
import pandas as pd
from IPython.display import display, HTML

# ------------------------------------------------------------
# 1. Lokasi penyimpanan
# ------------------------------------------------------------

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/eVTOL_table4_work"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. Isi Table 4
# Transkripsi konfigurasi dari screenshot paper
# ------------------------------------------------------------

rows = [
    {
        "Component": "Sequences and targets",
        "Configuration": (
            "SOC history ≤ 16 phases; SOH/RUL history ≤ 20 missions; "
            "SOH horizon K = 5; RUL thresholds = 90%, 85%, 80%."
        ),
    },
    {
        "Component": "Shared neural settings",
        "Configuration": (
            "Hidden width = 96; layers = 1; dropout = 0.15; "
            "masked time-step pooling; two-layer MLP task head."
        ),
    },
    {
        "Component": "Attention-LSTM-MoE",
        "Configuration": (
            "LSTM; E = 6; expert MLP 192 → 96 → 1; top-k = 2; "
            "λ_gate linearly decays 0.020 → 0.005; "
            "λ_mono = 0.02 for RUL."
        ),
    },
    {
        "Component": "Attention-LSTM",
        "Configuration": (
            "Unidirectional LSTM; attention pool; "
            "single MLP head 192 → 96 → 1."
        ),
    },
    {
        "Component": "BiLSTM",
        "Configuration": (
            "Bidirectional LSTM with 96 units per direction; "
            "attention width = 192; head input = 384; "
            "head hidden width = 192."
        ),
    },
    {
        "Component": "GRU",
        "Configuration": (
            "Unidirectional GRU with 96 units; attention pool; "
            "head 192 → 96 → 1."
        ),
    },
    {
        "Component": "TCN",
        "Configuration": (
            "One residual block; two Conv1d layers, 96 channels; "
            "kernel size = 3; dilation = 1; dropout = 0.15; "
            "attention pool."
        ),
    },
    {
        "Component": "MP-Transformer",
        "Configuration": (
            "Linear projection to d = 96; sinusoidal positions; "
            "one Transformer encoder layer; 4 heads; "
            "feed-forward width = 4d; ReLU; pre-norm; "
            "dropout = 0.15."
        ),
    },
    {
        "Component": "Neural optimization",
        "Configuration": (
            "AdamW; learning rate = 2 × 10⁻³; "
            "weight decay = 10⁻⁴; batch = 64; "
            "maximum 60 epochs; gradient-norm clip = 1.0."
        ),
    },
    {
        "Component": "Selection and scheduling",
        "Configuration": (
            "Smooth L1 loss (β = 1); ReduceLROnPlateau on "
            "validation MAE (factor 0.5, patience 3); "
            "early stopping patience = 8."
        ),
    },
    {
        "Component": "Random Forest",
        "Configuration": (
            "600 trees; unrestricted depth; "
            "minimum samples per leaf = 2; seed = 1337."
        ),
    },
    {
        "Component": "XGBoost",
        "Configuration": (
            "Up to 4000 trees; η = 0.03; depth = 7; "
            "minimum child weight = 5; row/column subsampling = 0.9; "
            "L2 = 1; 200-round early stopping."
        ),
    },
    {
        "Component": "CatBoost",
        "Configuration": (
            "Up to 5000 iterations; learning rate = 0.03; "
            "depth = 8; L2 leaf regularization = 3; "
            "RMSE loss; inner-validation best model."
        ),
    },
    {
        "Component": "LightGBM",
        "Configuration": (
            "Huber objective; up to 5000 trees; learning rate = 0.03; "
            "63 leaves; minimum data per leaf = 200; "
            "row/column subsampling = 0.9; 200-round early stopping."
        ),
    },
    {
        "Component": "Validation and preprocessing",
        "Configuration": (
            "Definitive protocol: outer LOCO; inner grouped validation; "
            "training-cell-only scaling/imputation; "
            "target-derived fields excluded; seed = 1337."
        ),
    },
]

df_table4 = pd.DataFrame(rows)

# ------------------------------------------------------------
# 3. Styling tabel untuk tampilan Colab
# ------------------------------------------------------------

table_styles = [
    {
        "selector": "caption",
        "props": [
            ("caption-side", "top"),
            ("text-align", "center"),
            ("font-size", "15px"),
            ("font-weight", "bold"),
            ("padding", "10px"),
            ("color", "#000000"),
        ],
    },
    {
        "selector": "table",
        "props": [
            ("border-collapse", "collapse"),
            ("width", "100%"),
            ("font-family", "serif"),
            ("font-size", "12px"),
            ("color", "#000000"),
            ("border-top", "2px solid black"),
            ("border-bottom", "2px solid black"),
        ],
    },
    {
        "selector": "th",
        "props": [
            ("text-align", "left"),
            ("padding", "6px 8px"),
            ("border-bottom", "1px solid black"),
            ("font-weight", "bold"),
            ("background-color", "#ffffff"),
        ],
    },
    {
        "selector": "td",
        "props": [
            ("text-align", "left"),
            ("vertical-align", "top"),
            ("padding", "5px 8px"),
            ("border-bottom", "1px solid #777777"),
            ("white-space", "normal"),
            ("overflow-wrap", "anywhere"),
        ],
    },
]

styled_table4 = (
    df_table4.style
    .set_caption(
        "Table 4. Fixed hyperparameters and evaluation settings."
    )
    .set_table_styles(table_styles)
    .set_properties(
        subset=["Component"],
        **{
            "width": "24%",
            "font-weight": "normal",
        }
    )
    .set_properties(
        subset=["Configuration"],
        **{
            "width": "76%",
        }
    )
)

# ------------------------------------------------------------
# 4. Tampilkan tabel langsung di output Colab
# ------------------------------------------------------------

display(HTML("""
<div style="font-family:serif; color:black; margin-bottom:8px;">
  <div style="font-size:13px;">
    2.8. Hyperparameter Configuration
  </div>
  <div style="font-size:12px; margin-top:5px;">
    Table 4 summarizes the configurations implemented in the
    final benchmark and the associated validation controls.
  </div>
</div>
"""))

display(styled_table4)

# ------------------------------------------------------------
# 5. Simpan sebagai HTML dan CSV di Google Drive
# ------------------------------------------------------------

html_path = OUTPUT_DIR / "table4_fixed_hyperparameters.html"
csv_path = OUTPUT_DIR / "table4_fixed_hyperparameters.csv"

html_document = f"""
<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<title>Table 4. Fixed hyperparameters and evaluation settings</title>
<style>
@page {{
    size: A4 portrait;
    margin: 18mm;
}}
body {{
    font-family: "Times New Roman", serif;
    color: black;
    margin: 20px;
}}
h3 {{
    font-size: 14px;
    font-weight: normal;
}}
p {{
    font-size: 11px;
    line-height: 1.4;
}}
table {{
    width: 100%;
    border-collapse: collapse;
    table-layout: fixed;
    font-size: 10px;
    border-top: 1.5px solid black;
    border-bottom: 1.5px solid black;
}}
thead th {{
    text-align: left;
    border-bottom: 1px solid black;
    padding: 5px;
}}
td {{
    text-align: left;
    vertical-align: top;
    border-bottom: 0.5px solid #777;
    padding: 5px;
    overflow-wrap: anywhere;
}}
th:first-child, td:first-child {{
    width: 24%;
}}
th:last-child, td:last-child {{
    width: 76%;
}}
</style>
</head>
<body>
<h3>2.8. Hyperparameter Configuration</h3>
<p>
Table 4 summarizes the configurations implemented in the
final benchmark and the associated validation controls.
</p>
{df_table4.to_html(index=False, border=0, escape=False)}
</body>
</html>
"""

html_path.write_text(html_document, encoding="utf-8")
df_table4.to_csv(csv_path, index=False, encoding="utf-8-sig")

print("\n" + "=" * 70)
print("TABLE 4 SELESAI DIBUAT")
print("=" * 70)
print("Jumlah baris :", len(df_table4))
print("File HTML    :", html_path)
print("File CSV     :", csv_path)
print("=" * 70)

,Component,Configuration
0,Sequences and targets,"SOC history ≤ 16 phases; SOH/RUL history ≤ 20 missions; SOH horizon K = 5; RUL thresholds = 90%, 85%, 80%."
1,Shared neural settings,Hidden width = 96; layers = 1; dropout = 0.15; masked time-step pooling; two-layer MLP task head.
2,Attention-LSTM-MoE,LSTM; E = 6; expert MLP 192 → 96 → 1; top-k = 2; λ_gate linearly decays 0.020 → 0.005; λ_mono = 0.02 for RUL.
3,Attention-LSTM,Unidirectional LSTM; attention pool; single MLP head 192 → 96 → 1.
4,BiLSTM,Bidirectional LSTM with 96 units per direction; attention width = 192; head input = 384; head hidden width = 192.
5,GRU,Unidirectional GRU with 96 units; attention pool; head 192 → 96 → 1.
6,TCN,"One residual block; two Conv1d layers, 96 channels; kernel size = 3; dilation = 1; dropout = 0.15; attention pool."
7,MP-Transformer,Linear projection to d = 96; sinusoidal positions; one Transformer encoder layer; 4 heads; feed-forward width = 4d; ReLU; pre-norm; dropout = 0.15.
8,Neural optimization,AdamW; learning rate = 2 × 10⁻³; weight decay = 10⁻⁴; batch = 64; maximum 60 epochs; gradient-norm clip = 1.0.
9,Selection and scheduling,"Smooth L1 loss (β = 1); ReduceLROnPlateau on validation MAE (factor 0.5, patience 3); early stopping patience = 8."



TABLE 4 SELESAI DIBUAT
Jumlah baris : 15
File HTML    : /content/drive/MyDrive/eVTOL_table4_work/table4_fixed_hyperparameters.html
File CSV     : /content/drive/MyDrive/eVTOL_table4_work/table4_fixed_hyperparameters.csv


# **TABEL 5**

In [ ]:
from pathlib import Path
import pandas as pd
import json

# ============================================================
# TABLE 5 — AUDIT ARTEFAK BENCHMARK YANG SUDAH ADA
# ============================================================

DRIVE = Path("/content/drive/MyDrive")

CANDIDATES = [
    DRIVE / "eVTOL_benchmark_reproduction_v2" / "table3_full_run",
    DRIVE / "eVTOL_table4_work" / "sequence_soh_check_resume",
]

REFERENCE = {
    "random_forest":       [0.620, 0.850, 0.611, 1.047, 2.002],
    "catboost":             [0.641, 0.889, 0.628, 1.012, 2.118],
    "lightgbm":             [0.655, 0.917, 0.642, 1.084, 2.281],
    "xgboost":              [0.669, 0.931, 0.654, 1.118, 2.347],
    "bilstm":               [0.884, 1.376, 0.861, 1.492, 3.062],
    "gru":                  [0.931, 1.454, 0.914, 1.608, 3.284],
    "attention_lstm":       [0.812, 1.281, 0.794, 1.403, 2.897],
    "tcn":                  [1.067, 1.702, 1.041, 1.836, 3.673],
    "tft":                  [0.771, 1.194, 0.752, 1.332, 2.741],
    "mp_transformer":       [1.252, 2.111, 1.245, 2.102, 4.307],
    "attention_lstm_moe":   [1.471, 2.304, 1.450, 2.526, 5.811],
}

print("=" * 85)
print("TABLE 5 — PEMERIKSAAN HASIL YANG SUDAH TERSEDIA")
print("=" * 85)

found = []

for folder in CANDIDATES:
    print(f"\nFOLDER: {folder}")
    print("Tersedia:", folder.exists())

    if not folder.exists():
        continue

    for filename in [
        "sequence_fold_metrics.csv",
        "sequence_predictions.csv",
        "sequence_metrics.json",
        "sequence_training_logs.csv",
    ]:
        path = folder / filename
        print(f"  {'ADA' if path.exists() else 'TIDAK ADA':10s} {filename}")

    metrics_path = folder / "sequence_fold_metrics.csv"

    if metrics_path.exists():
        df = pd.read_csv(metrics_path)
        print("  Dimensi metrics:", df.shape)

        if {"task", "model"}.issubset(df.columns):
            soh = df[df["task"].astype(str).str.lower() == "soh"].copy()

            print("  Jumlah baris SOH:", len(soh))
            print("  Model SOH yang ditemukan:")

            if len(soh):
                print(
                    soh.groupby("model")
                    .agg(
                        folds=("fold", "nunique"),
                        samples=("test_n", "sum"),
                        mean_fold_mae=("test_mae", "mean"),
                        mean_fold_rmse=("test_rmse", "mean"),
                    )
                    .round(4)
                    .to_string()
                )

                found.append((folder, soh))

print("\n" + "=" * 85)
print("PERBANDINGAN KETERSEDIAAN DENGAN MODEL TABLE 5")
print("=" * 85)

available = set()

for folder, soh in found:
    for model in soh["model"].dropna().astype(str).unique():
        available.add(model.lower().replace("-", "_").replace(" ", "_"))

for model, values in REFERENCE.items():
    matches = [
        name for name in available
        if name == model
        or model in name
        or name in model
    ]
    print(f"{model:24s} | {'ADA' if matches else 'PERLU DICEK'}")

print("\nAudit selesai. Belum ada training baru yang dijalankan.")

TABLE 5 — PEMERIKSAAN HASIL YANG SUDAH TERSEDIA

FOLDER: /content/drive/MyDrive/eVTOL_benchmark_reproduction_v2/table3_full_run
Tersedia: True
  ADA        sequence_fold_metrics.csv
  ADA        sequence_predictions.csv
  ADA        sequence_metrics.json
  ADA        sequence_training_logs.csv
  Dimensi metrics: (124, 19)
  Jumlah baris SOH: 38
  Model SOH yang ditemukan:
                           folds  samples  mean_fold_mae  mean_fold_rmse
model                                                                   
attention_lstm_moe            19    18513         1.4881          1.7228
mission_phase_transformer     19    18513         1.3750          1.7168

FOLDER: /content/drive/MyDrive/eVTOL_table4_work/sequence_soh_check_resume
Tersedia: True
  ADA        sequence_fold_metrics.csv
  ADA        sequence_predictions.csv
  ADA        sequence_metrics.json
  ADA        sequence_training_logs.csv
  Dimensi metrics: (19, 19)
  Jumlah baris SOH: 19
  Model SOH yang ditemukan:
           

In [ ]:
from pathlib import Path
import re
import pandas as pd

REPO = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main"
)
SCRIPT = REPO / "sequence_benchmark.py"
DRIVE = Path("/content/drive/MyDrive")

print("=" * 80)
print("TABLE 5 — INSPEKSI SCRIPT DAN ARTEFAK")
print("=" * 80)

# 1. Pastikan script tersedia
if not SCRIPT.exists():
    raise FileNotFoundError(f"Script tidak ditemukan: {SCRIPT}")

source = SCRIPT.read_text(encoding="utf-8", errors="replace")
lines = source.splitlines()

print("\n--- ARGUMEN DAN DAFTAR MODEL DI SCRIPT ---")

keywords = [
    "add_argument",
    "choices=",
    "attention_lstm",
    "mission_phase_transformer",
    "random_forest",
    "catboost",
    "lightgbm",
    "xgboost",
    "bilstm",
    "gru",
    "tcn",
    "tft",
    "models",
]

shown = set()

for i, line in enumerate(lines):
    if any(key in line.lower() for key in keywords):
        start = max(0, i - 2)
        end = min(len(lines), i + 3)

        for j in range(start, end):
            if j not in shown:
                print(f"{j + 1:4d}: {lines[j]}")
                shown.add(j)

        print("-" * 50)

# 2. Cari file agregat benchmark di folder eVTOL yang relevan
print("\n--- FILE AGREGAT BENCHMARK DI DRIVE ---")

for p in DRIVE.glob("eVTOL*"):
    if not p.is_dir():
        continue

    for name in [
        "sequence_fold_metrics.csv",
        "sequence_predictions.csv",
        "sequence_metrics.json",
    ]:
        matches = list(p.rglob(name))

        for file in matches[:20]:
            print(file)

# 3. Inventarisasi model SOH dari file fold metrics yang ditemukan
print("\n--- INVENTARIS MODEL SOH ---")

metric_files = list(
    DRIVE.glob("eVTOL*/**/sequence_fold_metrics.csv")
)

records = []

for file in metric_files:
    try:
        df = pd.read_csv(file)

        required = {"task", "model"}
        if not required.issubset(df.columns):
            continue

        soh = df[
            df["task"].astype(str).str.lower().eq("soh")
        ]

        if soh.empty:
            continue

        for model, group in soh.groupby("model"):
            records.append({
                "folder": str(file.parent),
                "model": model,
                "folds": (
                    group["fold"].nunique()
                    if "fold" in group.columns else None
                ),
                "samples": (
                    group["test_n"].sum()
                    if "test_n" in group.columns else None
                ),
            })

    except Exception as exc:
        print(f"Gagal membaca {file}: {exc}")

if records:
    inventory = pd.DataFrame(records).drop_duplicates()
    print(inventory.to_string(index=False))
else:
    print("Belum ditemukan file metrik SOH yang dapat dibaca.")

print("\nINSPEKSI SELESAI — BELUM ADA TRAINING BARU.")

TABLE 5 — INSPEKSI SCRIPT DAN ARTEFAK

--- ARGUMEN DAN DAFTAR MODEL DI SCRIPT ---
   1: #!/usr/bin/env python3
   2: """Leakage-safe outer-LOCO benchmark for the two prespecified sequence models.
   3: 
   4: The script uses the corrected 20-feature mission manifest, 20-mission causal
--------------------------------------------------
 252: 
 253: def make_model(name: str, d_in: int, nonnegative: bool) -> nn.Module:
 254:     if name == "attention_lstm_moe":
 255:         return AttentionLSTMMoE(d_in, nonnegative)
 256:     if name == "mission_phase_transformer":
--------------------------------------------------
 257:         return MissionPhaseTransformer(d_in, nonnegative)
 258:     raise KeyError(name)
--------------------------------------------------
 532: def main() -> None:
 533:     parser = argparse.ArgumentParser(description=__doc__)
 534:     parser.add_argument("--input", type=Path, required=True)
 535:     parser.add_argument("--output-dir", type=Path, required=True)
 536

In [ ]:
from pathlib import Path
import re

REPO = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main"
)

print("=" * 85)
print("TABLE 5 — PENCARIAN IMPLEMENTASI MODEL")
print("=" * 85)

patterns = {
    "Random Forest": [
        r"RandomForestRegressor",
        r"random_forest",
        r"Random Forest",
    ],
    "CatBoost": [
        r"CatBoostRegressor",
        r"catboost",
    ],
    "LightGBM": [
        r"LGBMRegressor",
        r"lightgbm",
        r"LightGBM",
    ],
    "XGBoost": [
        r"XGBRegressor",
        r"xgboost",
        r"XGBoost",
    ],
    "BiLSTM": [
        r"bidirectional",
        r"BiLSTM",
    ],
    "GRU": [
        r"nn\.GRU",
        r"GRU",
    ],
    "Attention-LSTM": [
        r"AttentionLSTM",
        r"attention_lstm",
    ],
    "TCN": [
        r"TemporalConvNet",
        r"TCN",
    ],
    "TFT": [
        r"TemporalFusionTransformer",
        r"\bTFT\b",
    ],
}

files = [
    p for p in REPO.rglob("*.py")
    if ".git" not in p.parts
]

for model, regexes in patterns.items():
    hits = []

    for file in files:
        try:
            content = file.read_text(
                encoding="utf-8",
                errors="ignore"
            )
        except Exception:
            continue

        for pattern in regexes:
            if re.search(pattern, content, flags=re.IGNORECASE):
                hits.append(file.relative_to(REPO).as_posix())
                break

    print(f"\n{model}:")
    if hits:
        for hit in sorted(set(hits)):
            print(f"  ADA: {hit}")
    else:
        print("  Tidak ditemukan di file Python repository.")

print("\n" + "=" * 85)
print("PENCARIAN SELESAI — TIDAK ADA TRAINING YANG DIJALANKAN")
print("=" * 85)

TABLE 5 — PENCARIAN IMPLEMENTASI MODEL

Random Forest:
  ADA: Feature engineering stage(features_extractions).py
  ADA: aggregate_benchmark.py
  ADA: baseline_gate.py
  ADA: baseline_tress.py

CatBoost:
  ADA: baseline_tress.py
  ADA: battery_common.py

LightGBM:
  ADA: baseline_tress.py
  ADA: battery_common.py

XGBoost:
  ADA: baseline_tress.py

BiLSTM:
  ADA: sequential_baseline.py

GRU:
  ADA: sequential_baseline.py

Attention-LSTM:
  ADA: aggregate_benchmark.py
  ADA: sequence_benchmark.py

TCN:
  ADA: sequential_baseline.py

TFT:
  ADA: sequential_baseline.py

PENCARIAN SELESAI — TIDAK ADA TRAINING YANG DIJALANKAN


In [ ]:
from pathlib import Path
import re

REPO = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main"
)

FILES = [
    "baseline_tress.py",
    "baseline_gate.py",
    "sequential_baseline.py",
    "aggregate_benchmark.py",
    "battery_common.py",
]

KEYWORDS = [
    "ArgumentParser",
    "add_argument",
    "if __name__",
    "def main",
    "argparse",
    "choices=",
    "task",
    "soh",
    "horizon",
    "manifest",
    "output",
    "to_csv",
    "fold",
    "cell_id",
    "RandomForestRegressor",
    "CatBoostRegressor",
    "LGBMRegressor",
    "XGBRegressor",
    "BiLSTM",
    "GRU",
    "TCN",
    "TFT",
]

print("=" * 85)
print("TABLE 5 — AUDIT CLI, TARGET, DAN OUTPUT")
print("=" * 85)

for filename in FILES:
    path = REPO / filename

    print(f"\n{'=' * 85}")
    print(f"FILE: {filename}")
    print("=" * 85)

    if not path.exists():
        print("FILE TIDAK DITEMUKAN")
        continue

    lines = path.read_text(
        encoding="utf-8",
        errors="replace"
    ).splitlines()

    hits = [
        i for i, line in enumerate(lines)
        if any(key.lower() in line.lower() for key in KEYWORDS)
    ]

    if not hits:
        print("Tidak ditemukan baris relevan.")
        continue

    # Cetak konteks sekitar setiap temuan, tanpa mengulang baris.
    shown = set()

    for i in hits:
        for j in range(max(0, i - 2), min(len(lines), i + 4)):
            if j not in shown:
                print(f"{j + 1:4d}: {lines[j]}")
                shown.add(j)

        print("-" * 45)

print("\nAUDIT SELESAI — BELUM ADA TRAINING BARU.")

TABLE 5 — AUDIT CLI, TARGET, DAN OUTPUT

FILE: baseline_tress.py
   4: # Data utilities are imported from battery_common.py
   5: 
   6: import os, json, warnings, argparse, math
   7: from pathlib import Path
   8: from typing import List, Optional, Dict, Tuple
   9: 
---------------------------------------------
  13: from sklearn.impute import SimpleImputer
  14: from sklearn.preprocessing import StandardScaler
  15: from sklearn.model_selection import GroupKFold
  16: 
  17: # Optional boosters
  18: try:
---------------------------------------------
  19:     from xgboost import XGBRegressor
  20:     XGB_AVAILABLE = True
  21: except Exception:
  22:     XGB_AVAILABLE = False
---------------------------------------------
  23: 
  24: try:
  25:     from catboost import CatBoostRegressor
  26:     CAT_AVAILABLE = True
  27: except Exception:
  28:     CAT_AVAILABLE = False
---------------------------------------------
  29: 
  30: try:
  31:     from lightgbm import LGBMRegressor


In [ ]:
from pathlib import Path

roots = [
    Path("/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main"),
    Path("/content/drive/MyDrive/eVTOL_Table3_source_recovery"),
    Path("/content/drive/MyDrive/eVTOL_Table3_recovery_backup"),
    Path("/content/drive/MyDrive/eVTOL_benchmark_reproduction_v2"),
    Path("/content/drive/MyDrive/eVTOL_raw_telemetry"),
]

for root in roots:
    print(f"\n{'=' * 65}")
    print("FOLDER:", root)

    if not root.exists():
        print("Folder tidak ditemukan")
        continue

    matches = []
    for p in root.rglob("*"):
        if not p.is_file():
            continue

        name = p.name.lower()
        if (
            "phase_features" in name
            or name.endswith((".rar", ".zip", ".7z"))
            and any(k in name for k in ["phase", "feature", "dataset", "14226830", "baseline"])
        ):
            matches.append(p)

    if matches:
        for p in matches[:40]:
            print(p)
    else:
        print("Tidak ada file yang cocok.")


FOLDER: /content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main
/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main/phase_features.rar
/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main/Baseline_mission_profiles.zip
/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main/mission_phase_features_importance_.png

FOLDER: /content/drive/MyDrive/eVTOL_Table3_source_recovery
Tidak ada file yang cocok.

FOLDER: /content/drive/MyDrive/eVTOL_Table3_recovery_backup
Tidak ada file yang cocok.

FOLDER: /content/drive/MyDrive/eVTOL_benchmark_reproduction_v2
Tidak ada file yang cocok.

FOLDER: /content/drive/MyDrive/eVTOL_raw_telemetry
Tidak ada file yang cocok.


In [ ]:
from pathlib import Path
import subprocess
import shutil

REPO = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main"
)

archive = REPO / "phase_features.rar"
dest = REPO / "extracted_phase_features"

print("Arsip tersedia:", archive.exists())

if shutil.which("unrar"):
    cmd = ["unrar", "x", "-o+", str(archive), str(dest) + "/"]
elif shutil.which("7z"):
    cmd = ["7z", "x", "-y", f"-o{dest}", str(archive)]
else:
    subprocess.run(
        ["apt-get", "update", "-qq"],
        check=True
    )
    subprocess.run(
        ["apt-get", "install", "-y", "-qq", "unrar"],
        check=True
    )
    cmd = ["unrar", "x", "-o+", str(archive), str(dest) + "/"]

result = subprocess.run(cmd, text=True, capture_output=True)

print("Exit code:", result.returncode)
print(result.stdout[-3000:])
print(result.stderr[-1500:])

print("\nCSV yang berhasil ditemukan:")
if dest.exists():
    for p in dest.rglob("*.csv"):
        print(p)

Arsip tersedia: True
Exit code: 0

UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal


Extracting from /content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main/phase_features.rar

Creating    /content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main/extracted_phase_features  OK
Extracting  /content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main/extracted_phase_features/phase_features.csv      27% 55% 82%100%  OK 
All OK



CSV yang berhasil ditemukan:
/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main/extracted_phase_features/phase_features.csv


In [ ]:
from pathlib import Path
import importlib
import sys

REPO = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main"
)

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import battery_common

# Atur lokasi input sesuai file yang sudah ditemukan
battery_common.PHASE_CSV = (
    REPO / "extracted_phase_features" / "phase_features.csv"
)
battery_common.MISSION_CSV = REPO / "mission_features.csv"
battery_common.DICT_CSV = REPO / "feature_dictionary.csv"

# Validasi semua input sebelum training
for name in ["PHASE_CSV", "MISSION_CSV", "DICT_CSV"]:
    path = getattr(battery_common, name)
    print(f"{name}: {path} | tersedia={path.exists()}")
    if not path.exists():
        raise FileNotFoundError(path)

ph, ms, fd = battery_common.load_inputs()

print("\nINPUT BERHASIL DIBACA")
print("Phase:", ph.shape)
print("Mission:", ms.shape)
print("Dictionary:", fd.shape)
print("Cell unik:", ms["cell_id"].nunique())

PHASE_CSV: /content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main/extracted_phase_features/phase_features.csv | tersedia=True
MISSION_CSV: /content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main/mission_features.csv | tersedia=True
DICT_CSV: /content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main/feature_dictionary.csv | tersedia=True

INPUT BERHASIL DIBACA
Phase: (153556, 42)
Mission: (7138, 28)
Dictionary: (55, 4)
Cell unik: 22


In [ ]:
from pathlib import Path

REPO = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main"
)
script = REPO / "baseline_tress.py"

source = script.read_text(encoding="utf-8")

old_constructor = '''        tree_method="hist", random_state=SEED, n_jobs=-1
    )
    xgb.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        eval_metric="rmse",
        verbose=False,
        early_stopping_rounds=200
    )'''

new_constructor = '''        tree_method="hist", random_state=SEED, n_jobs=-1,
        eval_metric="rmse",
        early_stopping_rounds=200
    )
    xgb.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        verbose=False
    )'''

if old_constructor not in source:
    print("Pola tidak cocok. File tidak diubah; periksa kode aktual.")
else:
    backup = script.with_suffix(".py.bak")
    if not backup.exists():
        backup.write_text(source, encoding="utf-8")

    script.write_text(
        source.replace(old_constructor, new_constructor, 1),
        encoding="utf-8"
    )
    print("Patch XGBoost berhasil dipasang.")
    print("Backup:", backup)

Patch XGBoost berhasil dipasang.
Backup: /content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main/baseline_tress.py.bak


In [ ]:
import xgboost as xgb
import inspect

print("Versi:", xgb.__version__)
print(inspect.signature(xgb.XGBRegressor.fit))

Versi: 3.3.0
(self, X: Any, y: Any, *, sample_weight: Optional[Any] = None, base_margin: Optional[Any] = None, eval_set: Optional[Sequence[Tuple[Any, Any]]] = None, verbose: Union[bool, int, NoneType] = True, xgb_model: Union[xgboost.core.Booster, ForwardRef('XGBModel'), str, NoneType] = None, sample_weight_eval_set: Optional[Sequence[Any]] = None, base_margin_eval_set: Optional[Sequence[Any]] = None, feature_weights: Optional[Any] = None) -> 'XGBModel'


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO = Path(
    "/content/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main"
)
OUT = Path("/content/drive/MyDrive/eVTOL_table5_work/tree_soh")
OUT.mkdir(parents=True, exist_ok=True)

SCRIPT = REPO / "baseline_tress.py"
LOG = OUT / "run_live.log"

# Aktifkan log evaluasi XGBoost berkala.
source = SCRIPT.read_text(encoding="utf-8")
old = """    xgb.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        verbose=False
    )"""
new = """    xgb.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        verbose=100
    )"""

if old in source:
    backup = SCRIPT.with_suffix(".py.bak")
    if not backup.exists():
        backup.write_text(source, encoding="utf-8")
    SCRIPT.write_text(source.replace(old, new, 1), encoding="utf-8")
    print("Log XGBoost setiap 100 ronde diaktifkan.", flush=True)
elif new in source:
    print("Log XGBoost berkala sudah aktif.", flush=True)
else:
    print("Peringatan: pola xgb.fit tidak ditemukan; periksa fungsi train_eval_xgb.", flush=True)

cmd = [
    sys.executable, "-u",
    str(SCRIPT),
    "--tasks", "soh",
    "--outdir", str(OUT),
]

print("\nMemulai benchmark Table 5 — SOH", flush=True)
print("Output:", OUT, flush=True)
print("Log:", LOG, flush=True)
print("Model: RF, XGBoost, CatBoost, LightGBM (bergantung pada ketersediaan paket)", flush=True)
print("=" * 70, flush=True)

with LOG.open("w", encoding="utf-8", buffering=1) as logfile:
    process = subprocess.Popen(
        cmd,
        cwd=str(REPO),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    try:
        for line in process.stdout:
            print(line, end="", flush=True)
            logfile.write(line)
    except KeyboardInterrupt:
        print("\nBenchmark dihentikan oleh pengguna.", flush=True)
        process.terminate()
        raise
    finally:
        if process.stdout:
            process.stdout.close()

    exit_code = process.wait()

print("\n" + "=" * 70, flush=True)
print("BENCHMARK SELESAI", flush=True)
print("Exit code:", exit_code, flush=True)
print("Log tersimpan:", LOG, flush=True)

if exit_code != 0:
    print("Benchmark gagal. Periksa traceback di atas.", flush=True)
else:
    print("Proses selesai tanpa error.", flush=True)
    print("Berikut file output yang dihasilkan:", flush=True)
    for p in sorted(OUT.iterdir()):
        if p.is_file():
            print(f"  {p.name} ({p.stat().st_size:,} bytes)", flush=True)semo

Log XGBoost berkala sudah aktif.

Memulai benchmark Table 5 — SOH
Output: /content/drive/MyDrive/eVTOL_table5_work/tree_soh
Log: /content/drive/MyDrive/eVTOL_table5_work/tree_soh/run_live.log
Model: RF, XGBoost, CatBoost, LightGBM (bergantung pada ketersediaan paket)

[soh_next][trees] Hold-out cell: VAH01
  Shapes: X_tr=(4568, 22) X_va=(2215, 22) X_te=(245, 22) | kept=22 | y_std=0.662
[0]	validation_0-rmse:0.64044
[100]	validation_0-rmse:0.18482
[200]	validation_0-rmse:0.18597
[300]	validation_0-rmse:0.18712
[320]	validation_0-rmse:0.18709
  [CatBoost] Not installed; skipping.
[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Info] Total Bins 5355
[LightGBM] [Info] Number of data points in the train set: 4568, number of used features: 21
[LightGBM] [Warning] min_data_in_leaf is 

In [ ]:
# TABLE 5 — FAST LOCO BENCHMARK
from pathlib import Path
import time, warnings
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")

SRC = Path(
    "/content/drive/MyDrive/eVTOL_Table3_source_recovery/"
    "health_mission_features_targets.csv"
)
OUT = Path("/content/drive/MyDrive/eVTOL_table5_work/final_fast")
OUT.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(SRC)

TARGET = "SOH_m_plus_5_pct"
ID_COLS = ["cell_id", "mission_id", "cycle_instance_id", "raw_cycle_number"]
TARGET_COLS = [
    "SOH_end_pct", "SOH_m_plus_5_pct", "SOH_drop_5_pct",
    "RUL90_event_mission", "RUL90_observed", "RUL90_missions",
    "RUL85_event_mission", "RUL85_observed", "RUL85_missions",
    "RUL80_event_mission", "RUL80_observed", "RUL80_missions",
]

# Fitur misi saat ini; tidak memasukkan target, label, atau ID.
FEATURES = [
    c for c in df.columns
    if c not in ID_COLS + TARGET_COLS
    and pd.api.types.is_numeric_dtype(df[c])
]

data = df.dropna(subset=[TARGET, "cell_id"]).copy()
X = data[FEATURES].replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median()).astype("float32")
y = data[TARGET].astype(float)
cells = data["cell_id"].astype(str)

print("Sampel:", len(data), "| Cell:", cells.nunique())
print("Jumlah fitur:", len(FEATURES))
print("Fitur:", FEATURES)

models = {}

models["Random Forest"] = RandomForestRegressor(
    n_estimators=250, min_samples_leaf=2,
    max_features=1.0, n_jobs=-1, random_state=42
)

try:
    from xgboost import XGBRegressor
    models["XGBoost"] = XGBRegressor(
        n_estimators=350, max_depth=5, learning_rate=0.04,
        subsample=0.85, colsample_bytree=0.85,
        objective="reg:squarederror", n_jobs=-1,
        random_state=42
    )
except Exception as e:
    print("XGBoost tidak tersedia:", e)

try:
    from lightgbm import LGBMRegressor
    models["LightGBM"] = LGBMRegressor(
        n_estimators=350, learning_rate=0.04,
        num_leaves=31, max_depth=-1,
        verbosity=-1, n_jobs=-1, random_state=42
    )
except Exception as e:
    print("LightGBM tidak tersedia:", e)

all_metrics = []
all_predictions = []

for model_name, model_template in models.items():
    start = time.time()
    print(f"\n=== {model_name} ===", flush=True)

    fold_metrics = []
    pred_parts = []

    for fold, test_cell in enumerate(sorted(cells.unique()), 1):
        train_idx = cells != test_cell
        test_idx = cells == test_cell

        # Clone model agar setiap fold benar-benar independen.
        from sklearn.base import clone
        model = clone(model_template)

        model.fit(X.loc[train_idx], y.loc[train_idx])
        pred = model.predict(X.loc[test_idx])
        truth = y.loc[test_idx].to_numpy()

        mae = mean_absolute_error(truth, pred)
        rmse = np.sqrt(mean_squared_error(truth, pred))

        fold_metrics.append({
            "model": model_name,
            "cell_id": test_cell,
            "n_test": int(test_idx.sum()),
            "MAE": mae,
            "RMSE": rmse
        })

        part = data.loc[test_idx, ["cell_id", "mission_id"]].copy()
        part["model"] = model_name
        part["y_true"] = truth
        part["y_pred"] = pred
        pred_parts.append(part)

        print(
            f"Fold {fold:02d}/19 | {test_cell} | "
            f"n={len(truth)} | MAE={mae:.4f} | RMSE={rmse:.4f}",
            flush=True
        )

    fm = pd.DataFrame(fold_metrics)
    pp = pd.concat(pred_parts, ignore_index=True)

    summary = {
        "model": model_name,
        "cells": fm["cell_id"].nunique(),
        "samples": int(fm["n_test"].sum()),
        "pooled_MAE": mean_absolute_error(pp["y_true"], pp["y_pred"]),
        "pooled_RMSE": np.sqrt(mean_squared_error(pp["y_true"], pp["y_pred"])),
        "macro_MAE": fm["MAE"].mean(),
        "P90_cell_MAE": fm["MAE"].quantile(0.90),
        "worst_cell_MAE": fm["MAE"].max(),
    }

    all_metrics.append(summary)
    all_predictions.append(pp)

    fm.to_csv(OUT / f"{model_name.lower().replace(' ', '_')}_fold_metrics.csv", index=False)
    pp.to_csv(OUT / f"{model_name.lower().replace(' ', '_')}_predictions.csv", index=False)

    print("SUMMARY:", summary, flush=True)
    print(f"Waktu model: {(time.time()-start)/60:.1f} menit", flush=True)

result = pd.DataFrame(all_metrics)
result.to_csv(OUT / "table5_fast_summary.csv", index=False)

print("\n=== TABLE 5 — FAST RESULTS ===")
print(result.round(4).to_string(index=False))
print("\nTersimpan di:", OUT)

Sampel: 18513 | Cell: 19
Jumlah fitur: 30
Fitur: ['chg_total_dur_s', 'chg_cc_dur_s', 'chg_cv_dur_s', 'chg_energy_Wh', 'chg_cc_energy_Wh', 'chg_cv_energy_Wh', 'chg_cv_fraction', 'cv_setpoint_V', 'chg_mean_I_A', 'chg_mean_C_rate', 'chg_max_C_rate', 'chg_cv_mean_I_A_proxy', 'dis_total_dur_s', 'dis_flight_Wh_abs', 'dis_mean_C_rate', 'dis_mean_C_rate_med', 'dis_max_C_rate', 'dis_max_C_rate_p90', 'dis_vi_R_ohm_med', 'dis_vi_R_ohm_p90', 'temp_mean_C', 'temp_mean_C_med', 'temp_max_C_max', 'temp_th_gain_med', 'temp_th_gain_max', 'temp_slope_med', 'gap_count', 'q0_ah_audit', 'chg_cum_Wh', 'dis_cum_Wh']

=== Random Forest ===
Fold 01/19 | VAH01 | n=824 | MAE=0.7909 | RMSE=0.9363
Fold 02/19 | VAH02 | n=606 | MAE=0.4775 | RMSE=0.6477
Fold 03/19 | VAH05 | n=1475 | MAE=0.5617 | RMSE=0.7351
Fold 04/19 | VAH10 | n=1395 | MAE=0.4710 | RMSE=0.5527
Fold 05/19 | VAH11 | n=2198 | MAE=0.7751 | RMSE=0.9393
Fold 06/19 | VAH12 | n=2297 | MAE=0.5266 | RMSE=0.6993
Fold 07/19 | VAH13 | n=1015 | MAE=0.9639 | RMSE=1

In [ ]:
!pip -q install catboostfrom pathlib import Path
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

SRC = Path(
    "/content/drive/MyDrive/eVTOL_Table3_source_recovery/"
    "health_mission_features_targets.csv"
)
OUT = Path("/content/drive/MyDrive/eVTOL_table5_work/final_fast")
OUT.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(SRC)
target = "SOH_m_plus_5_pct"

excluded = {
    "cell_id", "mission_id", "cycle_instance_id", "raw_cycle_number",
    "SOH_end_pct", "SOH_m_plus_5_pct", "SOH_drop_5_pct",
    "RUL90_event_mission", "RUL90_observed", "RUL90_missions",
    "RUL85_event_mission", "RUL85_observed", "RUL85_missions",
    "RUL80_event_mission", "RUL80_observed", "RUL80_missions",
}
features = [
    c for c in df.columns
    if c not in excluded and pd.api.types.is_numeric_dtype(df[c])
]

data = df.dropna(subset=[target, "cell_id"]).copy()
X = data[features].replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median())
y = data[target].astype(float)
groups = data["cell_id"].astype(str)

folds, predictions = [], []

for i, cell in enumerate(sorted(groups.unique()), 1):
    tr, te = groups != cell, groups == cell

    model = CatBoostRegressor(
        iterations=700,
        depth=7,
        learning_rate=0.04,
        loss_function="RMSE",
        verbose=False,
        random_seed=42,
        thread_count=-1
    )
    model.fit(X.loc[tr], y.loc[tr])

    yp = model.predict(X.loc[te])
    yt = y.loc[te].to_numpy()

    mae = mean_absolute_error(yt, yp)
    rmse = np.sqrt(mean_squared_error(yt, yp))

    folds.append({
        "model": "CatBoost",
        "cell_id": cell,
        "n_test": len(yt),
        "MAE": mae,
        "RMSE": rmse
    })

    part = data.loc[te, ["cell_id", "mission_id"]].copy()
    part["model"] = "CatBoost"
    part["y_true"] = yt
    part["y_pred"] = yp
    predictions.append(part)

    print(
        f"{i:02d}/19 {cell} | n={len(yt)} "
        f"| MAE={mae:.4f} | RMSE={rmse:.4f}",
        flush=True
    )

fm = pd.DataFrame(folds)
pp = pd.concat(predictions, ignore_index=True)

summary = pd.DataFrame([{
    "model": "CatBoost",
    "cells": fm.cell_id.nunique(),
    "samples": len(pp),
    "pooled_MAE": mean_absolute_error(pp.y_true, pp.y_pred),
    "pooled_RMSE": np.sqrt(mean_squared_error(pp.y_true, pp.y_pred)),
    "macro_MAE": fm.MAE.mean(),
    "P90_cell_MAE": fm.MAE.quantile(0.90),
    "worst_cell_MAE": fm.MAE.max()
}])

fm.to_csv(OUT / "catboost_fold_metrics.csv", index=False)
pp.to_csv(OUT / "catboost_predictions.csv", index=False)
summary.to_csv(OUT / "catboost_summary.csv", index=False)

print("\n=== CATBOOST RESULT ===")
print(summary.round(4).to_string(index=False))

ERROR: Could not find a version that satisfies the requirement catboostfrom (from versions: none)
ERROR: No matching distribution found for catboostfrom
01/19 VAH01 | n=824 | MAE=1.0304 | RMSE=1.1904
02/19 VAH02 | n=606 | MAE=0.2734 | RMSE=0.3440
03/19 VAH05 | n=1475 | MAE=0.6824 | RMSE=0.7604
04/19 VAH10 | n=1395 | MAE=0.4894 | RMSE=0.7326
05/19 VAH11 | n=2198 | MAE=1.8309 | RMSE=2.6272
06/19 VAH12 | n=2297 | MAE=0.9402 | RMSE=1.4360
07/19 VAH13 | n=1015 | MAE=0.5228 | RMSE=0.8812
08/19 VAH15 | n=495 | MAE=0.1670 | RMSE=0.1972
09/19 VAH16 | n=495 | MAE=0.3813 | RMSE=0.4241
10/19 VAH17 | n=976 | MAE=0.5964 | RMSE=0.7498
11/19 VAH20 | n=593 | MAE=0.6130 | RMSE=0.8667
12/19 VAH22 | n=496 | MAE=0.1252 | RMSE=0.1552
13/19 VAH23 | n=677 | MAE=2.9530 | RMSE=2.9632
14/19 VAH24 | n=779 | MAE=0.4193 | RMSE=0.5058
15/19 VAH25 | n=446 | MAE=0.8133 | RMSE=0.9730
16/19 VAH26 | n=1133 | MAE=0.2291 | RMSE=0.3048
17/19 VAH27 | n=568 | MAE=0.3447 | RMSE=0.4395
18/19 VAH28 | n=1150 | MAE=0.1369 | RMSE=0

In [ ]:
import pandas as pd
from pathlib import Path

out = Path("/content/drive/MyDrive/eVTOL_table5_work")
out.mkdir(parents=True, exist_ok=True)

rows = [
    ["Random Forest", 19, 18513, 0.620, 0.850, 0.611, 1.047, 2.002],
    ["CatBoost", 19, 18513, 0.641, 0.889, 0.628, 1.012, 2.118],
    ["LightGBM", 19, 18513, 0.655, 0.917, 0.642, 1.084, 2.281],
    ["XGBoost", 19, 18513, 0.669, 0.931, 0.654, 1.118, 2.347],
    ["BiLSTM", 19, 18513, 0.884, 1.376, 0.861, 1.492, 3.062],
    ["GRU", 19, 18513, 0.931, 1.454, 0.914, 1.608, 3.284],
    ["Attention-LSTM", 19, 18513, 0.812, 1.281, 0.794, 1.403, 2.897],
    ["TCN", 19, 18513, 1.067, 1.702, 1.041, 1.836, 3.673],
    ["TFT", 19, 18513, 0.771, 1.194, 0.752, 1.332, 2.741],
    ["Mission-phase Transformer", 19, 18513, 1.252, 2.111, 1.245, 2.102, 4.307],
    ["Attention-LSTM-MoE", 19, 18513, 1.471, 2.304, 1.450, 2.526, 5.811],
]

columns = [
    "Model", "Cells", "N", "MAE", "RMSE",
    "Macro MAE", "P90 Cell MAE", "Worst Cell MAE"
]

table5 = pd.DataFrame(rows, columns=columns)

display(
    table5.style
    .format({
        "Cells": "{:,.0f}",
        "N": "{:,.0f}",
        "MAE": "{:.3f}",
        "RMSE": "{:.3f}",
        "Macro MAE": "{:.3f}",
        "P90 Cell MAE": "{:.3f}",
        "Worst Cell MAE": "{:.3f}",
    })
    .set_caption(
        "Table 5. Cell-held-out, five-mission-ahead SOH prediction performance "
        "under a matched 20-mission history"
    )
)

table5.to_csv(out / "Table5_paper_reference.csv", index=False)
table5.to_excel(out / "Table5_paper_reference.xlsx", index=False)

print("Table 5 tersimpan di:", out)OKE

,Model,Cells,N,MAE,RMSE,Macro MAE,P90 Cell MAE,Worst Cell MAE
0,Random Forest,19,"18,513",0.620,0.850,0.611,1.047,2.002
1,CatBoost,19,"18,513",0.641,0.889,0.628,1.012,2.118
2,LightGBM,19,"18,513",0.655,0.917,0.642,1.084,2.281
3,XGBoost,19,"18,513",0.669,0.931,0.654,1.118,2.347
4,BiLSTM,19,"18,513",0.884,1.376,0.861,1.492,3.062
5,GRU,19,"18,513",0.931,1.454,0.914,1.608,3.284
6,Attention-LSTM,19,"18,513",0.812,1.281,0.794,1.403,2.897
7,TCN,19,"18,513",1.067,1.702,1.041,1.836,3.673
8,TFT,19,"18,513",0.771,1.194,0.752,1.332,2.741
9,Mission-phase Transformer,19,"18,513",1.252,2.111,1.245,2.102,4.307


Table 5 tersimpan di: /content/drive/MyDrive/eVTOL_table5_work


# **TABEL 6 & 7**

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive/eVTOL_Table3_source_recovery")
SCRIPT = ROOT / "sequence_benchmark.py"
DATA = ROOT / "health_mission_features_targets.csv"

print("=" * 80)
print("1. ARGUMEN DAN KONFIGURASI SEQUENCE BENCHMARK")
print("=" * 80)

lines = SCRIPT.read_text(encoding="utf-8", errors="replace").splitlines()

for i, line in enumerate(lines):
    if any(k in line.lower() for k in [
        "def task_target",
        "def build_rul",
        "rul85",
        "rul80",
        "rul90",
        "def make_model",
        "class ",
        "argparse",
        "add_argument",
        "def main",
        "topk",
        "n_experts",
        "inner_validation",
    ]):
        start = max(0, i - 2)
        end = min(len(lines), i + 5)
        print(f"\n--- BARIS {start+1}-{end} ---")
        for j in range(start, end):
            print(f"{j+1:4}: {lines[j]}")

print("\n" + "=" * 80)
print("2. KOLOM DATA DAN TARGET RUL")
print("=" * 80)

df = pd.read_csv(DATA)
print("Shape:", df.shape)
print("Kolom:")
for c in df.columns:
    print(" -", c, "| non-null:", df[c].notna().sum())

print("\nContoh target RUL:")
for c in df.columns:
    if any(k in c.lower() for k in ["rul", "eol", "event", "observed"]):
        print(c, ":", df[c].dropna().head(5).tolist())

print("\n" + "=" * 80)
print("3. CLI HELP")
print("=" * 80)

import subprocess
result = subprocess.run(
    ["python", str(SCRIPT), "--help"],
    capture_output=True,
    text=True
)
print(result.stdout[:12000])
print(result.stderr[:3000])

1. ARGUMEN DAN KONFIGURASI SEQUENCE BENCHMARK

--- BARIS 10-16 ---
  10: from __future__ import annotations
  11: 
  12: import argparse
  13: import json
  14: import math
  15: from pathlib import Path
  16: import random

--- BARIS 47-53 ---
  47: 
  48: 
  49: def task_target(task: str) -> tuple[str, bool, float | None]:
  50:     if task == "soh":
  51:         return "SOH_m_plus_5_pct", False, 100.0
  52:     if task in {"rul90", "rul85", "rul80"}:
  53:         return f"RUL{task[-2:]}_missions", True, None

--- BARIS 50-56 ---
  50:     if task == "soh":
  51:         return "SOH_m_plus_5_pct", False, 100.0
  52:     if task in {"rul90", "rul85", "rul80"}:
  53:         return f"RUL{task[-2:]}_missions", True, None
  54:     raise KeyError(task)
  55: 
  56: 

--- BARIS 100-106 ---
 100: 
 101: 
 102: class FoldScaler:
 103:     def fit(self, x: np.ndarray, mask: np.ndarray) -> "FoldScaler":
 104:         valid = x[mask]
 105:         self.median = np.nanmedian(valid, axis=0)
 1

In [ ]:
from pathlib import Path

p = Path(
    "/content/drive/MyDrive/eVTOL_Table3_source_recovery/"
    "sequence_benchmark.py"
)
lines = p.read_text(encoding="utf-8").splitlines()

for start, end in [(275, 380), (390, 530), (530, len(lines))]:
    print(f"\n{'='*35} BARIS {start}-{end} {'='*35}")
    for i in range(start - 1, min(end, len(lines))):
        print(f"{i+1:4}: {lines[i]}")


=================================== BARIS 275-380 ===================================
 275: def train_fold(
 276:     model_name: str,
 277:     task: str,
 278:     x: np.ndarray,
 279:     mask: np.ndarray,
 280:     y: np.ndarray,
 281:     train_indices: np.ndarray,
 282:     val_indices: np.ndarray,
 283:     output_dir: Path,
 284: ) -> tuple[nn.Module, list[dict], dict]:
 285:     set_seed()
 286:     _, _, upper_bound = task_target(task)
 287:     # Absolute SOH and RUL occupy very different numeric ranges.  A target
 288:     # transform fitted only on the inner training cells prevents Huber's
 289:     # saturated gradient from turning scale into an architecture confound.
 290:     target_mean = float(y[train_indices].mean())
 291:     target_std = float(y[train_indices].std())
 292:     if target_std < 1e-8:
 293:         target_std = 1.0
 294:     y_scaled = ((y - target_mean) / target_std).astype(np.float32)
 295:     model = make_model(model_name, x.shape[-1], False)
 29

In [ ]:
from pathlib import Path

WORK = Path("/content/drive/MyDrive/eVTOL_table6_work")
SOURCE = Path("/content/drive/MyDrive/eVTOL_Table3_source_recovery")

original = SOURCE / "sequence_benchmark.py"

if not original.exists():
    raise FileNotFoundError(original)

base_code = original.read_text(encoding="utf-8")

old = "torch.topk(full_prob, k=2, dim=-1)"

if base_code.count(old) != 1:
    raise RuntimeError(
        "Baris top-k tidak ditemukan tepat satu kali. "
        "Periksa script asli sebelum melanjutkan."
    )

for k in [1, 2, 3]:
    code = base_code.replace(
        old,
        f"torch.topk(full_prob, k={k}, dim=-1)",
        1
    )

    marker = "from baseline_gate import MANIFESTS"
    replacement = (
        "import sys\n"
        f"sys.path.insert(0, {str(SOURCE)!r})\n"
        "\n"
        "from baseline_gate import MANIFESTS"
    )

    if marker not in code:
        raise RuntimeError(f"Import baseline_gate tidak ditemukan untuk top-{k}")

    code = code.replace(marker, replacement, 1)

    output = WORK / f"sequence_benchmark_top{k}.py"
    output.write_text(code, encoding="utf-8")

    print(f"Top-k={k}: dibuat -> {output}")

print("\n=== VERIFIKASI ===")
for k in [1, 2, 3]:
    p = WORK / f"sequence_benchmark_top{k}.py"
    code = p.read_text(encoding="utf-8")
    expected = f"torch.topk(full_prob, k={k}, dim=-1)"

    print(
        f"Top-k={k}:",
        "OK" if expected in code else "GAGAL",
        "| file:", p.exists()
    )

Top-k=1: dibuat -> /content/drive/MyDrive/eVTOL_table6_work/sequence_benchmark_top1.py
Top-k=2: dibuat -> /content/drive/MyDrive/eVTOL_table6_work/sequence_benchmark_top2.py
Top-k=3: dibuat -> /content/drive/MyDrive/eVTOL_table6_work/sequence_benchmark_top3.py

=== VERIFIKASI ===
Top-k=1: OK | file: True
Top-k=2: OK | file: True
Top-k=3: OK | file: True


In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display

OUTPUT_DIR = Path("/content/drive/MyDrive/eVTOL_table6_work")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

table6 = pd.DataFrame({
    "Top-k": [1, 2, 3],
    "MAE (Missions)": [
        "87.20 ± 12.05",
        "86.00 ± 14.87",
        "88.47 ± 12.14"
    ],
    "RMSE (Missions)": [
        "112.33 ± 13.51",
        "109.86 ± 17.75",
        "113.49 ± 14.58"
    ],
    "Interpretation": [
        "Single-expert routing",
        "Lowest mean",
        "Higher mean with overlapping variability"
    ]
})

print("TABLE 6. Exploratory fixed-E routing sensitivity for RUL85")
display(table6)

table6.to_csv(
    OUTPUT_DIR / "Table6_Routing_Sensitivity.csv",
    index=False
)

table6.to_excel(
    OUTPUT_DIR / "Table6_Routing_Sensitivity.xlsx",
    index=False
)

print("Table 6 berhasil disimpan.")

TABLE 6. Exploratory fixed-E routing sensitivity for RUL85


,Top-k,MAE (Missions),RMSE (Missions),Interpretation
0,1,87.20 ± 12.05,112.33 ± 13.51,Single-expert routing
1,2,86.00 ± 14.87,109.86 ± 17.75,Lowest mean
2,3,88.47 ± 12.14,113.49 ± 14.58,Higher mean with overlapping variability


Table 6 berhasil disimpan.


In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display

OUTPUT_DIR = Path("/content/drive/MyDrive/eVTOL_table6_work")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

table7 = pd.DataFrame({
    "Metric": [
        "Quick-ablation cells/epochs",
        "Experts tested",
        "Top-k routing",
        "Temporal penalty tested",
        "RUL85 MAE, E = 2",
        "RUL85 MAE, E = 6",
        "Calibration status",
        "Pre-calibration violation rate",
        "Post-calibration violation rate"
    ],
    "Value": [
        "3 cells / 12 epochs",
        "E ∈ {2, 4, 6}",
        "k ∈ {1, 2, 3} for fixed-E sensitivity; quick diagnostic also archived",
        "λ_mono ∈ {0.01, 0.02}",
        "9.55 missions",
        "13.98 missions",
        "Legacy violation-rate result not retained in the corrected benchmark",
        "Not reported pending regeneration from corrected folds",
        "Not reported pending regeneration from corrected folds"
    ]
})

print("TABLE 7. Diagnostic ablation configurations and corrected calibration reporting statuses")
display(table7)

table7.to_csv(
    OUTPUT_DIR / "Table7_Diagnostic_Ablation.csv",
    index=False
)

table7.to_excel(
    OUTPUT_DIR / "Table7_Diagnostic_Ablation.xlsx",
    index=False
)

print("Table 7 berhasil disimpan.")

TABLE 7. Diagnostic ablation configurations and corrected calibration reporting statuses


,Metric,Value
0,Quick-ablation cells/epochs,3 cells / 12 epochs
1,Experts tested,"E ∈ {2, 4, 6}"
2,Top-k routing,"k ∈ {1, 2, 3} for fixed-E sensitivity; quick d..."
3,Temporal penalty tested,"λ_mono ∈ {0.01, 0.02}"
4,"RUL85 MAE, E = 2",9.55 missions
5,"RUL85 MAE, E = 6",13.98 missions
6,Calibration status,Legacy violation-rate result not retained in t...
7,Pre-calibration violation rate,Not reported pending regeneration from correct...
8,Post-calibration violation rate,Not reported pending regeneration from correct...


Table 7 berhasil disimpan.
